# Diabetic Retinopathy Stage Detection — EfficientNetB3 Transfer Learning
**BSc (Hons) Computer Science — Computer Vision Coursework (CW1)**

This notebook detects diabetic retinopathy (DR) and grades its stage from colour fundus photographs using ImageNet transfer learning (EfficientNetB3), with Grad-CAM explainability, embedding-based similar-case retrieval, a multi-agent decision pipeline and a Gradio interface.

**Dataset:** [APTOS 2019 Blindness Detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection) (Kaggle competition). Only the **3,662 labelled images** in `train_images/` are used (`train.csv`: `id_code`, `diagnosis`); the competition's `test_images/` have no public labels and are ignored. Class counts are 1,805 No DR · 370 Mild · 999 Moderate · 193 Severe · 295 Proliferative, a **9.4× imbalance** between the largest and smallest class (recomputed in Section 1.9). All images come from **one source** (Aravind Eye Hospital, India), so results may not transfer to other cameras or populations, and there are **no patient IDs**. No DR images are also mostly much lower-resolution than the diseased ones (Section 1.10b), which points to a different camera and is a possible shortcut.

**Classes (ICDR 0–4 scale):** 0 No DR · 1 Mild NPDR · 2 Moderate NPDR · 3 Severe NPDR · 4 Proliferative DR (PDR). The 5-stage output is also collapsed into *DR vs No DR* and *referable DR (stage ≥ 2)* screening results in Section 7.4d.

### How to run
- **Kaggle (recommended):** join the APTOS 2019 competition and accept its rules, then in the editor use **Add Input → Competitions → APTOS 2019 Blindness Detection** (mounted at `/kaggle/input/aptos2019-blindness-detection/`). Turn on the GPU and Internet (needed for ImageNet weights, `pip` and the repo clone), then use **Save Version → Save & Run All** so the run continues in the background.
- **Google Colab / local:** upload `kaggle.json` when asked; Section 1.5 runs `kaggle competitions download -c aptos2019-blindness-detection` and extracts only `train.csv` and `train_images/`. You can also point the `APTOS_DATA_DIR` environment variable at an existing copy. Enable a GPU.

**Integrity rules.** Section 4.1 builds a **duplicate-aware stratified 70/15/15 split** (perceptual hashing + union-find; whole duplicate groups placed by a seeded, label-only split search, with groups of more than 50 images kept in training) and asserts that no image or near-duplicate group appears in two splits. Each run evaluates the **test split once**, in the Section 7 final evaluation; early stopping, pilot selection, TTA choice and QWK thresholds all use validation only. In the reported work the test split was seen twice, because an earlier final fit (run A) was evaluated before the stopping rule was changed (Section 6). Every reported number is computed by the notebook.

**Defaults (anti-overfitting):** 300×300 input, dropout 0.5, Phase 1 LR 1e-3 (≤8 epochs), Phase 2 LR 1e-5 with AdamW weight decay 1e-4 (≤25 epochs), top-120 fine-tuning with BatchNorm frozen, early stopping and checkpointing on **validation QWK** (patience 6), balanced class weights on. With `RUN_HYPERPARAMETER_TUNING=True`, pilot trials (hyperparameters, balancing, preprocessing/augmentation ablations, five ImageNet backbones and a from-scratch CNN) train on the full training split and are compared on the full validation split; the best selectable trial then trains the final model. Pilot metrics are screening evidence, not final results.

---
## Pipeline Sections
1. Setup & Data Acquisition
2. Preprocessing (incl. step-by-step gallery, histograms, edge detection and morphology)
3. Data Augmentation & Class Balancing
4. Duplicate-Aware Train / Validation / Test Split
5. CNN Architecture & Transfer Learning (EfficientNetB3)
6. Training Strategy (pilot trials, ablations, backbone comparison, two-phase fine-tuning)
7. Evaluation (accuracy, QWK, precision/recall/F1, confusion matrix, ROC, DR/referable-DR screening)
8. Explainability — Grad-CAM
9. Innovation A — Embedding-Based Similar-Case Retrieval
10. Innovation B — Multi-Agent Clinical Decision Pipeline
11. UI — Gradio Interface
12. Exploratory — U-Net Pseudo-Mask Demonstration
13. Research Evidence — Error Analysis & Reproducibility
---

## Section 1: Setup & Data Acquisition
**Goal:** Install dependencies, configure all project constants in one place (Config class),
verify GPU, locate or download the APTOS 2019 competition data, and load/validate the labels —
including corrupt-image filtering, a class-distribution summary and a resolution audit.

## Project Mindmap

The mindmap gives a high-level overview of the project pipeline, from dataset acquisition to deployment, and was used during planning to scope each deliverable.

![RetinaGuard AI project mindmap](https://raw.githubusercontent.com/ShazzySal/ComputerVision_CW/main/report_images/mindmap.png)

> **Figure:** Project mindmap — the central node branches into nine phases: Dataset Acquisition, Preprocessing & Enhancement, Augmentation & Balancing, Model & Transfer Learning, Training Strategy, Evaluation Metrics, Explainability Innovations (Grad-CAM, CBR, U-Net), Multi-Agent Decision Pipeline, and UI & Deployment.

## Notebook Artifact Guide: Section-by-Section Outputs

This guide maps the notebook cells to the files and visible outputs they are designed to produce. The notebook cells have not all been executed in this workspace, so these are expected outputs, not confirmation that every file currently exists. Most saved files go under the run-specific `Config.REPORTS_DIR`; classifier and U-Net weights go under `Config.CHECKPOINT_DIR`. In Colab or Kaggle, these are run folders, not automatically the app's root `report_images/` or `checkpoints/` folders.

| Notebook section | Graphs, images, CSVs, and other outputs |
|---|---|
| Section 1: Setup & Data Acquisition | `class_distribution.png` graph; `resolution_by_class.csv`, `resolution_by_class_summary.csv` and `resolution_by_class.png` (native resolution vs DR stage, Section 1.10b). Corrupt-image and image-dimension checks are also printed in notebook output. `report_images/mindmap.png` is referenced as a project figure; it is not generated by the training cells. |
| Section 2: Preprocessing | `preprocessing_class*.png` per-class before/after examples, `preprocessing_comparison.png`, `preprocessing_step_gallery.png` (each filter + histograms) and `edge_morphology_analysis.png` (Sobel, Canny, opening, top-hat, black-hat). The cells also display preprocessing examples inline. |
| Section 3: Data Augmentation & Class Balancing | `augmentation_examples.png` (one row per DR stage) and `augmentation_class0.png`-`augmentation_class4.png` (three strips per stage); augmentation previews and calculated class weights are also displayed in the notebook. |
| Section 4: Train / Validation / Test Split | `train_split.csv`, `validation_split.csv`, `test_split.csv` (also copied to the project's `splits/` folder), `duplicate_groups.csv`, `deduplication_log.csv` (including the chosen split-search seed and distance), `duplicate_group_sizes.csv`, `duplicate_group_sizes.png`, `largest_duplicate_groups.png`, `duplicate_examples.png` and `dataset_audit_summary.csv` (image and duplicate-group overlap counts); dataset audit and split-count tables; `preprocessing_quality_ablation.csv`, `preprocessing_quality_summary.csv`, `split_class_counts.csv`, `class_index_check.csv` (label -> class name -> count per split), `imbalance_ratio_summary.csv`, and `class_weighting_effect.csv` (balanced and sqrt weights); plots `preprocessing_contrast_sharpness.png`, `preprocessing_ablation_examples.png`, `split_class_distribution.png`, and `class_weighting_effect.png`. Some tables and checks are inline only. |
| Section 5: CNN Architecture & Transfer Learning | EfficientNetB3 architecture/model summary and configuration are printed inline. No separate graph or CSV is saved by this section. |
| Section 6: Training Strategy | `checkpoints/best_phase1.weights.h5` and `checkpoints/best_phase2.weights.h5`; `phase1_training_history.csv`, `phase2_training_history.csv`, and `experiment_log.csv`; `overfitting_diagnostics.csv`; `learning_rate_history.csv` and `lr_curve_final.png`; when pilot tuning is enabled, `hyperparameter_tuning_results.csv`, `model_comparison_table.csv`, `comparison_trials_results.csv` (balancing, ablations, backbones and scratch CNN vs baseline), `pilot_comparison.png` (validation QWK and macro-F1 by trial group), `pilot_recall_heatmap.png`, and per trial `pilots/<trial_id>/history.csv`, `training_curves.png` and `validation_confusion_matrix.png`; phase learning-curve PNGs (accuracy and loss with augmented-train, clean-train-subset and validation lines, plus val_qwk) under the reports folder; phase history CSVs include `clean_train_loss`, `clean_train_accuracy`, `val_qwk` and `learning_rate`. Training logs and model summaries also appear inline. |
| Section 7: Evaluation | `test_predictions.npz`, `model_vs_majority_baseline.csv`, `test_loss_and_metrics.csv` (single test-loss number) and `test_decision_rule_summary.csv` (argmax vs validation-QWK thresholds: accuracy / QWK / macro-F1); `classification_report.csv`, `confusion_matrix.png`, `top_confusion_pairs.csv`, `classification_report_argmax.csv`, and `confusion_matrix_argmax.png`; `roc_auc_curves.png` and `roc_auc_summary.csv`; `per_class_recall_comparison.csv` and `per_class_recall_comparison.png`; `screening_metrics.csv` and `screening_confusion_matrices.png` (DR vs No DR, referable DR); validation calibration/class-weight evidence CSVs. Metric tables and error interpretations are also displayed inline. |
| Section 8: Explainability — Grad-CAM | `gradcam_correct_and_incorrect_validation.png`, showing validation images, heatmaps, and overlays. Quadrant descriptions and example details are printed inline. |
| Section 9: Similar-Case Retrieval | `artifacts/embeddings.npz` with the 256-D reference embeddings, labels, and file paths; `similar_cases_demo.png`; `retrieval_validation_metrics.csv`, `retrieval_validation_by_stage.csv`, and `retrieval_validation_comparison.png`. Retrieved cases and metric tables are also shown inline. |
| Section 10: Multi-Agent Decision Pipeline | Agent classes and pipeline outputs are demonstrated in notebook output. This section does not save a separate report image or CSV. |
| Section 11: Optional Gradio Interface | Builds the interface in memory and prints launch/status messages. The optional launch is commented out, so running all cells does not itself create a report file or start the web app. |
| Section 12: U-Net Pseudo-Mask Demonstration | `checkpoints/unet_pseudomask.weights.h5` and `three_layer_explainability_stack.png`; pseudo-mask examples, training metrics, and comparisons are displayed inline. The masks are synthetic targets, not expert annotations. The saved checkpoint name differs from the deployed app's configured `unet_lesion_best.weights.h5`. |
| Section 13: Research evidence | `report_images/error_analysis/error_cases.csv`, `report_images/error_analysis/error_analysis_summary.json`, and `report_images/reproducibility_manifest.json`; split CSVs are referenced by the manifest. This section creates analysis/evidence files and records configured ablation variants; model-level ablations are run by the Section 6 pilot runner. |

The exact run folder depends on the environment selected in the configuration cell. Keep artifacts from one run together so metrics, split manifests, checkpoints, and embeddings are not mixed across runs.

In [ ]:
# ============================================================
# SECTION 1.1 — INSTALL ONLY MISSING DEPENDENCIES
# Keep Kaggle's preinstalled TensorFlow, NumPy, and other packages
# intact; install only distributions whose import is unavailable.
# ============================================================

import importlib.util
import subprocess
import sys

_required_packages = {
    "kaggle": "kaggle",
    "sklearn": "scikit-learn",
    "seaborn": "seaborn",
    "PIL": "pillow",
    "cv2": "opencv-python-headless",
    "gradio": "gradio",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "pandas": "pandas",
    "numpy": "numpy",
    "tensorflow": "tensorflow",
    "imagehash": "imagehash",  # perceptual hashing for the duplicate-aware split
}
_missing_packages = [
    package
    for module, package in _required_packages.items()
    if importlib.util.find_spec(module) is None
]
if _missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *_missing_packages]
    )
else:
    print("[Dependencies] All required packages are already available.")

In [ ]:
# ============================================================
# SECTION 1.2 — GLOBAL CONFIGURATION (single source of truth)
# All magic numbers and tunable hyperparameters live here.
# Modify Config values rather than hunting through code.
#
# Dataset: APTOS 2019 Blindness Detection (Kaggle competition).
# Only the 3,662 labelled images in train_images/ are used; the
# competition's test_images/ folder has no public labels and is ignored.
# ============================================================

import os
import random
import datetime
from pathlib import Path
from typing import List, Optional
import numpy as np
import tensorflow as tf

_IN_KAGGLE = os.path.isdir("/kaggle/input") and os.path.isdir("/kaggle/working")
_IN_COLAB = False
_DRIVE_MOUNTED = False
if not _IN_KAGGLE:
    try:
        from google.colab import drive
        _IN_COLAB = True
        try:
            drive.mount("/content/drive")
            _DRIVE_MOUNTED = True
        except (NotImplementedError, OSError, ValueError) as exc:
            print(f"[Environment] Google Drive mount unavailable; using Colab temporary storage. ({exc})")
    except ImportError:
        pass

_PROJECT_ROOT = os.getcwd()
_APTOS_COMPETITION = "aptos2019-blindness-detection"


def _is_aptos_root(candidate: Path) -> bool:
    """True when a folder holds the APTOS train.csv and train_images/ folder."""
    return (candidate / "train.csv").is_file() and (candidate / "train_images").is_dir()


def _find_aptos_root() -> Optional[str]:
    """
    Locate the APTOS 2019 dataset root for the current environment.

    Order: APTOS_DATA_DIR environment variable, the standard Kaggle
    competition mount, any other attached Kaggle input with the same
    layout, then local/Colab folders. Returns None if nothing is found.
    """
    candidates = []
    if os.environ.get("APTOS_DATA_DIR"):
        candidates.append(Path(os.environ["APTOS_DATA_DIR"]))
    if _IN_KAGGLE:
        kaggle_input = Path("/kaggle/input")
        candidates.append(kaggle_input / _APTOS_COMPETITION)
        candidates.append(kaggle_input / "competitions" / _APTOS_COMPETITION)
        candidates.extend(path.parent for path in kaggle_input.glob("*/train.csv"))
        candidates.extend(path.parent for path in kaggle_input.glob("*/*/train.csv"))
    else:
        candidates.append(Path(_PROJECT_ROOT) / "data" / _APTOS_COMPETITION)
        candidates.append(Path(_PROJECT_ROOT) / "data" / "aptos2019")
        candidates.append(Path("/content") / _APTOS_COMPETITION)
    for candidate in candidates:
        if _is_aptos_root(candidate):
            return str(candidate)
    return None


_DATA_DIR = _find_aptos_root()
if _DATA_DIR is None:
    if _IN_KAGGLE:
        raise FileNotFoundError(
            "The APTOS 2019 Blindness Detection data is not attached. In the Kaggle editor use "
            "Add Input -> Competitions -> 'APTOS 2019 Blindness Detection' (join the competition and "
            "accept its rules first), then rerun this cell. Expected /kaggle/input/"
            f"{_APTOS_COMPETITION}/train.csv and train_images/."
        )
    # Colab / local: Section 1.5 downloads the competition files into this folder.
    _DATA_DIR = str(Path("/content" if _IN_COLAB else _PROJECT_ROOT) / "data" / _APTOS_COMPETITION)
print(f"[Dataset] APTOS 2019 root: {_DATA_DIR}")



class Config:
    """Central configuration for data, training, and output paths."""

    SEED: int = 42

    # Dataset paths (APTOS 2019 Blindness Detection, labelled training images only)
    KAGGLE_COMPETITION: str = _APTOS_COMPETITION
    DATA_DIR: str = _DATA_DIR
    LABELS_CSV: str = os.path.join(_DATA_DIR, "train.csv")          # id_code, diagnosis
    TRAIN_IMAGES_DIR: str = os.path.join(_DATA_DIR, "train_images")  # <id_code>.png
    EXPECTED_IMAGE_COUNT: int = 3662  # published size of the labelled APTOS training set

    # Image preprocessing
    IMG_SIZE: int = 300
    BEN_GRAHAM_SIGMA: int = 10
    BEN_GRAHAM_ALPHA: float = 4.0
    BEN_GRAHAM_BETA: float = -4.0
    BEN_GRAHAM_GAMMA: float = 128

    # Class labels (ICDR 0-4)
    CLASS_NAMES: list = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]
    NUM_CLASSES: int = 5

    # Duplicate-aware split (Section 4.1)
    TRAIN_RATIO: float = 0.70
    VAL_RATIO: float = 0.15
    TEST_RATIO: float = 0.15
    PHASH_SIZE: int = 256                 # images are cropped and resized to this before hashing
    DUPLICATE_HAMMING_THRESHOLD: int = 4  # 64-bit pHash distance <= 4 counts as a near-duplicate

    # Model and training hyperparameters (anti-overfitting defaults)
    BATCH_SIZE: int = 16
    PHASE1_EPOCHS: int = 8
    PHASE2_EPOCHS: int = 25
    PHASE1_LR: float = 1e-3
    PHASE2_LR: float = 1e-5
    WEIGHT_DECAY: float = 1e-4      # AdamW decoupled weight decay in Phase 2
    PHASE2_LR_SCHEDULE: str = "plateau"  # "plateau" (ReduceLROnPlateau) or "cosine" (warm-up + cosine)
    LR_WARMUP_EPOCHS: int = 2            # cosine mode: linear warm-up from 10% of PHASE2_LR
    LABEL_SMOOTHING: float = 0.1
    DROPOUT_RATE: float = 0.5
    DENSE_UNITS: int = 256
    UNFREEZE_TOP_N: int = 120       # fine-tune only the top 120 EfficientNetB3 layers (385 in Kaggle's Keras 3.13)
    USE_CLASS_WEIGHTS: bool = True  # class weights from the training split only
    CLASS_WEIGHT_MODE: str = "balanced"  # "balanced", "sqrt" (softer) or "none"; must agree with USE_CLASS_WEIGHTS
    USE_OVERSAMPLING: bool = False  # on-the-fly class-balanced sampling (never with class weights)

    # Final-fit callbacks (Sections 6.3 and 6.5): EarlyStopping and ModelCheckpoint follow
    # validation QWK; ReduceLROnPlateau follows validation loss. The 6.2b pilots stop on val_loss.
    STOP_MONITOR: str = "val_qwk"
    STOP_MODE: str = "max"
    STOP_PATIENCE: int = 6  # QWK on ~550 validation images is noisy, so allow more epochs than a val_loss rule
    RLROP_FACTOR: float = 0.5
    RLROP_PATIENCE: int = 2
    CLEAN_TRAIN_SUBSET_SIZE: int = 500  # fixed training subset for clean-train diagnostic curves

    # Pilot runner (Section 6.2b)
    # Skipped trials stay defined in 6.2b and are written to the tuning CSV as "skipped_by_config".
    # They were dropped after an earlier validation-only pilot showed they underperformed the
    # baseline or duplicated other evidence, to fit the compute budget. Remove an ID to re-enable it.
    PILOT_SKIP_TRIALS: List[str] = [
        "phase2_lr_3e5", "unfreeze_top240", "dropout_040", "ablation_all_filters", "backbone_vgg16",
    ]
    PILOT_TRIAL_LIMIT: Optional[int] = None         # e.g. 3 for a memory test: only the first N non-skipped trials
    PILOT_TIME_BUDGET_HOURS: Optional[float] = 5.0  # no new pilot trial starts after this many hours
    # Set to an earlier run ID (e.g. "20260930_114110") to reuse retinatrace_runs/run_<ID>
    # and its completed pilot trials instead of creating a new run folder.
    RESUME_RUN_ID: Optional[str] = None

    # Governance threshold
    CONFIDENCE_THRESHOLD: float = 0.70

    # Output paths (set below from the run folder)
    CHECKPOINT_DIR: str = ""
    REPORTS_DIR: str = ""
    EMBEDDINGS_PATH: str = ""


# Run folder: a new timestamped folder, or the earlier one named by Config.RESUME_RUN_ID.
_RUN_ID = Config.RESUME_RUN_ID or datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")
if _DRIVE_MOUNTED:
    _RUN_ROOT = os.path.join("/content/drive/MyDrive/colab_run_evidence", f"run_{_RUN_ID}")
elif _IN_KAGGLE:
    _RUN_ROOT = os.path.join("/kaggle/working/retinatrace_runs", f"run_{_RUN_ID}")
elif _IN_COLAB:
    _RUN_ROOT = os.path.join("/content/retinatrace_runs", f"run_{_RUN_ID}")
else:
    _RUN_ROOT = os.path.join(_PROJECT_ROOT, "retinatrace_runs", f"run_{_RUN_ID}")
if Config.RESUME_RUN_ID and not os.path.isdir(_RUN_ROOT):
    raise FileNotFoundError(
        f"Config.RESUME_RUN_ID={Config.RESUME_RUN_ID!r}, but {_RUN_ROOT} does not exist. "
        "Check the ID, or set RESUME_RUN_ID = None to start a new run."
    )

_CHECKPOINT_DIR = os.path.join(_RUN_ROOT, "checkpoints")
_REPORTS_DIR = os.path.join(_RUN_ROOT, "report_images")
_EMBEDDINGS_PATH = os.path.join(_RUN_ROOT, "artifacts", "embeddings.npz")
Config.CHECKPOINT_DIR = _CHECKPOINT_DIR
Config.REPORTS_DIR = _REPORTS_DIR
Config.EMBEDDINGS_PATH = _EMBEDDINGS_PATH
if Config.RESUME_RUN_ID:
    print(f"[Resume] Reusing run folder: {_RUN_ROOT}")
if _IN_COLAB:
    print(f"Colab run ID: {_RUN_ID}")
    print(f"Run evidence folder: {_RUN_ROOT}")
elif _IN_KAGGLE:
    print(f"Kaggle output folder: {_RUN_ROOT}")


def set_all_seeds(seed: int = Config.SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"[Config] All random seeds set to {seed}.")


set_all_seeds()


In [ ]:
# ============================================================
# SECTION 1.3 — IMPORT ALL LIBRARIES
# Centralised imports make missing-dependency errors easy to spot
# and prevent the notebook from failing midway through a long run.
# ============================================================

import sys
import glob
import shutil
import warnings
import pathlib
from typing import Any, Dict, List, Optional, Tuple, Union

import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError
import imagehash
from tqdm import tqdm
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

warnings.filterwarnings("ignore")

for _dir in [
    Config.CHECKPOINT_DIR,
    Config.REPORTS_DIR,
    Config.DATA_DIR,
    os.path.dirname(Config.EMBEDDINGS_PATH),
]:
    os.makedirs(_dir, exist_ok=True)

print(f"Python     : {sys.version}")
print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print("All libraries imported successfully.")

In [ ]:
# ============================================================
# SECTION 1.4 — GPU VERIFICATION
# Training EfficientNetB3 on CPU, even for 3,662 images, is very slow.
# We detect GPUs here and warn loudly if none is found.
# In Colab: Runtime > Change runtime type > GPU (T4 or A100).
# ============================================================

def verify_gpu() -> None:
    """
    Detect and report available GPU devices.

    Enables memory growth on each GPU to prevent TensorFlow from
    grabbing all VRAM at startup, which can cause OOM errors when
    sharing a Colab GPU with other processes.

    Prints a warning (but does not raise) if no GPU is found, so
    the notebook can still be used for small-batch testing on CPU.
    """
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            # Incremental VRAM allocation rather than reserving all memory up-front
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"[GPU] {len(gpus)} GPU(s) detected:")
        for g in gpus:
            print(f"      {g.name}")
        os.system("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
    else:
        print(
            "[WARNING] No GPU detected. Training will be very slow on CPU.\n"
            "          In Google Colab: Runtime > Change runtime type > GPU."
        )


verify_gpu()

In [ ]:
# ============================================================
# SECTION 1.5 — KAGGLE COMPETITION DATA (APTOS 2019)
# On Kaggle, attach the competition with Add Input -> Competitions ->
# "APTOS 2019 Blindness Detection"; it is mounted read-only, so no token
# is needed. On Colab/local runs the data is downloaded once with
#   kaggle competitions download -c aptos2019-blindness-detection
# which requires a kaggle.json token AND having accepted the competition
# rules on kaggle.com. Only train.csv and train_images/ are extracted.
# ============================================================

import zipfile


def setup_kaggle_credentials() -> None:
    """
    Place kaggle.json in ~/.kaggle/kaggle.json with correct permissions.

    The Kaggle CLI refuses to run if permissions on kaggle.json are too
    permissive (it is a security token). We enforce 600 (owner read/write).

    Raises:
        FileNotFoundError: If kaggle.json cannot be found or uploaded.
    """
    kaggle_dir = os.path.expanduser("~/.kaggle")
    os.makedirs(kaggle_dir, exist_ok=True)
    target = os.path.join(kaggle_dir, "kaggle.json")

    if os.path.exists(target):
        print("[Kaggle] kaggle.json already in place — skipping upload.")
        return

    try:
        from google.colab import files  # type: ignore
        print("[Kaggle] Please upload your kaggle.json when the dialog appears...")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError(
                "kaggle.json was not found in the uploaded files. "
                "Download it from https://www.kaggle.com/settings > API > Create New Token."
            )
        shutil.move("kaggle.json", target)
    except ImportError:
        if os.path.exists("kaggle.json"):
            shutil.copy("kaggle.json", target)
        else:
            raise FileNotFoundError(
                "kaggle.json not found in the current directory. "
                "Place it here to download the competition data with the Kaggle API."
            )

    os.chmod(target, 0o600)
    print(f"[Kaggle] Credentials saved to {target} with 600 permissions.")


def download_aptos_competition(data_dir: str = Config.DATA_DIR) -> None:
    """
    Download the APTOS 2019 competition archive and extract the labelled training data.

    Only train.csv and train_images/ are extracted; test_images/ has no public
    labels and is not used anywhere in this notebook.

    Args:
        data_dir: Destination folder (becomes Config.DATA_DIR).

    Raises:
        RuntimeError: If the Kaggle CLI fails (missing token or rules not accepted).
    """
    os.makedirs(data_dir, exist_ok=True)
    archive = os.path.join(data_dir, f"{Config.KAGGLE_COMPETITION}.zip")
    if not os.path.exists(archive):
        print(f"[Dataset] Downloading competition '{Config.KAGGLE_COMPETITION}' — this may take several minutes...")
        exit_code = os.system(
            f"kaggle competitions download -c {Config.KAGGLE_COMPETITION} -p {data_dir} -q"
        )
        if exit_code != 0 or not os.path.exists(archive):
            raise RuntimeError(
                f"Kaggle download failed (exit code {exit_code}). Check kaggle.json and make sure you have "
                "joined the APTOS 2019 competition and accepted its rules on kaggle.com."
            )
    with zipfile.ZipFile(archive) as bundle:
        members = [
            name for name in bundle.namelist()
            if name == "train.csv" or name.startswith("train_images/")
        ]
        bundle.extractall(data_dir, members=members)
    print(f"[Dataset] Extracted {len(members):,} files (train.csv + train_images/) -> {data_dir}")


if _is_aptos_root(Path(Config.DATA_DIR)):
    print(f"[Dataset] Using the APTOS 2019 data already available at {Config.DATA_DIR}; download skipped.")
else:
    setup_kaggle_credentials()
    download_aptos_competition(Config.DATA_DIR)
if not _is_aptos_root(Path(Config.DATA_DIR)):
    raise FileNotFoundError(f"APTOS train.csv/train_images/ not found under {Config.DATA_DIR}.")


In [ ]:
# ============================================================
# SECTION 1.6 — DISCOVER DATASET STRUCTURE
# Print the top-level directory tree so we know exactly what Kaggle
# delivered before attempting label loading.
# ============================================================

def print_directory_tree(root: str, max_depth: int = 3, max_items: int = 20) -> None:
    """
    Print a human-readable directory tree for quick structure inspection.

    Args:
        root:      Root directory to start from.
        max_depth: Maximum folder depth to recurse into.
        max_items: Max items shown per directory (prevents flooding output).
    """
    root_path = pathlib.Path(root)
    if not root_path.exists():
        print(f"[Tree] Path does not exist: {root}")
        return

    def _recurse(path: pathlib.Path, depth: int, prefix: str) -> None:
        if depth > max_depth:
            return
        try:
            children = sorted(path.iterdir())
        except PermissionError:
            return
        dirs  = [c for c in children if c.is_dir()]
        files = [c for c in children if c.is_file()]
        items = dirs + files
        truncated = len(items) > max_items
        items = items[:max_items]
        for i, item in enumerate(items):
            is_last = (i == len(items) - 1) and not truncated
            connector = "\u2514\u2500\u2500 " if is_last else "\u251c\u2500\u2500 "
            suffix = "/" if item.is_dir() else ""
            print(f"{prefix}{connector}{item.name}{suffix}")
            if item.is_dir():
                ext = "    " if is_last else "\u2502   "
                _recurse(item, depth + 1, prefix + ext)
        if truncated:
            print(f"{prefix}    ... (showing first {max_items} items)")

    print(f"\n[Tree] {root}/")
    _recurse(root_path, 1, "")


print_directory_tree(Config.DATA_DIR)

In [ ]:
# ============================================================
# SECTION 1.7 — LABEL LOADING (APTOS 2019 train.csv)
#
# train.csv has two columns:
#   id_code   -> image file name without extension (train_images/<id_code>.png)
#   diagnosis -> ICDR grade 0-4 assigned by clinicians (0 No DR ... 4 PDR)
#
# APTOS 2019 is a single-source dataset (Aravind Eye Hospital, India),
# already graded on the ICDR 0-4 scale, so no cross-dataset label mapping
# is required. APTOS provides no patient identifiers.
# ============================================================

def load_aptos_labels(
    labels_csv: str = Config.LABELS_CSV,
    image_dir: str = Config.TRAIN_IMAGES_DIR,
) -> pd.DataFrame:
    """
    Load the APTOS 2019 training labels and resolve each image path.

    Validates that every label is an ICDR grade in {0,1,2,3,4}, drops rows
    whose image file is missing (reporting how many), and warns if the row
    count differs from the published 3,662 labelled images.

    Args:
        labels_csv: Path to the competition train.csv.
        image_dir:  Folder containing <id_code>.png files.

    Returns:
        DataFrame with columns 'id_code', 'filepath' (absolute) and 'label' (int 0-4).
    """
    raw_df = pd.read_csv(labels_csv)
    missing_columns = {"id_code", "diagnosis"} - set(raw_df.columns)
    if missing_columns:
        raise ValueError(f"{labels_csv} is missing columns {sorted(missing_columns)}; expected id_code, diagnosis.")
    print(f"[Label] {len(raw_df):,} rows loaded from {labels_csv}")

    df = pd.DataFrame({
        "id_code": raw_df["id_code"].astype(str).str.strip(),
        "label": pd.to_numeric(raw_df["diagnosis"], errors="coerce"),
    })
    bad_mask = ~df["label"].isin({0, 1, 2, 3, 4})
    if bad_mask.any():
        raise ValueError(
            f"{int(bad_mask.sum())} rows have labels outside ICDR 0-4: "
            f"{df.loc[bad_mask, 'label'].unique().tolist()[:10]}"
        )
    if df["id_code"].duplicated().any():
        raise ValueError("train.csv contains repeated id_code values.")
    df["label"] = df["label"].astype(int)
    df["filepath"] = [
        str((Path(image_dir) / f"{id_code}.png").resolve()) for id_code in df["id_code"]
    ]

    exists_mask = df["filepath"].map(os.path.isfile)
    if not exists_mask.all():
        print(f"[Label] WARNING: {int((~exists_mask).sum())} labelled image(s) not found on disk. Removing.")
        df = df[exists_mask].reset_index(drop=True)
    if len(df) != Config.EXPECTED_IMAGE_COUNT:
        print(
            f"[Label] WARNING: {len(df):,} usable images, but the APTOS 2019 training set has "
            f"{Config.EXPECTED_IMAGE_COUNT:,}. Check that the full competition data is attached."
        )
    print(f"[Label] Final usable dataset size: {len(df):,} images.")
    return df[["id_code", "filepath", "label"]]


df_labels = load_aptos_labels()
df_labels.head()


In [ ]:
# ============================================================
# SECTION 1.8 — CORRUPT IMAGE DETECTION
# Corrupt images (truncated, zero-byte, unreadable) crash the pipeline
# or silently produce garbage. PIL.Image.verify() reads only the header
# and raises on corrupt files — much faster than full pixel decoding.
# ============================================================

def filter_corrupt_images(
    df: pd.DataFrame, filepath_col: str = "filepath"
) -> pd.DataFrame:
    """
    Remove rows referencing corrupt or unreadable image files.

    Uses PIL.Image.verify() which reads image metadata without decoding
    pixel data. This catches JPEG truncation and file-format errors that
    OpenCV silently ignores (returning a black or partial image instead).

    Args:
        df:           DataFrame with a file path column.
        filepath_col: Name of the file path column.

    Returns:
        DataFrame with corrupt-image rows removed.
    """
    corrupt_indices = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Checking image integrity"):
        try:
            with Image.open(row[filepath_col]) as img:
                img.verify()  # header-only check — does not decode pixel data
        except (UnidentifiedImageError, OSError, SyntaxError):
            # SyntaxError is raised by PIL for some truncated JPEG files
            corrupt_indices.append(idx)

    if corrupt_indices:
        examples = df.loc[corrupt_indices, filepath_col].tolist()[:5]
        print(f"[Corrupt] {len(corrupt_indices)} corrupt image(s) found and removed. Examples:")
        for p in examples:
            print(f"  {p}")
        df = df.drop(index=corrupt_indices).reset_index(drop=True)
    else:
        print("[Corrupt] All images passed integrity check — dataset is clean.")

    return df


df_labels = filter_corrupt_images(df_labels)
print(f"Dataset size after integrity check: {len(df_labels):,} images.")

In [ ]:
# ============================================================
# SECTION 1.9 — CLASS DISTRIBUTION SUMMARY & VISUALISATION
# Print a per-class count table and plot a bar chart using the
# APTOS 2019 training labels.
# ============================================================

def print_class_distribution(df: pd.DataFrame, label_col: str = "label") -> pd.DataFrame:
    """Compute and print a class distribution summary table."""
    counts = df[label_col].value_counts().sort_index()
    total = counts.sum()

    summary = pd.DataFrame({
        "class_id": counts.index,
        "class_name": [Config.CLASS_NAMES[i] for i in counts.index],
        "count": counts.values,
        "percentage": (counts.values / total * 100).round(2),
    })

    print("\n" + "=" * 58)
    print(f"  CLASS DISTRIBUTION  (total: {total:,} images)")
    print("=" * 58)
    print(summary.to_string(index=False))
    print("-" * 58)
    ratio = counts.max() / counts.min()
    print(f"  Imbalance ratio (max / min class): {ratio:.1f}x")
    if ratio > 1:
        print(f"  [ACTION] Class weighting remains useful for the observed {ratio:.1f}x imbalance.")
    print("=" * 58 + "\n")
    return summary


def plot_class_distribution(
    df: pd.DataFrame,
    label_col: str = "label",
    save_path: Optional[str] = None,
) -> None:
    """Plot and optionally save a class count bar chart."""
    counts = df[label_col].value_counts().reindex(range(Config.NUM_CLASSES), fill_value=0)

    fig, ax = plt.subplots(figsize=(9, 5))
    palette = sns.color_palette("Blues_d", len(Config.CLASS_NAMES))
    bars = ax.bar(
        Config.CLASS_NAMES,
        counts.values,
        color=palette,
        edgecolor="black",
        linewidth=0.7,
    )
    for bar, count in zip(bars, counts.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts.max() * 0.01, 1),
            f"{count:,}",
            ha="center", va="bottom", fontsize=10, fontweight="bold",
        )

    ax.set_title("Class Distribution — APTOS 2019 (labelled training images)", fontsize=14, pad=15)
    ax.set_xlabel("DR Stage (ICDR 0-4 Scale)", fontsize=12)
    ax.set_ylabel("Number of Images", fontsize=12)
    ax.set_ylim(0, counts.max() * 1.15)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[Plot] Saved -> {save_path}")

    plt.show()


class_summary = print_class_distribution(df_labels)
plot_class_distribution(
    df_labels,
    save_path=os.path.join(Config.REPORTS_DIR, "class_distribution.png"),
)

In [ ]:
# ============================================================
# SECTION 1.10 — IMAGE DIMENSION AUDIT
# Report the native image resolution before resizing. APTOS 2019
# images were captured on different cameras and vary widely in size.
# We sample 500 images to report the range so the preprocessing section
# is designed with accurate expectations.
# ============================================================

def audit_image_dimensions(
    df: pd.DataFrame,
    sample_n: int = 500,
    filepath_col: str = "filepath",
) -> None:
    """
    Sample images and report width/height statistics.

    Reads only the image header (not full pixel data) via PIL, so
    auditing 500 images takes only a few seconds regardless of resolution.

    Args:
        df:           DataFrame with filepath column.
        sample_n:     Number of images to randomly sample.
        filepath_col: Name of the file path column.
    """
    sample = df.sample(min(sample_n, len(df)), random_state=Config.SEED)
    widths, heights = [], []

    for fp in tqdm(sample[filepath_col], desc="Auditing dimensions"):
        try:
            with Image.open(fp) as img:
                w, h = img.size  # PIL: (width, height)
                widths.append(w)
                heights.append(h)
        except Exception:
            pass  # already screened by corrupt filter

    w = np.array(widths)
    h = np.array(heights)
    print(f"\n[Dimensions] Sampled {len(w)} images:")
    print(f"  Width  — min:{w.min():6d}  max:{w.max():6d}  mean:{w.mean():.0f}")
    print(f"  Height — min:{h.min():6d}  max:{h.max():6d}  mean:{h.mean():.0f}")
    print(f"  All images will be resized to {Config.IMG_SIZE}x{Config.IMG_SIZE} in Section 2.")


audit_image_dimensions(df_labels)

# ---- Section 1 complete ----
print("\n" + "=" * 60)
print("  SECTION 1 COMPLETE")
print(f"  Total images loaded and validated : {len(df_labels):,}")
print(f"  Label column                      : 'label'    (int, ICDR 0-4)")
print(f"  Image path column                 : 'filepath' (absolute path)")
print("  Next -> Section 2: Preprocessing")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 1.10b - RESOLUTION VS DR STAGE
#
# WHY: APTOS 2019 images were collected over time at several clinics
# on different fundus cameras. If one camera contributes mostly one
# DR stage, image resolution becomes
# correlated with the label. A CNN can then learn a shortcut - "large
# or wide image => severe stage" - instead of the retinal lesions.
# This cell measures whether native resolution differs by stage.
#
# Descriptive only: it samples a COPY of df_labels, reads image
# headers only, and never filters images or changes any split.
# ============================================================

from scipy.stats import kruskal

RESOLUTION_SAMPLE_PER_CLASS: int = 200
STAGE_COLORS: List[str] = ["#2E7D32", "#0284C7", "#D97706", "#EA580C", "#BE123C"]


def sample_per_class(
    df: pd.DataFrame,
    n_per_class: int = RESOLUTION_SAMPLE_PER_CLASS,
    label_col: str = "label",
) -> pd.DataFrame:
    """
    Draw up to n_per_class rows from every DR stage so small stages are represented.

    Classes with fewer than n_per_class images contribute all of their rows.
    The input DataFrame is not modified.

    Args:
        df:          DataFrame with filepath and label columns.
        n_per_class: Maximum rows sampled per class.
        label_col:   Name of the integer label column.

    Returns:
        A new DataFrame containing the sampled rows.
    """
    parts = []
    for class_id in range(Config.NUM_CLASSES):
        class_rows = df[df[label_col] == class_id]
        if len(class_rows):
            parts.append(class_rows.sample(min(n_per_class, len(class_rows)), random_state=Config.SEED))
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[0:0].copy()


def read_image_resolutions(
    sample: pd.DataFrame,
    filepath_col: str = "filepath",
    label_col: str = "label",
) -> Tuple[pd.DataFrame, int]:
    """
    Read width and height from each image header without decoding pixels.

    Args:
        sample:       DataFrame of sampled images.
        filepath_col: Name of the file path column.
        label_col:    Name of the integer label column.

    Returns:
        (per-image resolution table, number of unreadable files skipped)
    """
    records = []
    skipped = 0
    for filepath, label in tqdm(
        zip(sample[filepath_col], sample[label_col]),
        total=len(sample),
        desc="Reading image headers",
    ):
        try:
            with Image.open(filepath) as img:
                width, height = img.size  # header only; pixels are not decoded
        except (OSError, UnidentifiedImageError, ValueError):
            skipped += 1
            continue
        records.append({
            "filepath": filepath,
            "label": int(label),
            "class_name": Config.CLASS_NAMES[int(label)],
            "width": int(width),
            "height": int(height),
            "megapixels": width * height / 1e6,
            "aspect_ratio": width / height,
        })
    columns = ["filepath", "label", "class_name", "width", "height", "megapixels", "aspect_ratio"]
    return pd.DataFrame(records, columns=columns), skipped


def summarise_resolution_by_class(resolution_df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarise native resolution per DR stage.

    Args:
        resolution_df: Per-image table from read_image_resolutions().

    Returns:
        One row per class with count, width/height, megapixel and aspect-ratio statistics.
    """
    summary = (
        resolution_df.groupby("label")
        .agg(
            count=("filepath", "size"),
            median_width=("width", "median"),
            min_width=("width", "min"),
            max_width=("width", "max"),
            median_height=("height", "median"),
            median_megapixels=("megapixels", "median"),
            median_aspect_ratio=("aspect_ratio", "median"),
        )
        .reindex(range(Config.NUM_CLASSES))
    )
    summary["count"] = summary["count"].fillna(0).astype(int)
    summary.insert(0, "class_name", [Config.CLASS_NAMES[c] for c in summary.index])
    summary.index.name = "label"
    return summary.reset_index()


def plot_resolution_by_class(resolution_df: pd.DataFrame, save_path: str) -> None:
    """
    Save a two-panel figure: (a) megapixels per DR stage, (b) width vs height by stage.

    Args:
        resolution_df: Per-image table from read_image_resolutions().
        save_path:     PNG output path.
    """
    present = [c for c in range(Config.NUM_CLASSES) if (resolution_df["label"] == c).any()]
    fig, (ax_box, ax_scatter) = plt.subplots(1, 2, figsize=(15, 6))

    box = ax_box.boxplot(
        [resolution_df.loc[resolution_df["label"] == c, "megapixels"] for c in present],
        patch_artist=True,
        widths=0.6,
        medianprops={"color": "black", "linewidth": 1.5},
        flierprops={"marker": "o", "markersize": 3, "alpha": 0.5},
    )
    for patch, class_id in zip(box["boxes"], present):
        patch.set_facecolor(STAGE_COLORS[class_id])
        patch.set_alpha(0.75)
    ax_box.set_xticks(range(1, len(present) + 1), [Config.CLASS_NAMES[c] for c in present], rotation=15)
    ax_box.set_title("(a) Native Resolution per DR Stage", fontweight="bold")
    ax_box.set_xlabel("Diabetic retinopathy stage")
    ax_box.set_ylabel("Megapixels (width x height / 1e6)")
    ax_box.grid(axis="y", alpha=0.3)

    # Fundus images come in a few fixed sizes, so points from different stages often share
    # exactly the same (width, height). Hollow rings of decreasing size per stage keep every
    # stage visible as nested rings instead of the last-drawn stage hiding the others.
    for class_id in present:
        rows = resolution_df[resolution_df["label"] == class_id]
        ax_scatter.scatter(
            rows["width"], rows["height"], s=190 - 35 * class_id, facecolors="none",
            edgecolors=STAGE_COLORS[class_id], linewidths=1.6, alpha=0.8,
            label=Config.CLASS_NAMES[class_id],
        )
    ax_scatter.set_title("(b) Image Width vs Height by DR Stage", fontweight="bold")
    ax_scatter.set_xlabel("Width (pixels)")
    ax_scatter.set_ylabel("Height (pixels)")
    ax_scatter.legend(title="DR stage (nested rings = same size)")
    ax_scatter.grid(alpha=0.3)

    fig.suptitle(
        f"Resolution vs DR Stage (up to {RESOLUTION_SAMPLE_PER_CLASS} images per stage; "
        f"all resized to {Config.IMG_SIZE}x{Config.IMG_SIZE} for training)",
        fontsize=12,
    )
    fig.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


resolution_sample = sample_per_class(df_labels)
resolution_by_class, n_resolution_skipped = read_image_resolutions(resolution_sample)
resolution_summary = summarise_resolution_by_class(resolution_by_class)
print(f"\n[Resolution] Read {len(resolution_by_class):,} image headers; skipped {n_resolution_skipped} unreadable file(s).")

resolution_csv_path = os.path.join(Config.REPORTS_DIR, "resolution_by_class.csv")
resolution_summary_path = os.path.join(Config.REPORTS_DIR, "resolution_by_class_summary.csv")
resolution_plot_path = os.path.join(Config.REPORTS_DIR, "resolution_by_class.png")
resolution_by_class.to_csv(resolution_csv_path, index=False)
resolution_summary.to_csv(resolution_summary_path, index=False)

print("\n[Resolution] Per-class summary (native image size before resizing):")
print(resolution_summary.to_string(
    index=False,
    formatters={
        "median_width": "{:.0f}".format,
        "min_width": "{:.0f}".format,
        "max_width": "{:.0f}".format,
        "median_height": "{:.0f}".format,
        "median_megapixels": "{:.2f}".format,
        "median_aspect_ratio": "{:.3f}".format,
    },
    na_rep="-",
))

plot_resolution_by_class(resolution_by_class, resolution_plot_path)

# Association check: does megapixel distribution differ between stages?
megapixel_groups = [
    resolution_by_class.loc[resolution_by_class["label"] == c, "megapixels"].to_numpy()
    for c in range(Config.NUM_CLASSES)
]
megapixel_groups = [group for group in megapixel_groups if len(group)]
try:
    kruskal_stat, kruskal_p = kruskal(*megapixel_groups)
except ValueError:
    # scipy raises when every value is identical: no difference between stages at all.
    kruskal_stat, kruskal_p = 0.0, 1.0
class_medians = resolution_summary["median_megapixels"].dropna()
median_ratio = float(class_medians.max() / class_medians.min())

print(f"\n[Resolution] Kruskal-Wallis on megapixels across {len(megapixel_groups)} stages: "
      f"H = {kruskal_stat:.2f}, p-value = {kruskal_p:.3g}")
print(f"[Resolution] Largest / smallest per-class median megapixels: {median_ratio:.2f}x")
if kruskal_p < 0.05 and median_ratio > 1.5:
    print(
        "[Resolution] Resolution differs noticeably between stages — possible source/camera bias; "
        "the model could partly learn resolution instead of lesions. Resizing to "
        f"Config.IMG_SIZE ({Config.IMG_SIZE}x{Config.IMG_SIZE}) reduces but may not remove this; "
        "discuss as a limitation."
    )
else:
    print(
        "[Resolution] No strong resolution difference between stages; "
        "resolution is unlikely to be a shortcut feature."
    )
print(f"[Resolution] Saved -> {resolution_csv_path}")
print(f"[Resolution] Saved -> {resolution_summary_path}")
print(f"[Resolution] Saved -> {resolution_plot_path}")


## Section 2: Preprocessing

**Goal:** Transform raw fundus photographs into clean, normalised tensors at `Config.IMG_SIZE` (300×300 by default).

**Default training pipeline** (applied to every image, in order):
1. **Black-border crop** — fundus images are circular; the black padding carries no information and wastes model capacity.
2. **Resize** to `Config.IMG_SIZE` square pixels (area interpolation when shrinking).
3. **Ben Graham enhancement** — subtracts a heavily Gaussian-blurred copy of the image to remove uneven illumination and boost local lesion/vessel contrast.
4. **Normalise** pixel values to [0, 1] (the model's input adapter rescales to what EfficientNet expects).

**Optional filters** available in `core.preprocessing` and shown in the Section 2.5b gallery: bilateral **denoising**, **CLAHE** (contrast-limited adaptive histogram equalisation on the LAB L-channel) and **unsharp-mask edge enhancement**. They are off by default; Section 6 trains pilot models with them switched on (and with Ben Graham switched off) to measure whether each one actually helps.

Before/after figures, histograms and edge/morphology maps are saved to `Config.REPORTS_DIR`.

In [ ]:
# ============================================================
# SECTION 2.1 — IMAGE LOADING UTILITY
# A single function that loads an image file into an RGB NumPy
# array, used as the entry point for every preprocessing step.
# ============================================================

def load_image_rgb(filepath: str) -> np.ndarray:
    """
    Load an image from disk and convert to an RGB NumPy array.

    Uses OpenCV (fast, handles many formats) and converts BGR -> RGB
    so colours are correct for matplotlib and TensorFlow.

    Args:
        filepath: Absolute path to the image file.

    Returns:
        NumPy array of shape (H, W, 3), dtype uint8, RGB channel order.

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError:        If OpenCV fails to decode the image.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Image not found: {filepath}")

    img_bgr = cv2.imread(filepath)
    if img_bgr is None:
        raise ValueError(
            f"OpenCV could not decode image: {filepath}. "
            "The file may be corrupt or in an unsupported format."
        )
    # OpenCV loads as BGR; convert to RGB for compatibility with PIL and TF
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
# ============================================================
# SECTION 2.2 — BLACK-BORDER CROPPING
#
# WHY: Fundus cameras capture a circular retinal image centred
# on a rectangular sensor. The corners of the photograph are
# pure black (pixel value ≈ 0). This padding:
#   - Wastes model capacity on uninformative pixels
#   - Can confuse augmentation (random crops may include mostly black)
#   - Inflates the apparent image size
#
# APPROACH (Ben Graham / crop_image_from_gray style):
#   1. Convert to grayscale.
#   2. Apply a binary threshold at a low value (e.g. 7) to create a
#      mask that is 1 where the retina is and 0 where it is black.
#   3. Find the bounding box of the non-zero mask region.
#   4. Crop the original colour image to that bounding box.
#   5. If the crop is degenerate (all black image), return the original.
# ============================================================

def crop_image_from_gray(
    img: np.ndarray,
    threshold: int = 7,
    tol: int = 7,
) -> np.ndarray:
    """
    Remove black borders from a fundus photograph.

    The retinal fundus image is circular; surrounding pixels are near-black
    (value < threshold). We threshold the grayscale image to find the retinal
    disc boundary and crop the colour image to that bounding box.

    This is commonly called the "crop_image_from_gray" technique, popularised
    by Ben Graham's APTOS competition solution.

    Args:
        img:       RGB NumPy array (H, W, 3), uint8.
        threshold: Grayscale pixel value below which a pixel is considered
                   background (black border). Default 7 is empirically robust
                   across APTOS 2019 fundus photographs.
        tol:       Additional tolerance in pixels to expand the crop slightly
                   and avoid clipping the very edge of the optic disc.

    Returns:
        Cropped RGB NumPy array. Falls back to the original if the crop is
        degenerate (e.g. an entirely black image).
    """
    if img.ndim == 2:
        # Already grayscale — work directly
        mask = img > threshold
        if not mask.any():
            return img  # guard: entirely black image
        row_mask = mask.any(axis=1)
        col_mask = mask.any(axis=0)
        rmin, rmax = np.where(row_mask)[0][[0, -1]]
        cmin, cmax = np.where(col_mask)[0][[0, -1]]
        # Add tolerance so we do not clip the disc edge
        rmin = max(0, rmin - tol)
        rmax = min(img.shape[0] - 1, rmax + tol)
        cmin = max(0, cmin - tol)
        cmax = min(img.shape[1] - 1, cmax + tol)
        return img[rmin:rmax + 1, cmin:cmax + 1]

    # For an RGB image, compute the grayscale mask from a single channel
    # (green channel has best contrast for retinal vessels)
    gray = img[:, :, 1]
    mask = gray > threshold

    if not mask.any():
        return img  # guard: entirely black or near-black image

    row_mask = mask.any(axis=1)
    col_mask = mask.any(axis=0)
    rmin, rmax = np.where(row_mask)[0][[0, -1]]
    cmin, cmax = np.where(col_mask)[0][[0, -1]]

    rmin = max(0, rmin - tol)
    rmax = min(img.shape[0] - 1, rmax + tol)
    cmin = max(0, cmin - tol)
    cmax = min(img.shape[1] - 1, cmax + tol)

    cropped = img[rmin:rmax + 1, cmin:cmax + 1]

    # Sanity check: if the crop is nearly empty, return the original
    if cropped.size == 0 or min(cropped.shape[:2]) < 10:
        return img

    return cropped

In [ ]:
# ============================================================
# SECTION 2.3 — BEN GRAHAM CONTRAST ENHANCEMENT
#
# WHY: Fundus photographs suffer from uneven illumination — the
# centre of the image is often brighter than the periphery due to
# the ophthalmoscope light path. This hides subtle lesion contrast
# (microaneurysms, haemorrhages, exudates) in darker regions.
#
# THE TECHNIQUE: Subtract a heavily blurred (low-frequency) version
# of the image from the original, then re-centre the distribution:
#
#   enhanced = alpha * original + beta * GaussianBlur(original, sigma) + gamma
#
# With alpha=4, beta=-4, gamma=128:
#   - The blurred image captures the slow illumination gradient
#   - Subtracting it removes that gradient (local contrast normalisation)
#   - gamma=128 re-centres pixel values near the middle of [0,255]
#     so we do not clip large negative or positive residuals
#   - alpha=4 amplifies fine-detail (vessel edges, lesion boundaries)
#
# This is NOT the same as CLAHE (which operates on histograms of tiles);
# Ben Graham enhancement operates globally in the spatial domain and
# specifically targets the illumination gradient rather than histogram spread.
# It was used by the winning solution of the 2015 Kaggle DR competition.
# ============================================================

def ben_graham_enhance(
    img: np.ndarray,
    sigma: int = Config.BEN_GRAHAM_SIGMA,
    alpha: float = Config.BEN_GRAHAM_ALPHA,
    beta: float  = Config.BEN_GRAHAM_BETA,
    gamma: float = Config.BEN_GRAHAM_GAMMA,
) -> np.ndarray:
    """
    Apply Ben Graham-style contrast enhancement to a fundus image.

    Subtracts a Gaussian-blurred version of the image from itself to
    remove the slow illumination gradient present in fundus photography,
    then re-centres pixel values with an additive bias.

    Formula: enhanced = clip(alpha * img + beta * blur(img, sigma) + gamma)

    Args:
        img:   RGB uint8 NumPy array (H, W, 3).
        sigma: Gaussian blur kernel sigma. Large sigma (default 10) captures
               the broad illumination gradient across the whole image.
        alpha: Weight on the original image. Value of 4 amplifies fine detail.
        beta:  Weight on the blurred image (typically -alpha to subtract it).
        gamma: Additive constant to re-centre the output around 128.

    Returns:
        Enhanced RGB uint8 NumPy array, same shape as input.
    """
    # Build the blur kernel size from sigma: must be odd and positive
    # ksize = 2 * (4 * sigma) + 1 is the standard "full-width" Gaussian kernel
    ksize = int(2 * round(4 * sigma) + 1)

    # Gaussian blur captures the low-frequency illumination pattern
    blurred = cv2.GaussianBlur(img, (ksize, ksize), sigma)

    # Weighted addition: original (amplified) minus blur (background), re-centred
    enhanced = cv2.addWeighted(img, alpha, blurred, beta, gamma)

    # clip is implicit in addWeighted for uint8, but we apply it explicitly
    # to avoid artefacts if the input dtype differs
    enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
    return enhanced

In [ ]:
# ============================================================
# SECTION 2.4 — FULL PREPROCESSING PIPELINE (Unified with core/)
# Reusable preprocessing pipeline imported from core.preprocessing
# ensuring a single source of truth across research notebook,
# standalone GPU training, and the clinical web application.
# ============================================================

import subprocess
import sys
import inspect  # CHANGED: print the active wrapper source for resolution verification
from pathlib import Path

_PROJECT_ROOT = Path.cwd()
if not (_PROJECT_ROOT / "core").is_dir():
    if _IN_KAGGLE:
        _PROJECT_ROOT = Path("/kaggle/working/ComputerVision_Coursework")
    else:
        _PROJECT_ROOT = Path("/content/ComputerVision_Coursework")
    if not (_PROJECT_ROOT / "core").is_dir():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/ShaznaSalman/ComputerVision_Coursework_COBSCCOMP24.2P-019.git",
                str(_PROJECT_ROOT),
            ],
            check=True,
        )

if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from core.preprocessing import (
    crop_image_from_gray,
    ben_graham_enhance,
    preprocess_image as _core_preprocess_image,
)


def preprocess_image(image_input, img_size=None, **kwargs):
    target_size = Config.IMG_SIZE if img_size is None else img_size
    return _core_preprocess_image(image_input, img_size=target_size, **kwargs)


print("[Preprocessing] Active notebook wrapper source:")  # CHANGED: expose resolution forwarding for verification
print(inspect.getsource(preprocess_image))  # CHANGED: verify Config.IMG_SIZE is passed to core preprocessing

_test_fp = df_labels["filepath"].iloc[0]
_test_out = preprocess_image(_test_fp)
print("Preprocessing check:")
print(f"  Project root: {_PROJECT_ROOT}")
print(f"  Input file  : {_test_fp}")
print(f"  Output shape: {_test_out.shape}")
print(f"  Dtype       : {_test_out.dtype}")
print(f"  Value range : [{_test_out.min():.3f}, {_test_out.max():.3f}]")

In [ ]:
# ============================================================
# SECTION 2.5 — BEFORE / AFTER COMPARISON VISUALISATION
# Saves side-by-side comparison grids for the report.
# We show: (a) original raw image, (b) after crop+resize,
# (c) after Ben Graham enhancement. One grid per DR class.
# ============================================================

def visualise_preprocessing(
    df: pd.DataFrame,
    n_per_class: int = 2,
    save_dir: str = Config.REPORTS_DIR,
) -> None:
    """
    Generate and save before/after preprocessing comparison figures.

    For each DR class, randomly samples n_per_class images and renders
    a 3-column grid showing: raw, cropped+resized, and fully enhanced.
    Figures are saved to save_dir for inclusion in the report.

    Args:
        df:           DataFrame with 'filepath' and 'label' columns.
        n_per_class:  Number of example images per class to display.
        save_dir:     Directory where PNG comparison files are saved.
    """
    os.makedirs(save_dir, exist_ok=True)

    for class_id, class_name in enumerate(Config.CLASS_NAMES):
        class_df = df[df["label"] == class_id]
        if class_df.empty:
            print(f"[Preprocess] WARNING: No images found for class {class_id} ({class_name}). Skipping.")
            continue

        # Sample up to n_per_class images (fewer if the class is small)
        sample = class_df.sample(min(n_per_class, len(class_df)), random_state=Config.SEED)

        n_cols = 3   # raw | cropped+resized | Ben Graham enhanced
        n_rows = len(sample)
        fig, axes = plt.subplots(
            n_rows, n_cols,
            figsize=(n_cols * 4, n_rows * 4),
        )
        # Ensure axes is always 2D even for n_rows=1
        if n_rows == 1:
            axes = axes[np.newaxis, :]

        col_titles = ["Original (raw)", "Cropped + Resized", "Ben Graham Enhanced"]
        for ax, title in zip(axes[0], col_titles):
            ax.set_title(title, fontsize=10, fontweight="bold")

        for row_idx, (_, row) in enumerate(sample.iterrows()):
            fp = row["filepath"]

            # Column 0: original raw image
            raw = load_image_rgb(fp)
            axes[row_idx, 0].imshow(raw)
            axes[row_idx, 0].axis("off")

            # Column 1: cropped and resized (no Ben Graham)
            cropped = crop_image_from_gray(raw)
            resized = cv2.resize(
                cropped,
                (Config.IMG_SIZE, Config.IMG_SIZE),
                interpolation=cv2.INTER_AREA if (
                    cropped.shape[0] > Config.IMG_SIZE
                ) else cv2.INTER_LINEAR,
            )
            axes[row_idx, 1].imshow(resized)
            axes[row_idx, 1].axis("off")

            # Column 2: fully processed (Ben Graham applied to the resized image)
            enhanced = ben_graham_enhance(resized)
            axes[row_idx, 2].imshow(enhanced)
            axes[row_idx, 2].axis("off")

        fig.suptitle(
            f"Preprocessing Steps — Class {class_id}: {class_name}",
            fontsize=13,
            fontweight="bold",
            y=1.01,
        )
        plt.tight_layout()

        save_path = os.path.join(save_dir, f"preprocessing_class{class_id}.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"[Preprocess] Saved comparison figure -> {save_path}")


visualise_preprocessing(df_labels, n_per_class=2)

# ---- Also save a combined single-row comparison (all 5 classes side by side) ----
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2 as _cv2
import numpy as _np

_fig_comp, _axes_comp = plt.subplots(2, 5, figsize=(20, 8))
_class_colors = ["#27ae60", "#f39c12", "#e67e22", "#e74c3c", "#8e44ad"]
for _cls in range(5):
    _cls_df = df_labels[df_labels["label"] == _cls]
    _sample_row = _cls_df.sample(1, random_state=Config.SEED + _cls).iloc[0]
    _raw = load_image_rgb(_sample_row["filepath"])
    _cropped = crop_image_from_gray(_raw)
    _resized = _cv2.resize(_cropped, (Config.IMG_SIZE, Config.IMG_SIZE))
    _enhanced = ben_graham_enhance(_resized)

    _axes_comp[0, _cls].imshow(_resized)
    _axes_comp[0, _cls].axis("off")
    _axes_comp[0, _cls].set_title(
        Config.CLASS_NAMES[_cls], color=_class_colors[_cls], fontsize=10, fontweight="bold"
    )
    if _cls == 0:
        _axes_comp[0, _cls].set_ylabel("Before Enhancement", fontsize=9)

    _axes_comp[1, _cls].imshow(_enhanced)
    _axes_comp[1, _cls].axis("off")
    if _cls == 0:
        _axes_comp[1, _cls].set_ylabel("After Ben Graham", fontsize=9)

_fig_comp.suptitle(
    "Ben Graham Preprocessing: Before vs After — All 5 DR Stages",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
_comp_path = os.path.join(Config.REPORTS_DIR, "preprocessing_comparison.png")
plt.savefig(_comp_path, dpi=150, bbox_inches="tight")
plt.close(_fig_comp)
print(f"[Preprocess] Saved combined comparison -> {_comp_path}")


### 2.5b Step-by-step preprocessing, histograms, edges and morphology

The next cell visualises every preprocessing operation on one image with the intensity histogram of the retinal disc under each step, then applies classical convolution (Sobel, Canny) and morphological operators (opening, top-hat, black-hat) to the green channel. These link the pipeline to histogram operations, convolution/edge detection and morphology, and show which retinal structures each filter emphasises.

In [ ]:
# ============================================================
# SECTION 2.5b — STEP-BY-STEP PREPROCESSING GALLERY, HISTOGRAMS,
#                EDGE DETECTION AND MORPHOLOGY
#
# Figure 1 follows one fundus image through every available filter:
#   raw -> crop + resize -> bilateral denoise -> CLAHE -> unsharp-mask
#   edge enhancement, alongside Ben Graham (the default training step).
#   The row underneath shows the grey-level and green-channel histogram
#   of the retinal disc for each step, so the effect of each operation on
#   the intensity distribution is visible (histogram operations).
#
# Figure 2 applies classical convolution and morphology operators to the
# green channel, which carries the most vessel/lesion contrast:
#   Sobel gradient magnitude, Canny edges, morphological opening of the
#   retina mask, black-hat (dark structures: vessels, haemorrhages,
#   microaneurysms) and top-hat (bright structures: exudates).
#
# Denoise, CLAHE and edge enhancement are OFF in the default training
# pipeline; Section 6 trains pilot models with them ON to test whether
# they help (preprocessing ablation). All visuals are descriptive only;
# nothing here is fitted to data, so there is no leakage.
# ============================================================

from collections import OrderedDict
from core.preprocessing import apply_clahe, denoise_fundus, enhance_edges


def _resize_to_config(image: np.ndarray) -> np.ndarray:
    interpolation = cv2.INTER_AREA if image.shape[0] > Config.IMG_SIZE else cv2.INTER_LINEAR
    return cv2.resize(image, (Config.IMG_SIZE, Config.IMG_SIZE), interpolation=interpolation)


def build_preprocessing_stages(filepath: str) -> "OrderedDict[str, np.ndarray]":
    """Return each preprocessing stage of one image as uint8 RGB arrays."""
    raw = load_image_rgb(filepath)
    resized = _resize_to_config(crop_image_from_gray(raw))
    denoised = denoise_fundus(resized)
    clahe_image = apply_clahe(denoised)
    sharpened = enhance_edges(clahe_image)
    return OrderedDict([
        ("1. Raw photograph", raw),
        ("2. Crop borders + resize", resized),
        ("3. Bilateral denoise", denoised),
        ("4. CLAHE (LAB L-channel)", clahe_image),
        ("5. Unsharp-mask edges", sharpened),
        ("Ben Graham (training default)", ben_graham_enhance(resized)),
    ])


def retina_mask(image_rgb: np.ndarray, threshold: int = 10) -> np.ndarray:
    """Binary mask of the circular retinal disc, cleaned by morphological opening."""
    _, mask = cv2.threshold(image_rgb[:, :, 1], threshold, 255, cv2.THRESH_BINARY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    return cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)


def plot_preprocessing_gallery(filepath: str, label: int, save_path: str) -> None:
    stages = build_preprocessing_stages(filepath)
    disc_mask = retina_mask(stages["2. Crop borders + resize"]) > 0
    raw_mask = retina_mask(stages["1. Raw photograph"]) > 0

    fig, axes = plt.subplots(2, len(stages), figsize=(4 * len(stages), 8),
                             gridspec_kw={"height_ratios": [3, 2]})
    for column, (title, image) in enumerate(stages.items()):
        axes[0, column].imshow(image)
        axes[0, column].set_title(title, fontsize=10, fontweight="bold")
        axes[0, column].axis("off")

        mask = raw_mask if column == 0 else disc_mask
        grey = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)[mask]
        green = image[:, :, 1][mask]
        axes[1, column].hist(grey, bins=64, range=(0, 255), color="#555555", alpha=0.75, label="Grey")
        axes[1, column].hist(green, bins=64, range=(0, 255), color="#2E8B57", alpha=0.5, label="Green")
        axes[1, column].set_xlim(0, 255)
        axes[1, column].set_yticks([])
        axes[1, column].set_xlabel(f"Intensity (std {grey.std():.1f})", fontsize=9)
        if column == 0:
            axes[1, column].legend(fontsize=8)

    fig.suptitle(
        f"Preprocessing steps and retinal-disc histograms — dataset grade {label} "
        f"({Config.CLASS_NAMES[label]})",
        fontsize=13, fontweight="bold",
    )
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[Preprocess] Step gallery saved -> {save_path}")


def plot_edge_and_morphology(filepaths_by_label: Dict[int, str], save_path: str) -> None:
    columns = ["Green channel (CLAHE)", "Sobel magnitude", "Canny edges",
               "Retina mask (opened)", "Black-hat: dark lesions/vessels", "Top-hat: bright lesions"]
    fig, axes = plt.subplots(len(filepaths_by_label), len(columns),
                             figsize=(3.6 * len(columns), 3.8 * len(filepaths_by_label)), squeeze=False)
    ellipse_15 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))

    for row, (label, filepath) in enumerate(filepaths_by_label.items()):
        resized = _resize_to_config(crop_image_from_gray(load_image_rgb(filepath)))
        mask = retina_mask(resized)
        inner_mask = cv2.erode(mask, ellipse_15)   # drop the strong disc-boundary edge
        green = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(resized[:, :, 1])
        smoothed = cv2.GaussianBlur(green, (5, 5), 0)

        sobel_x = cv2.Sobel(smoothed, cv2.CV_32F, 1, 0, ksize=3)
        sobel_y = cv2.Sobel(smoothed, cv2.CV_32F, 0, 1, ksize=3)
        sobel = cv2.magnitude(sobel_x, sobel_y)
        sobel = cv2.normalize(sobel, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

        def _inside_disc(image: np.ndarray) -> np.ndarray:
            return cv2.bitwise_and(image, image, mask=inner_mask)

        sobel = _inside_disc(sobel)
        canny = _inside_disc(cv2.Canny(smoothed, 30, 90))
        black_hat = _inside_disc(cv2.morphologyEx(green, cv2.MORPH_BLACKHAT, ellipse_15))
        top_hat = _inside_disc(cv2.morphologyEx(green, cv2.MORPH_TOPHAT, ellipse_15))

        panels = [green, sobel, canny, mask, black_hat, top_hat]
        cmaps = ["gray", "magma", "gray", "gray", "inferno", "inferno"]
        for column, (panel, cmap) in enumerate(zip(panels, cmaps)):
            axes[row, column].imshow(panel, cmap=cmap)
            axes[row, column].axis("off")
            if row == 0:
                axes[row, column].set_title(columns[column], fontsize=10, fontweight="bold")
        axes[row, 0].text(-12, Config.IMG_SIZE / 2, f"Grade {label}\n{Config.CLASS_NAMES[label]}",
                          ha="right", va="center", fontsize=10, fontweight="bold")

    fig.suptitle("Classical edge detection and morphology on the green channel",
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[Preprocess] Edge/morphology figure saved -> {save_path}")


_gallery_row = df_labels[df_labels["label"] == 2].sample(1, random_state=Config.SEED).iloc[0]
plot_preprocessing_gallery(
    _gallery_row["filepath"],
    int(_gallery_row["label"]),
    save_path=os.path.join(Config.REPORTS_DIR, "preprocessing_step_gallery.png"),
)

_edge_examples = {
    label: df_labels[df_labels["label"] == label].sample(1, random_state=Config.SEED + label).iloc[0]["filepath"]
    for label in (0, 2, 4)
    if (df_labels["label"] == label).any()
}
plot_edge_and_morphology(
    _edge_examples,
    save_path=os.path.join(Config.REPORTS_DIR, "edge_morphology_analysis.png"),
)

In [ ]:
# ============================================================
# SECTION 2.6 — TF.DATA PIPELINE
#
# WHY tf.data: Loading and preprocessing images on-the-fly in
# Python (with a plain Python generator) creates a CPU bottleneck
# that starves the GPU. tf.data pipelines:
#   - Use parallel map workers (num_parallel_calls) so multiple
#     images are preprocessed concurrently on the CPU
#   - Use prefetch so the next batch is ready before the GPU
#     finishes the current one — zero GPU idle time
#   - Cache decoded images to RAM on second-epoch replay if the
#     dataset fits in memory (Section 4 caches preprocessed PNGs to disk instead)
#
# The pipeline wraps preprocess_image in a tf.py_function so our
# existing NumPy/OpenCV preprocessing functions work seamlessly
# inside the TF graph.
# ============================================================

def _preprocess_tf(filepath: tf.Tensor, label: tf.Tensor) -> tuple:
    """
    TensorFlow-compatible wrapper around preprocess_image.

    Called by tf.data.Dataset.map(). Uses tf.py_function to run
    our NumPy/OpenCV preprocessing inside the TF graph.

    Args:
        filepath: Scalar string tensor containing the image file path.
        label:    Scalar int32 tensor containing the ICDR label.

    Returns:
        Tuple of:
            img   — float32 tensor (IMG_SIZE, IMG_SIZE, 3)
            label — int32 tensor (scalar)
    """
    def _py_preprocess(fp_bytes):
        # Decode the byte string tensor to a Python str
        fp = fp_bytes.numpy().decode("utf-8")
        img = preprocess_image(fp)          # returns float32 (H, W, 3)
        return img

    img = tf.py_function(
        func=_py_preprocess,
        inp=[filepath],
        Tout=tf.float32,
    )
    # Set the static shape so Keras knows the tensor dimensions
    # (py_function outputs have unknown shape by default)
    img.set_shape([Config.IMG_SIZE, Config.IMG_SIZE, 3])

    # One-hot encode the label for categorical cross-entropy
    label_oh = tf.one_hot(tf.cast(label, tf.int32), depth=Config.NUM_CLASSES)
    return img, label_oh


def build_tf_dataset(
    filepaths: List[str],
    labels: List[int],
    batch_size: int = Config.BATCH_SIZE,
    shuffle: bool = True,
    augment_fn=None,
) -> tf.data.Dataset:
    """
    Build a tf.data.Dataset from file paths and integer labels.

    Args:
        filepaths:  List of absolute image file paths.
        labels:     Corresponding list of integer ICDR labels (0-4).
        batch_size: Number of images per batch.
        shuffle:    Whether to shuffle the dataset each epoch.
                    Should be True for training, False for val/test.
        augment_fn: Optional callable (image, label) -> (image, label)
                    applied after preprocessing for training augmentation.
                    Set in Section 3 once augmentation is defined.

    Returns:
        Batched, prefetched tf.data.Dataset ready for model.fit().
    """
    # Build a dataset of (filepath, label) pairs
    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))

    if shuffle:
        # Buffer size = full dataset ensures true random shuffling
        dataset = dataset.shuffle(
            buffer_size=len(filepaths),
            seed=Config.SEED,
            reshuffle_each_iteration=True,  # re-shuffle every epoch
        )

    # Preprocess each image in parallel; AUTOTUNE lets TF choose the
    # number of parallel workers based on available CPU cores
    dataset = dataset.map(
        _preprocess_tf,
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    # Apply augmentation if provided (training only)
    if augment_fn is not None:
        dataset = dataset.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)

    # Batch and prefetch — prefetch(AUTOTUNE) overlaps batch preparation
    # with GPU computation, eliminating the data-loading bottleneck
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return dataset


print("[Section 2] tf.data pipeline functions defined.")
print("  build_tf_dataset() will be called in Section 4 after the train/val/test split.")
print("  Augmentation will be wired in via the augment_fn argument in Section 3.")

In [ ]:
# ============================================================
# SECTION 2.7 — SECTION SUMMARY
# Print preprocessing pipeline confirmation so we can verify
# before moving on to augmentation.
# ============================================================

print("=" * 60)
print("  SECTION 2 COMPLETE — Preprocessing")
print("=" * 60)
print(f"  crop_image_from_gray()  : black border removal")
print(f"  ben_graham_enhance()    : local contrast normalisation")
print(f"  preprocess_image()      : full pipeline (load -> crop -> resize -> enhance -> normalise)")
print(f"  build_tf_dataset()      : tf.data pipeline with parallel map + prefetch")
print(f"")
print(f"  Target resolution : {Config.IMG_SIZE} x {Config.IMG_SIZE}")
print(f"  Output dtype      : float32, values in [0.0, 1.0]")
print(f"  Comparison images : saved to {Config.REPORTS_DIR}/preprocessing_class*.png")
print("=" * 60)
print("  Next -> Section 3: Data Augmentation & Class Balancing")

In [ ]:
# ============================================================
# SECTION 2.8 — TRAINING-SPLIT PREPROCESSING ABLATION SETUP
# Defines deterministic transforms and image-statistic proxies.
# The ablation runs after the saved split is loaded and samples train_df only.
# ============================================================

from core.preprocessing import ben_graham_enhance, crop_image_from_gray


def _load_rgb_for_ablation(filepath: str) -> np.ndarray:
    image_bgr = cv2.imread(filepath)
    if image_bgr is None:
        raise ValueError(f"Could not decode image: {filepath}")
    return cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)


def _bilateral_denoise_for_ablation(image: np.ndarray) -> np.ndarray:
    return cv2.bilateralFilter(image, d=5, sigmaColor=20, sigmaSpace=20)


def _prepare_preprocessing_variant(filepath: str, variant: str) -> np.ndarray:
    raw = _load_rgb_for_ablation(filepath)
    resized_raw = cv2.resize(raw, (Config.IMG_SIZE, Config.IMG_SIZE), interpolation=cv2.INTER_AREA)
    if variant == "raw_resize":
        return resized_raw

    cropped = crop_image_from_gray(raw)
    resized = cv2.resize(
        cropped,
        (Config.IMG_SIZE, Config.IMG_SIZE),
        interpolation=cv2.INTER_AREA if cropped.shape[0] > Config.IMG_SIZE else cv2.INTER_LINEAR,
    )
    if variant == "crop_resize":
        return resized
    if variant == "crop_resize_ben_graham":
        return ben_graham_enhance(resized)
    if variant == "crop_resize_ben_graham_denoise":
        return ben_graham_enhance(_bilateral_denoise_for_ablation(resized))
    raise ValueError(f"Unknown preprocessing variant: {variant}")


def _preprocessing_quality_metrics(image: np.ndarray) -> Dict[str, float]:
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    local_brightness = cv2.GaussianBlur(gray, (0, 0), sigmaX=15)
    high_frequency_residual = gray - cv2.GaussianBlur(gray, (0, 0), sigmaX=1)
    edges = cv2.Canny((gray * 255).astype(np.uint8), 50, 150)
    laplacian = cv2.Laplacian(gray * 255.0, cv2.CV_32F)
    return {
        "contrast_std": float(gray.std()),
        "edge_density": float((edges > 0).mean()),
        "brightness_variation": float(local_brightness.std()),
        "signal_noise_proxy": float(gray.std() / (high_frequency_residual.std() + 1e-8)),
        "sharpness_laplacian_variance": float(laplacian.var()),
    }


preprocessing_variants = [
    "raw_resize",
    "crop_resize",
    "crop_resize_ben_graham",
    "crop_resize_ben_graham_denoise",
]
print("[Preprocessing Ablation] Helpers ready; execution is deferred until saved train_df is loaded.")

## Section 3: Data Augmentation & Class Balancing

**Goal:** Address two separate but related problems:

1. **Generalisation** — augmentation expands the training distribution so the model does not memorise orientation, framing or exposure artefacts.
2. **Class imbalance** — APTOS 2019 is **9.4× imbalanced** (1,805 No DR vs 193 Severe). Augmentation keeps the observed class ratio, so on its own it does not change the loss's bias towards No DR.

**Augmentation (training images only; validation and test are never augmented):**
- Horizontal and vertical flips, and **full 360° rotation** — a fundus photograph has no canonical "up", so every orientation is plausible.
- **Zoom ±15%** — simulates differences in camera field of view and framing.
- **Brightness and contrast ±15%** — simulates exposure differences between cameras, kept small so lesions are not hidden or faked.
- Strong colour/hue shifts are deliberately **not** used, because colour is diagnostic (e.g. red haemorrhages, yellow exudates).

**Balancing:** class weights are computed from the **training split only** and are **on by default**. `Config.CLASS_WEIGHT_MODE` chooses between `"balanced"` (sklearn inverse-frequency, the default), `"sqrt"` (square root of the balanced weights, rescaled to mean 1 — a softer correction that over-weights the rare Severe stage less) and `"none"`; Section 4.4 prints both weightings and `class_weighting_effect.png` shows the loss share under each. Section 3.3b adds an optional **on-the-fly oversampling** mode (`tf.data.Dataset.sample_from_datasets`, one stream per class) that is tested as a separate pilot trial in Section 6 (`oversampling_instead`), alongside `class_weights_off` and `class_weights_sqrt` trials. Oversampling and class weights are never combined in the same run.

In [ ]:
# ============================================================
# SECTION 3.1 — KERAS AUGMENTATION LAYER (graph-mode)
#
# WHY Keras augmentation layers instead of imgaug or albumentations:
#   - They run inside tf.data; TensorFlow uses available devices,
#     including a GPU when the runtime provides one.
#   - They are part of the Keras functional API and behave correctly
#     at inference time (turned off automatically when training=False).
#   - They are seeded via tf.random, consistent with our global seed.
#
# Each transform is applied with a probability; not every image gets
# every transform on every pass — this stochasticity is intentional.
# ============================================================

def build_augmentation_layer() -> keras.Sequential:
    """
    Build a Keras Sequential augmentation pipeline.

    All layers only apply their transform during training
    (training=True); during validation/testing they act as identity
    functions. This is handled automatically by Keras.

    Augmentations applied:
    - Horizontal and vertical reflection for image-level ICDR grading.
    - RandomRotation factor=1.0 for full-circle rotation.  # CHANGED: document full rotation range
    - RandomZoom +/- 15 percent.  # CHANGED: document increased zoom range
    - RandomBrightness and RandomContrast factor=0.15 over normalized [0, 1].

    Returns:
        A compiled keras.Sequential model (used as a callable layer).
    """
    augmentation = keras.Sequential(
        [
            layers.RandomFlip("horizontal", seed=3),
            layers.RandomFlip("vertical", seed=4),
            layers.RandomRotation(factor=1.0, fill_mode="constant", fill_value=0.5, seed=1),  # CHANGED: full rotation for orientation robustness
            layers.RandomZoom(height_factor=(-0.15, 0.15), fill_mode="constant", fill_value=0.5, seed=2),  # CHANGED: widen zoom range conservatively
            layers.RandomBrightness(factor=0.15, value_range=(0.0, 1.0), seed=5),
            layers.RandomContrast(factor=0.15, value_range=(0.0, 1.0), seed=6),
        ],
        name="augmentation_pipeline",
    )
    return augmentation


# Build the augmentation layer once and reuse it
augmentation_layer = build_augmentation_layer()
print("[Augmentation] Layer built:")
augmentation_layer.summary()

In [ ]:
# ============================================================
# SECTION 3.2 — AUGMENTATION WRAPPER FOR tf.data
# Wraps the Keras augmentation layer in a function signature
# compatible with tf.data.Dataset.map().
# ============================================================

def augment_fn(image: tf.Tensor, label: tf.Tensor) -> tuple:
    """
    Apply augmentation to a single (image, label) pair.

    Used as the augment_fn argument to build_tf_dataset() so augmentation
    runs inside the tf.data pipeline. The training=True flag activates
    the random transforms; during inference (training=False) no transform
    is applied — this is handled automatically by Keras random layers.

    Args:
        image: float32 tensor (IMG_SIZE, IMG_SIZE, 3), values in [0, 1].
        label: one-hot float32 tensor of shape (NUM_CLASSES,).

    Returns:
        Tuple (augmented_image, label) — label is unchanged.
    """
    # The augmentation model is built for batched input; map() supplies one image.
    image = tf.expand_dims(image, axis=0)
    image = augmentation_layer(image, training=True)
    image = tf.squeeze(image, axis=0)
    return image, label


print("[Augmentation] augment_fn() defined and ready for build_tf_dataset().")

In [ ]:
# ============================================================
# SECTION 3.3 — CLASS WEIGHT COMPUTATION
#
# Augmentation increases variety but preserves the observed class ratio.
# APTOS 2019 is about 9.4x imbalanced (No DR vs Severe), so balanced
# inverse-frequency weighting is ON by default (Config.USE_CLASS_WEIGHTS).
# ============================================================

def compute_class_weights(labels: List[int]) -> Dict[int, float]:
    """Compute per-class loss weights using sklearn's balanced strategy."""
    unique_classes = sorted(set(labels))
    weights_array = compute_class_weight(
        class_weight="balanced",
        classes=np.array(unique_classes),
        y=np.array(labels),
    )
    class_weight_dict = {
        cls: float(weight) for cls, weight in zip(unique_classes, weights_array)
    }

    print("\n[Class Weights] Computed using sklearn balanced strategy:")
    print("-" * 45)
    for cls, weight in class_weight_dict.items():
        bar = "#" * int(weight * 3)
        print(f"  Class {cls} ({Config.CLASS_NAMES[cls]:20s}): {weight:6.3f}  {bar}")
    print("-" * 45)
    if Config.USE_CLASS_WEIGHTS:  # CHANGED: report whether Keras will apply computed weights
        print("  These weights are passed to model.fit(class_weight=...) in Section 6.")
    else:  # CHANGED: avoid claiming disabled weights affect training
        print("  Weights are computed for inspection only; class weighting is disabled for this run.")
    print()
    return class_weight_dict


def compute_sqrt_class_weights(balanced_weights: Dict[int, float]) -> Dict[int, float]:
    """
    Softer class weights: square root of the balanced weights, rescaled so the mean weight is 1.

    The square root shrinks the ratio between the rarest and most common class
    (e.g. 4x becomes 2x), so minority stages still count more without the
    loss being dominated by a handful of Severe images.
    """
    rooted = {cls: float(np.sqrt(weight)) for cls, weight in balanced_weights.items()}
    mean_weight = float(np.mean(list(rooted.values())))
    return {cls: weight / mean_weight for cls, weight in rooted.items()}


# Preview only; the final weights are recomputed from training-split labels in Section 4.
_all_labels = df_labels["label"].tolist()
class_weights_preview = compute_class_weights(_all_labels)
print("[Note] Final class weights will be recomputed from training-split labels in Section 4.")

In [ ]:
# ============================================================
# SECTION 3.3b — OPTIONAL ON-THE-FLY OVERSAMPLING (tf.data)
#
# An alternative to class weights: instead of re-weighting the loss,
# draw training batches with equal probability from each DR stage.
# One tf.data stream is built per class and repeated indefinitely;
# tf.data.Dataset.sample_from_datasets picks a class uniformly at random
# for every example, so minority stages (Severe, Proliferative) are seen
# as often as No DR. Images are re-used, never copied to disk, and
# augmentation is applied after sampling so repeats look different.
#
# Oversampling and class weights are never combined in one run: both
# correct the same imbalance and together would over-correct.
# Only the training split is ever oversampled.
# ============================================================

def build_oversampled_dataset(
    filepaths: List[str],
    labels: List[int],
    load_fn,
    seed: int = Config.SEED,
) -> tf.data.Dataset:
    """
    Return an infinite, class-balanced stream of (image, one-hot label) pairs.

    Args:
        filepaths: Image paths for the training split (e.g. cached PNGs).
        labels:    Integer ICDR labels aligned with filepaths.
        load_fn:   tf.data map function (path, label) -> (image, one_hot_label).
        seed:      Seed for per-class shuffling and class sampling.

    Returns:
        Unbatched, infinite tf.data.Dataset; pass steps_per_epoch to model.fit().
    """
    filepaths = np.asarray(filepaths)
    labels = np.asarray(labels, dtype=np.int32)
    per_class = []
    for class_id in range(Config.NUM_CLASSES):
        class_mask = labels == class_id
        if not class_mask.any():
            continue
        class_ds = tf.data.Dataset.from_tensor_slices((filepaths[class_mask], labels[class_mask]))
        class_ds = class_ds.shuffle(int(class_mask.sum()), seed=seed + class_id, reshuffle_each_iteration=True)
        per_class.append(class_ds.repeat())
    if not per_class:
        raise ValueError("No training labels available for oversampling.")
    weights = [1.0 / len(per_class)] * len(per_class)
    balanced = tf.data.Dataset.sample_from_datasets(per_class, weights=weights, seed=seed)
    return balanced.map(load_fn, num_parallel_calls=tf.data.AUTOTUNE)


def check_balancing_choice(use_class_weights: bool, use_oversampling: bool) -> None:
    """Refuse to combine class weights with oversampling in the same training run."""
    if use_class_weights and use_oversampling:
        raise ValueError("Class weights and oversampling must not be combined in one run; choose one.")


check_balancing_choice(Config.USE_CLASS_WEIGHTS, Config.USE_OVERSAMPLING)
if Config.USE_CLASS_WEIGHTS != (Config.CLASS_WEIGHT_MODE != "none"):
    raise ValueError("Config.USE_CLASS_WEIGHTS must be True exactly when CLASS_WEIGHT_MODE is not 'none'.")
print("[Balancing] build_oversampled_dataset() ready "
      f"(default run: class weights={Config.USE_CLASS_WEIGHTS}, oversampling={Config.USE_OVERSAMPLING}).")


In [ ]:
# ============================================================
# SECTION 3.4 — VISUALISE AUGMENTED SAMPLES
# Show examples of augmented images for the report.
# Helps verify that the augmentations look realistic (not so
# extreme that they destroy diagnostic features).
# ============================================================

def visualise_augmented_samples(
    df: pd.DataFrame,
    n_per_class: int = 1,
    n_augmentations: int = 4,
    save_path: str = None,
    classes: Optional[List[int]] = None,
    title: str = "Original vs Augmented Fundus Images (every DR stage)",
) -> None:
    """
    Display and save a grid of original and augmented images.

    Samples n_per_class images from EVERY class (not a random subset of
    the whole dataset), so each DR stage is guaranteed to appear. Each row
    shows the preprocessed original and n_augmentations augmented versions.

    Args:
        df:              DataFrame with 'filepath' and 'label' columns.
        n_per_class:     Number of source images per class.
        n_augmentations: Number of augmented versions per source image.
        save_path:       If provided, save the figure as a PNG.
        classes:         Class ids to include (default: all classes).
        title:           Figure title.
    """
    classes = list(range(Config.NUM_CLASSES)) if classes is None else list(classes)
    sample = pd.concat(
        [
            df[df["label"] == cls].sample(
                min(n_per_class, int((df["label"] == cls).sum())),
                random_state=Config.SEED,
            )
            for cls in classes
        ],
        ignore_index=True,
    )
    n_cols = n_augmentations + 1  # original + N augmented
    n_rows = len(sample)
    if n_rows == 0:
        print("[Augmentation] No images available for the requested classes.")
        return

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    # Column header labels
    col_labels = ["Original"] + [f"Aug {i+1}" for i in range(n_augmentations)]
    for ax, lbl in zip(axes[0], col_labels):
        ax.set_title(lbl, fontsize=9, fontweight="bold")

    for row_idx, (_, row) in enumerate(sample.iterrows()):
        # Load and preprocess the original (no augmentation)
        img_arr = preprocess_image(row["filepath"])          # float32, [0,1]
        img_tensor = tf.constant(img_arr)[tf.newaxis, ...]  # add batch dim

        # Column 0: original preprocessed image
        axes[row_idx, 0].imshow(img_arr)
        axes[row_idx, 0].axis("off")
        # Row label as text: set_ylabel is hidden once the axis is turned off.
        axes[row_idx, 0].text(
            -0.08, 0.5, f"Class {int(row['label'])}\n{Config.CLASS_NAMES[int(row['label'])]}",
            transform=axes[row_idx, 0].transAxes, fontsize=9, fontweight="bold",
            ha="right", va="center",
        )

        # Columns 1..n_augmentations: different augmented versions
        for aug_idx in range(n_augmentations):
            # Apply the augmentation layer (training=True activates random ops)
            aug_tensor = augmentation_layer(img_tensor, training=True)
            aug_arr = aug_tensor[0].numpy()  # remove batch dim
            # Clip to [0,1] in case brightness/contrast pushed values out of range
            aug_arr = np.clip(aug_arr, 0.0, 1.0)
            axes[row_idx, aug_idx + 1].imshow(aug_arr)
            axes[row_idx, aug_idx + 1].axis("off")

    fig.suptitle(title, fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[Augmentation] Grid saved -> {save_path}")

    plt.show()


# One row per DR stage in the overview figure, so no stage can be missed.
visualise_augmented_samples(
    df_labels,
    n_per_class=1,
    n_augmentations=4,
    save_path=os.path.join(Config.REPORTS_DIR, "augmentation_examples.png"),
)

# Three source images (strips) per stage, saved as one figure per class.
for _aug_class in range(Config.NUM_CLASSES):
    visualise_augmented_samples(
        df_labels,
        n_per_class=3,
        n_augmentations=4,
        classes=[_aug_class],
        title=f"Augmentation Examples - Class {_aug_class}: {Config.CLASS_NAMES[_aug_class]}",
        save_path=os.path.join(Config.REPORTS_DIR, f"augmentation_class{_aug_class}.png"),
    )


In [ ]:
# ============================================================
# SECTION 3.5 — OVERFITTING MITIGATION SUMMARY
# ============================================================

print("[Section 3] Training regularization configuration:")
print("  Augmentation  : existing rotation, flips, zoom, brightness, contrast")
print(f"  Class weights : {Config.USE_CLASS_WEIGHTS} (mode '{Config.CLASS_WEIGHT_MODE}', training split only)")
print(f"  Oversampling  : {Config.USE_OVERSAMPLING} (never combined with class weights)")
print(f"  Dropout       : {Config.DROPOUT_RATE}")
print(f"  Fine-tuning   : top {Config.UNFREEZE_TOP_N} layers at LR={Config.PHASE2_LR}")
print(f"  Early stopping: patience={Config.STOP_PATIENCE} on {Config.STOP_MONITOR} "
      f"(mode {Config.STOP_MODE}, best weights restored)")
print(f"  LR reduction  : factor={Config.RLROP_FACTOR}, patience={Config.RLROP_PATIENCE} on val_loss")
print(f"  Phase 2 LR    : schedule '{Config.PHASE2_LR_SCHEDULE}' (plateau or warm-up + cosine)")

print("\n" + "=" * 60)
print("  SECTION 3 COMPLETE — Augmentation & Class Balancing")
print("=" * 60)
print("  augmentation_layer       : Keras Sequential (6 random transforms)")
print("  augment_fn()             : tf.data-compatible wrapper")
print("  compute_class_weights()  : inverse-frequency loss weighting")
print("  augmentation_examples.png saved to REPORTS_DIR")
print("  Next -> Section 4: Train / Validation / Test Split")
print("=" * 60)

## Section 4: Duplicate-Aware Train / Validation / Test Split

**Goal:** Build three non-overlapping subsets and the final `tf.data` pipelines for each.

- **≈70% Training** — used for weight updates.
- **≈15% Validation** — used for early stopping and checkpoint selection (validation QWK), learning-rate reduction (validation loss), pilot-trial selection, TTA choice and QWK threshold calibration.
- **≈15% Test** — held out until the final evaluation in Section 7 and evaluated once per run (see the Section 6 note on runs A and B).

**Duplicate control (Section 4.1):** APTOS 2019 contains repeated photographs of the same retina, sometimes with different grades. Every image is border-cropped, resized and given a 64-bit perceptual hash (`imagehash.phash`); images whose hashes differ by at most 4 bits are joined into one `duplicate_group` with union-find. Because union-find grouping is transitive, a few very large groups can form; groups of more than 50 images are therefore always placed in training. Every remaining group is placed whole, largest first, into the split (train, validation or test) that most needs that group's classes to reach 70/15/15 per class; the order of equal-sized groups is shuffled by a seed, 200 seeds (starting at 42) are tried, and the candidate closest to 70/15/15 overall and per class is kept. The search uses only labels and group IDs, never model results (`duplicate_group_sizes.csv`/`.png` and `largest_duplicate_groups.png` show the group sizes). Whole groups are always kept together, and the notebook **asserts** that validation and test each hold at least 13% of images with at least 20 images of every class, and that no image path, `patient_id` or `duplicate_group` appears in more than one split and that every near-duplicate pair shares a split. Groups whose copies carry conflicting grades are kept together in one split and listed in `duplicate_groups.csv`; nothing is deleted silently. Example pairs are saved to `duplicate_examples.png`.

**Limitations:** APTOS provides **no patient identifiers**, so `patient_id = id_code`; two photographs of the same patient that are not near-duplicates (for example left and right eyes) can fall into different splits. The dataset is **single-source**, and the test split is small (550 images, with only **29 Severe** cases), so per-class test metrics — especially for Severe — have wide uncertainty.

The split is regenerated deterministically on every run (the chosen search seed is printed) and saved to `Config.REPORTS_DIR` and the project's `splits/` folder. Class weights are recomputed on training labels only, so validation and test labels never influence training.

In [ ]:
# ============================================================
# SECTION 4.1 — DUPLICATE-AWARE STRATIFIED SPLIT (70 / 15 / 15)
#
# APTOS 2019 contains exact and near-duplicate photographs (the same
# retina saved more than once, sometimes with different grades). If one
# copy lands in training and another in test, the test score measures
# memorisation, not generalisation. So:
#   1. Every image is cropped (black border removed), resized to
#      Config.PHASH_SIZE and given a 64-bit perceptual hash (imagehash.phash).
#   2. Images whose hashes differ in <= Config.DUPLICATE_HAMMING_THRESHOLD
#      bits are joined with union-find into one duplicate_group.
#   3. Split search (labels and group IDs only, never model results):
#      groups larger than FORCE_TRAIN_GROUP_SIZE images go to train;
#      every other group is placed whole, largest first, into the split
#      (train / validation / test) that most needs that group's classes to
#      reach 70/15/15 per class. The order of equal-sized groups is shuffled
#      by a seed; SPLIT_SEARCH_SEEDS seeds are tried and the candidate
#      closest to 70/15/15 (overall and per class) is kept. Union-find
#      chaining can create a few very large groups; forcing them into train
#      stops a single group from unbalancing validation or test.
# Groups whose members carry different grades are kept together in ONE
# split and listed in duplicate_groups.csv; nothing is deleted silently.
# APTOS has no patient identifiers, so patient_id = id_code; both eyes of
# one person may still fall into different splits (documented limitation).
# ============================================================

import imagehash
from concurrent.futures import ThreadPoolExecutor

SPLITS_DIR = Path(_PROJECT_ROOT) / "splits"
SPLIT_SEARCH_SEEDS = 200         # candidate splits tried (seeds Config.SEED ... Config.SEED + 199)
FORCE_TRAIN_GROUP_SIZE = 50      # duplicate groups larger than this always go to train
SPLIT_TARGETS = {"train": Config.TRAIN_RATIO, "validation": Config.VAL_RATIO, "test": Config.TEST_RATIO}
SPLIT_NAMES = list(SPLIT_TARGETS)
MIN_EVAL_SHARE = 0.13            # validation and test must each hold at least 13% of images
MIN_EVAL_CLASS_COUNT = 20        # every class needs at least 20 images in validation and in test


def compute_phash(filepath: str, size: int = Config.PHASH_SIZE) -> imagehash.ImageHash:
    """Perceptual hash of the border-cropped, resized fundus image."""
    image = crop_image_from_gray(load_image_rgb(filepath))
    image = cv2.resize(image, (size, size), interpolation=cv2.INTER_AREA)
    return imagehash.phash(Image.fromarray(image))


def hamming_distance_matrix(hashes: List[imagehash.ImageHash]) -> np.ndarray:
    """Pairwise Hamming distances between 64-bit perceptual hashes (N x N, int32)."""
    bits = np.stack([h.hash.flatten() for h in hashes]).astype(np.int32)
    return bits @ (1 - bits).T + (1 - bits) @ bits.T


def union_find_groups(n_items: int, pairs: np.ndarray) -> np.ndarray:
    """Connected components of the near-duplicate graph; returns a group id per item."""
    parent = np.arange(n_items)

    def find(item: int) -> int:
        while parent[item] != item:
            parent[item] = parent[parent[item]]
            item = parent[item]
        return item

    for first, second in pairs:
        root_a, root_b = find(int(first)), find(int(second))
        if root_a != root_b:
            parent[max(root_a, root_b)] = min(root_a, root_b)
    roots = np.array([find(item) for item in range(n_items)])
    _, group_ids = np.unique(roots, return_inverse=True)  # consecutive ids in first-seen root order
    return group_ids


# ---- 1. Perceptual hashes (reused if this cell already hashed the same images) --
split_source_df = df_labels.reset_index(drop=True).copy()
_hash_cache = globals().get("_phash_by_filepath", {})
if not set(split_source_df["filepath"]).issubset(_hash_cache):
    with ThreadPoolExecutor(max_workers=min(os.cpu_count() or 1, 8)) as executor:
        _new_hashes = list(tqdm(
            executor.map(compute_phash, split_source_df["filepath"]),
            total=len(split_source_df),
            desc="Perceptual hashing",
        ))
    _hash_cache = dict(zip(split_source_df["filepath"], _new_hashes))
    _phash_by_filepath = _hash_cache
else:
    print("[Duplicates] Reusing perceptual hashes computed earlier in this session.")
image_hashes = [_hash_cache[path] for path in split_source_df["filepath"]]
split_source_df["phash"] = [str(h) for h in image_hashes]

# ---- 2. Near-duplicate graph + union-find ------------------------------------
distance_matrix = hamming_distance_matrix(image_hashes)
pair_i, pair_j = np.where(np.triu(distance_matrix <= Config.DUPLICATE_HAMMING_THRESHOLD, k=1))
near_duplicate_pairs = pd.DataFrame({
    "index_a": pair_i,
    "index_b": pair_j,
    "hamming_distance": distance_matrix[pair_i, pair_j],
})
split_source_df["duplicate_group"] = union_find_groups(
    len(split_source_df), near_duplicate_pairs[["index_a", "index_b"]].to_numpy()
)
split_source_df["patient_id"] = split_source_df["id_code"]  # APTOS has no patient IDs

group_summary = split_source_df.groupby("duplicate_group").agg(
    n_images=("id_code", "size"),
    id_codes=("id_code", lambda values: ";".join(values)),
    labels=("label", lambda values: ";".join(str(v) for v in values)),
    n_distinct_labels=("label", "nunique"),
)
group_summary["conflicting_labels"] = group_summary["n_distinct_labels"] > 1
multi_image_groups = group_summary[group_summary["n_images"] > 1]
conflicting_groups = multi_image_groups[multi_image_groups["conflicting_labels"]]
print(f"[Duplicates] pHash threshold: Hamming distance <= {Config.DUPLICATE_HAMMING_THRESHOLD} of 64 bits")
print(f"[Duplicates] Near-duplicate pairs found     : {len(near_duplicate_pairs):,}")
print(f"[Duplicates] Groups with more than one image: {len(multi_image_groups):,} "
      f"({int(multi_image_groups['n_images'].sum()):,} images)")
print(f"[Duplicates] Groups with conflicting labels : {len(conflicting_groups):,} "
      f"({int(conflicting_groups['n_images'].sum()):,} images) — kept together in one split, not deleted")


# ---- 3. Split search: whole groups placed greedily, scored on labels only -----
def split_distance(labels: np.ndarray, assignment: np.ndarray) -> float:
    """
    Distance of a candidate split from the 70/15/15 target.

    Sum of |overall share - target| over the three splits, plus the mean over
    classes of sum_s |share of that class placed in split s - target_s|.
    Uses labels and split names only; no model output is involved.
    """
    size_term = sum(abs(np.mean(assignment == name) - target) for name, target in SPLIT_TARGETS.items())
    class_terms = []
    for class_id in np.unique(labels):
        in_class = assignment[labels == class_id]
        class_terms.append(sum(abs(np.mean(in_class == name) - target) for name, target in SPLIT_TARGETS.items()))
    return float(size_term + np.mean(class_terms))


all_labels = split_source_df["label"].to_numpy()
all_groups = split_source_df["duplicate_group"].to_numpy()
n_total = len(split_source_df)
group_sizes = split_source_df["duplicate_group"].map(split_source_df["duplicate_group"].value_counts())
forced_train_mask = (group_sizes > FORCE_TRAIN_GROUP_SIZE).to_numpy()

# Pre-compute each free group's rows and per-class counts once.
_free_rows = np.where(~forced_train_mask)[0]
_free_group_ids, _free_inverse = np.unique(all_groups[_free_rows], return_inverse=True)
group_rows = [_free_rows[_free_inverse == k] for k in range(len(_free_group_ids))]
group_class_counts = np.stack([np.bincount(all_labels[rows], minlength=Config.NUM_CLASSES) for rows in group_rows])
group_n = group_class_counts.sum(axis=1)
split_ratios = np.array([SPLIT_TARGETS[name] for name in SPLIT_NAMES])
target_counts = split_ratios[:, None] * np.bincount(all_labels, minlength=Config.NUM_CLASSES)[None, :]
forced_counts = np.bincount(all_labels[forced_train_mask], minlength=Config.NUM_CLASSES)


def greedy_group_split(seed: int) -> np.ndarray:
    """Place whole groups, largest first, into the split that most needs their classes."""
    rng = np.random.default_rng(seed)
    counts = np.zeros_like(target_counts)
    counts[SPLIT_NAMES.index("train")] += forced_counts
    order = rng.permutation(len(group_rows))
    order = order[np.argsort(-group_n[order], kind="stable")]   # largest first, ties in seeded order
    split_of_group = np.empty(len(group_rows), dtype=int)
    for k in order:
        relative_need = (target_counts - counts) / np.maximum(target_counts, 1.0)
        best_split = int(np.argmax(relative_need @ group_class_counts[k]))
        counts[best_split] += group_class_counts[k]
        split_of_group[k] = best_split
    assignment = np.full(n_total, "train", dtype=object)
    for k, rows in enumerate(group_rows):
        assignment[rows] = SPLIT_NAMES[split_of_group[k]]
    return assignment


print(f"[Split] {int(forced_train_mask.sum()):,} images in {split_source_df.loc[forced_train_mask, 'duplicate_group'].nunique()} "
      f"groups larger than {FORCE_TRAIN_GROUP_SIZE} images are forced into train.")
print(f"[Split] Searching {SPLIT_SEARCH_SEEDS} seeds over {len(group_rows):,} remaining groups "
      f"({len(_free_rows):,} images).")

best_seed, best_distance, best_assignment = None, np.inf, None
for search_seed in tqdm(range(Config.SEED, Config.SEED + SPLIT_SEARCH_SEEDS), desc="Split search"):
    assignment = greedy_group_split(search_seed)
    distance = split_distance(all_labels, assignment)
    if distance < best_distance:
        best_seed, best_distance, best_assignment = search_seed, distance, assignment

split_source_df["split"] = best_assignment
print(f"[Split] Chosen search seed: {best_seed} | distance from 70/15/15 (sizes + per-class shares): {best_distance:.4f}")

manifest_columns = ["filepath", "label", "split", "patient_id", "duplicate_group"]
train_df = split_source_df[split_source_df["split"] == "train"].reset_index(drop=True)
val_df = split_source_df[split_source_df["split"] == "validation"].reset_index(drop=True)
test_df = split_source_df[split_source_df["split"] == "test"].reset_index(drop=True)

# ---- 4. Integrity assertions: no image or duplicate group in two splits ------
_split_frames = {"train": train_df, "validation": val_df, "test": test_df}
assert sum(len(frame) for frame in _split_frames.values()) == len(split_source_df)
assert split_source_df["filepath"].is_unique, "An image path appears more than once."
for _group_col in ("filepath", "patient_id", "duplicate_group"):
    _groups = [set(frame[_group_col]) for frame in _split_frames.values()]
    assert not (_groups[0] & _groups[1] or _groups[0] & _groups[2] or _groups[1] & _groups[2]), \
        f"Split overlap found in {_group_col}"
    print(_group_col, "overlap check passed")
_pair_splits = split_source_df["split"].to_numpy()
assert (_pair_splits[near_duplicate_pairs["index_a"]] == _pair_splits[near_duplicate_pairs["index_b"]]).all(), \
    "A near-duplicate pair was split across train/validation/test."
print("near-duplicate pair check passed: every pair shares one split")
assert split_source_df.loc[forced_train_mask, "split"].eq("train").all(), "Large duplicate groups must stay in train."
for _name in ("validation", "test"):
    _share = len(_split_frames[_name]) / n_total
    assert _share >= MIN_EVAL_SHARE, f"{_name} holds only {_share:.1%} of images (minimum {MIN_EVAL_SHARE:.0%})."
    _class_counts = _split_frames[_name]["label"].value_counts().reindex(range(Config.NUM_CLASSES), fill_value=0)
    assert (_class_counts >= MIN_EVAL_CLASS_COUNT).all(), \
        f"{_name} has a class with fewer than {MIN_EVAL_CLASS_COUNT} images: {_class_counts.to_dict()}"
print(f"size checks passed: validation and test each >= {MIN_EVAL_SHARE:.0%} of images, "
      f"every class >= {MIN_EVAL_CLASS_COUNT} images in both")

# ---- 5. Save manifests, duplicate groups and the dedup log --------------------
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
for _name, _frame in (("train_split.csv", train_df), ("validation_split.csv", val_df), ("test_split.csv", test_df)):
    _frame[manifest_columns].to_csv(Path(Config.REPORTS_DIR) / _name, index=False)
    _frame[manifest_columns].to_csv(SPLITS_DIR / _name, index=False)

cross_split_group_leaks = int(split_source_df.groupby("duplicate_group")["split"].nunique().gt(1).sum())
group_split = split_source_df.groupby("duplicate_group")["split"].first()
duplicate_groups_export = multi_image_groups.assign(split=group_split.reindex(multi_image_groups.index))
duplicate_groups_export.reset_index().to_csv(Path(Config.REPORTS_DIR) / "duplicate_groups.csv", index=False)
deduplication_log = pd.DataFrame([
    {"metric": "total_images_evaluated", "value": len(split_source_df)},
    {"metric": "phash_hamming_threshold", "value": Config.DUPLICATE_HAMMING_THRESHOLD},
    {"metric": "largest_duplicate_group_size", "value": int(group_summary["n_images"].max())},
    {"metric": "groups_forced_into_train", "value": int(split_source_df.loc[forced_train_mask, "duplicate_group"].nunique())},
    {"metric": "images_forced_into_train", "value": int(forced_train_mask.sum())},
    {"metric": "split_search_method", "value": "greedy whole-group placement, largest first, seeded tie order"},
    {"metric": "split_search_seeds_tried", "value": SPLIT_SEARCH_SEEDS},
    {"metric": "split_search_chosen_seed", "value": int(best_seed)},
    {"metric": "split_search_distance", "value": round(best_distance, 6)},
    {"metric": "near_duplicate_pairs", "value": len(near_duplicate_pairs)},
    {"metric": "duplicate_groups_with_more_than_one_image", "value": len(multi_image_groups)},
    {"metric": "images_in_duplicate_groups", "value": int(multi_image_groups["n_images"].sum())},
    {"metric": "groups_with_conflicting_labels", "value": len(conflicting_groups)},
    {"metric": "images_in_conflicting_groups", "value": int(conflicting_groups["n_images"].sum())},
    {"metric": "cross_split_group_leaks", "value": cross_split_group_leaks},
    {"metric": "leakage_status", "value": "ZERO_LEAKAGE_CONFIRMED" if cross_split_group_leaks == 0 else "LEAKAGE_DETECTED"},
])
deduplication_log.to_csv(Path(Config.REPORTS_DIR) / "deduplication_log.csv", index=False)

df_labels = pd.concat([train_df, val_df, test_df], ignore_index=True)
print("Split sizes:", {name: f"{len(frame)} ({len(frame) / n_total:.1%})" for name, frame in _split_frames.items()})
print("Class counts per split:",
      {name: frame["label"].value_counts().sort_index().to_dict() for name, frame in _split_frames.items()})
print(f"[Split] Manifests saved to {Config.REPORTS_DIR} and {SPLITS_DIR}")

In [ ]:
# SECTION 4.2 — DUPLICATE AUDIT AND EXAMPLE PAIRS
# Shows the deduplication summary, the duplicate-group size distribution,
# thumbnails of the three largest groups (true duplicates or union-find
# chaining?), lists groups whose copies carry different grades (kept
# together in one split), and saves 3-4 example near-duplicate pairs.
_dedupe_path = Path(Config.REPORTS_DIR) / "deduplication_log.csv"
display(pd.read_csv(_dedupe_path))

# ---- Group-size distribution -------------------------------------------------
group_size_table = (
    group_summary["n_images"].value_counts().sort_index()
    .rename_axis("group_size").reset_index(name="n_groups")
)
group_size_table["n_images"] = group_size_table["group_size"] * group_size_table["n_groups"]
group_size_table.to_csv(Path(Config.REPORTS_DIR) / "duplicate_group_sizes.csv", index=False)
print("[Duplicates] Group-size distribution (groups of 1 = unique images):")
print(group_size_table.to_string(index=False))

multi_sizes = group_summary.loc[group_summary["n_images"] > 1, "n_images"]
fig, axis = plt.subplots(figsize=(9, 4.5))
if len(multi_sizes):
    axis.hist(multi_sizes, bins=np.arange(1.5, multi_sizes.max() + 1.5, max(1, int(multi_sizes.max() // 40))),
              color="#367C8A", edgecolor="white")
axis.axvline(FORCE_TRAIN_GROUP_SIZE + 0.5, color="#B91C1C", linestyle="--",
             label=f"> {FORCE_TRAIN_GROUP_SIZE} images: forced into train")
axis.set_yscale("log")
axis.set_xlabel("Images per duplicate group")
axis.set_ylabel("Number of groups (log scale)")
axis.set_title("Duplicate-group sizes (groups with more than one image)")
axis.legend()
fig.tight_layout()
fig.savefig(Path(Config.REPORTS_DIR) / "duplicate_group_sizes.png", dpi=150, bbox_inches="tight")
plt.show()


def plot_largest_groups(source: pd.DataFrame, n_groups: int = 3, per_group: int = 8, save_path: str = None) -> None:
    """Thumbnail montage of the largest duplicate groups, to check for chaining vs true duplicates."""
    largest = source["duplicate_group"].value_counts().head(n_groups)
    fig, axes = plt.subplots(len(largest), per_group, figsize=(1.9 * per_group, 2.3 * len(largest)), squeeze=False)
    for row, (group_id, size) in enumerate(largest.items()):
        members = source[source["duplicate_group"] == group_id].head(per_group)
        for column in range(per_group):
            axis = axes[row, column]
            axis.axis("off")
            if column < len(members):
                record = members.iloc[column]
                thumb = cv2.resize(crop_image_from_gray(load_image_rgb(record["filepath"])), (160, 160),
                                   interpolation=cv2.INTER_AREA)
                axis.imshow(thumb)
                axis.set_title(f"{record['id_code'][:8]} g{record['label']}", fontsize=7)
        axes[row, 0].text(-0.15, 0.5, f"group {group_id}\n{size} images", transform=axes[row, 0].transAxes,
                          ha="right", va="center", fontsize=9, fontweight="bold")
    fig.suptitle("Largest duplicate groups (first images of each; g = grade)", fontsize=11, fontweight="bold")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


plot_largest_groups(split_source_df, save_path=os.path.join(Config.REPORTS_DIR, "largest_duplicate_groups.png"))

if len(conflicting_groups):
    print("[Duplicates] Groups with conflicting labels (kept together in one split, not deleted):")
    display(duplicate_groups_export[duplicate_groups_export["conflicting_labels"]]
            [["n_images", "id_codes", "labels", "split"]].head(20))
else:
    print("[Duplicates] No duplicate group has conflicting labels.")


def plot_duplicate_examples(pairs: pd.DataFrame, source: pd.DataFrame, save_path: str, max_pairs: int = 4) -> None:
    """Plot near-duplicate pairs side by side, conflicting-label pairs first."""
    if pairs.empty:
        print("[Duplicates] No near-duplicate pairs to plot.")
        return
    ranked = pairs.assign(
        conflict=[
            source.at[a, "label"] != source.at[b, "label"]
            for a, b in zip(pairs["index_a"], pairs["index_b"])
        ]
    ).sort_values(["conflict", "hamming_distance"], ascending=[False, True]).head(max_pairs)

    fig, axes = plt.subplots(len(ranked), 2, figsize=(8, 4 * len(ranked)), squeeze=False)
    for row_index, pair in enumerate(ranked.itertuples()):
        for column, image_index in enumerate((pair.index_a, pair.index_b)):
            record = source.loc[image_index]
            image = cv2.resize(crop_image_from_gray(load_image_rgb(record["filepath"])), (Config.PHASH_SIZE, Config.PHASH_SIZE))
            axes[row_index, column].imshow(image)
            axes[row_index, column].axis("off")
            axes[row_index, column].set_title(
                f"{record['id_code']}\nGrade {record['label']} ({Config.CLASS_NAMES[record['label']]}) | {record['split']}",
                fontsize=9,
            )
        axes[row_index, 0].text(
            1.05, -0.08,
            f"Hamming distance {pair.hamming_distance}"
            + (" — CONFLICTING GRADES" if pair.conflict else ""),
            transform=axes[row_index, 0].transAxes, ha="center", fontsize=9,
            color="#B91C1C" if pair.conflict else "#334155", fontweight="bold",
        )
    fig.suptitle("Example near-duplicate pairs (grouped into the same split)", fontsize=12, fontweight="bold")
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[Duplicates] Example pairs saved -> {save_path}")


plot_duplicate_examples(
    near_duplicate_pairs,
    split_source_df,
    save_path=os.path.join(Config.REPORTS_DIR, "duplicate_examples.png"),
)


In [ ]:
# Confirm the generated split is complete and uses the three expected split names.
assert len(train_df) + len(val_df) + len(test_df) == len(df_labels)
assert set(train_df["split"]) == {"train"}
assert set(val_df["split"]) == {"validation"}
assert set(test_df["split"]) == {"test"}
print("Split is complete; no post-training repartition is performed.")

In [ ]:
# ============================================================
# SECTION 4.2b — TRAINING-ONLY PREPROCESSING QUALITY ABLATION
# This descriptive image-statistics audit samples the training split only.
# It does not fit preprocessing parameters or use validation/test images.
# ============================================================

sample_frames = []
for class_id in range(Config.NUM_CLASSES):
    class_train = train_df[train_df["label"] == class_id]
    if class_train.empty:
        continue
    sample_frames.append(
        class_train.sample(
            min(20, len(class_train)),
            random_state=Config.SEED + class_id,
        )
    )

if not sample_frames:
    raise ValueError("No labeled training images are available for preprocessing quality analysis.")

sample_rows = pd.concat(sample_frames, ignore_index=True)
if not sample_rows["split"].eq("train").all():
    raise AssertionError("Preprocessing quality sample contains a non-training record.")

quality_records = []
for _, row in tqdm(sample_rows.iterrows(), total=len(sample_rows), desc="Training-only preprocessing ablation"):
    for variant in preprocessing_variants:
        variant_image = _prepare_preprocessing_variant(row["filepath"], variant)
        quality_records.append({
            "filepath": row["filepath"],
            "split": row["split"],
            "label": int(row["label"]),
            "variant": variant,
            **_preprocessing_quality_metrics(variant_image),
        })

quality_df = pd.DataFrame(quality_records)
quality_metrics = [
    "contrast_std",
    "edge_density",
    "brightness_variation",
    "signal_noise_proxy",
    "sharpness_laplacian_variance",
]
quality_summary = quality_df.groupby("variant")[quality_metrics].agg(["mean", "std"]).round(6)

quality_path = os.path.join(Config.REPORTS_DIR, "preprocessing_quality_ablation.csv")
quality_df.to_csv(quality_path, index=False)
quality_summary.to_csv(os.path.join(Config.REPORTS_DIR, "preprocessing_quality_summary.csv"))
print("[Training-only preprocessing ablation] Sample split counts:", sample_rows["split"].value_counts().to_dict())
print("[Training-only preprocessing ablation] Mean image-statistic proxies:")
print(quality_df.groupby("variant")[quality_metrics].mean().round(6).to_string())
print(f"[Training-only preprocessing ablation] Saved -> {quality_path}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
variant_order = preprocessing_variants
sns.boxplot(data=quality_df, x="variant", y="contrast_std", order=variant_order, ax=axes[0], color="#3B82A0")
axes[0].set_title("Grayscale contrast")
axes[0].set_xlabel("Preprocessing variant")
axes[0].set_ylabel("Pixel standard deviation (0-1)")
axes[0].tick_params(axis="x", rotation=20)
sns.boxplot(data=quality_df, x="variant", y="sharpness_laplacian_variance", order=variant_order, ax=axes[1], color="#D28E45")
axes[1].set_title("Sharpness proxy")
axes[1].set_xlabel("Preprocessing variant")
axes[1].set_ylabel("Laplacian variance (grayscale 0-255)")
axes[1].tick_params(axis="x", rotation=20)
fig.suptitle("Training-only preprocessing statistics (descriptive, not model performance)", fontsize=12)
plt.tight_layout()
preprocessing_metrics_path = os.path.join(Config.REPORTS_DIR, "preprocessing_contrast_sharpness.png")
plt.savefig(preprocessing_metrics_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[Training-only preprocessing ablation] Contrast/sharpness plot saved -> {preprocessing_metrics_path}")

comparison_row = sample_rows.iloc[0]
comparison_images = [
    _prepare_preprocessing_variant(comparison_row["filepath"], variant)
    for variant in preprocessing_variants
]
fig, axes = plt.subplots(1, len(preprocessing_variants), figsize=(16, 4))
for axis, variant, image in zip(axes, preprocessing_variants, comparison_images):
    axis.imshow(image)
    axis.set_title(variant.replace("_", "\n"), fontsize=9)
    axis.axis("off")
plt.tight_layout()
comparison_path = os.path.join(Config.REPORTS_DIR, "preprocessing_ablation_examples.png")
plt.savefig(comparison_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[Training-only preprocessing ablation] Visual comparison saved -> {comparison_path}")

In [ ]:
# ============================================================
# SECTION 4.3 — PER-CLASS SIZE CONFIRMATION
# Print class counts for each split to confirm stratification
# worked correctly. All three splits should have roughly the
# same percentage breakdown as the full dataset.
# ============================================================

def print_split_distribution(
    train_df: pd.DataFrame,
    val_df:   pd.DataFrame,
    test_df:  pd.DataFrame,
) -> None:
    """
    Print per-class counts for all three splits side by side.

    Allows visual verification that stratification preserved class
    proportions. Large deviations from expected ratios would indicate
    a problem with the stratified split.

    Args:
        train_df: Training split DataFrame.
        val_df:   Validation split DataFrame.
        test_df:  Test split DataFrame.
    """
    all_classes = sorted(train_df["label"].unique())

    header = f"{'Class':<5} {'Name':<22} {'Train':>7} {'Val':>7} {'Test':>7}  {'Train%':>7} {'Val%':>7} {'Test%':>7}"
    print("\n" + "=" * len(header))
    print("  PER-CLASS SPLIT SIZES")
    print("=" * len(header))
    print(header)
    print("-" * len(header))

    for cls in all_classes:
        n_train = (train_df["label"] == cls).sum()
        n_val   = (val_df["label"]   == cls).sum()
        n_test  = (test_df["label"]  == cls).sum()
        n_total = n_train + n_val + n_test

        pct_train = n_train / n_total * 100 if n_total else 0
        pct_val   = n_val   / n_total * 100 if n_total else 0
        pct_test  = n_test  / n_total * 100 if n_total else 0

        print(
            f"  {cls:<5} {Config.CLASS_NAMES[cls]:<22} "
            f"{n_train:>7,} {n_val:>7,} {n_test:>7,}  "
            f"{pct_train:>6.1f}% {pct_val:>6.1f}% {pct_test:>6.1f}%"
        )

    print("=" * len(header))
    print("  Expected: ~70% / ~15% / ~15% per class (stratification check)")
    print("=" * len(header) + "\n")


print_split_distribution(train_df, val_df, test_df)


# ------------------------------------------------------------
# CLASS-TO-INDEX CHECK TABLE (label -> class name -> count, per split)
# Confirms every split uses the same integer->stage mapping, that all
# labels are valid ICDR ids, and that each image's class folder (when the
# path is .../<split>/<class_id>/<file>) agrees with its manifest label.
# ------------------------------------------------------------
def _folder_label(filepath: str) -> Optional[int]:
    parent = Path(str(filepath)).parent.name
    return int(parent) if parent.isdigit() else None


_class_index_rows = []
for _split_name, _frame in (("train", train_df), ("validation", val_df), ("test", test_df)):
    _invalid = sorted(set(_frame["label"]) - set(range(Config.NUM_CLASSES)))
    if _invalid:
        raise ValueError(f"{_split_name} split contains labels outside 0-{Config.NUM_CLASSES - 1}: {_invalid}")
    _folder_labels = _frame["filepath"].map(_folder_label)
    for _class_id in range(Config.NUM_CLASSES):
        _mask = _frame["label"] == _class_id
        _checked = _folder_labels[_mask].dropna()
        _class_index_rows.append({
            "split": _split_name,
            "label_index": _class_id,
            "class_name": Config.CLASS_NAMES[_class_id],
            "image_count": int(_mask.sum()),
            "percent_of_split": float(_mask.mean() * 100),
            "folder_checked": int(len(_checked)),
            "folder_mismatches": int((_checked != _class_id).sum()),
        })

class_index_check = pd.DataFrame(_class_index_rows)
class_index_check_path = os.path.join(Config.REPORTS_DIR, "class_index_check.csv")
class_index_check.to_csv(class_index_check_path, index=False)

print("[Class-Index Check] label -> class name -> count, per split:")
for _split_name, _table in class_index_check.groupby("split", sort=False):
    print(f"\n  {_split_name.upper()} ({int(_table['image_count'].sum()):,} images)")
    print(f"  {'Index':<6} {'Class name':<22} {'Count':>7} {'%':>7} {'Folder mismatches':>18}")
    for _, _row in _table.iterrows():
        print(
            f"  {_row['label_index']:<6} {_row['class_name']:<22} {_row['image_count']:>7,} "
            f"{_row['percent_of_split']:>6.1f}% {_row['folder_mismatches']:>18}"
        )
if class_index_check["folder_mismatches"].sum() > 0:
    raise ValueError("Some image class folders disagree with their manifest labels; see class_index_check.csv.")
print(f"\n[Class-Index Check] Mapping consistent across splits. Saved -> {class_index_check_path}")


In [ ]:
# ============================================================
# SECTION 4.4 — RECOMPUTE CLASS WEIGHTS ON TRAINING SET ONLY
#
# WHY recompute here (not use the full-dataset preview from Section 3):
#   class_weight must reflect only the training distribution.
#   Using weights computed on the full dataset would incorporate
#   val/test label counts into the training objective — a subtle
#   form of data leakage. The difference is small (70% vs 100% of
#   data) but it is the principled approach and correct by convention.
#
# Two weightings are computed from the training labels:
#   balanced -> sklearn "balanced" inverse-frequency weights
#   sqrt     -> square root of the balanced weights, rescaled so the mean
#               weight is 1 (a softer correction; less over-weighting of
#               the rare Severe stage)
# Config.CLASS_WEIGHT_MODE ("balanced", "sqrt" or "none") picks which one
# model.fit() receives; active_class_weights() is the single source used
# by the pilot trials and the final fit.
# ============================================================

# Recompute class weights on training labels only
train_labels = train_df["label"].tolist()
class_weight_balanced = compute_class_weights(train_labels)
class_weight_sqrt = compute_sqrt_class_weights(class_weight_balanced)
class_weight_dict = class_weight_balanced  # balanced weights (kept for existing references)


def active_class_weights() -> Optional[Dict[int, float]]:
    """Class weights for model.fit() under the current Config (None when weighting is off)."""
    if not Config.USE_CLASS_WEIGHTS or Config.CLASS_WEIGHT_MODE == "none":
        return None
    if Config.CLASS_WEIGHT_MODE == "sqrt":
        return class_weight_sqrt
    if Config.CLASS_WEIGHT_MODE == "balanced":
        return class_weight_balanced
    raise ValueError(f"Unknown Config.CLASS_WEIGHT_MODE: {Config.CLASS_WEIGHT_MODE!r}")


print("[Class Weights] Final weights (computed on training split only):")
print(f"  {'Class':<25} {'balanced':>9} {'sqrt':>9}")
for cls in sorted(class_weight_balanced):
    print(f"  {cls} {Config.CLASS_NAMES[cls]:<23} {class_weight_balanced[cls]:>9.4f} {class_weight_sqrt[cls]:>9.4f}")
print(f"\n  Mode used for training: CLASS_WEIGHT_MODE='{Config.CLASS_WEIGHT_MODE}', "
      f"USE_CLASS_WEIGHTS={Config.USE_CLASS_WEIGHTS}.")
print("  The class_weights_off, class_weights_sqrt and oversampling_instead pilot trials in Section 6 test the alternatives.")


In [ ]:
# ============================================================
# SECTION 4.4b — SPLIT AND CLASS-WEIGHT FIGURES
# Split counts come from the unchanged manifests. Weight effects use
# training labels only; no images are duplicated or oversampled.
# ============================================================

class_ids = list(range(Config.NUM_CLASSES))
class_names = [Config.CLASS_NAMES[class_id] for class_id in class_ids]
split_frames = {
    "Train": train_df,
    "Validation": val_df,
    "Test": test_df,
}
split_counts = pd.DataFrame(
    {
        split_name: frame["label"].value_counts().reindex(class_ids, fill_value=0).to_numpy()
        for split_name, frame in split_frames.items()
    },
    index=class_names,
)
split_counts.index.name = "class_name"
split_counts.to_csv(os.path.join(Config.REPORTS_DIR, "split_class_counts.csv"))

fig, ax = plt.subplots(figsize=(11, 6))
split_counts.plot(kind="bar", ax=ax, color=["#367C8A", "#D49A45", "#7A8F55"], width=0.78)
ax.set_title("Class Distribution Across the Saved Dataset Splits")
ax.set_xlabel("Diabetic retinopathy stage")
ax.set_ylabel("Number of images")
ax.legend(title="Saved split")
ax.tick_params(axis="x", rotation=18)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
split_plot_path = os.path.join(Config.REPORTS_DIR, "split_class_distribution.png")
fig.savefig(split_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"[Split Figures] Saved manifest class counts -> {split_plot_path}")

train_counts = train_df["label"].value_counts().reindex(class_ids, fill_value=0).astype(float)
# Always plot both computed weightings so the figure shows what each would do,
# whichever mode (balanced, sqrt or none) is used for training.
applied_weights = pd.Series(class_weight_balanced, dtype=float).reindex(class_ids)
sqrt_weights = pd.Series(class_weight_sqrt, dtype=float).reindex(class_ids)
if applied_weights.isna().any() or sqrt_weights.isna().any():
    raise ValueError("A training class is missing its computed class weight.")

unweighted_share = train_counts / train_counts.sum() * 100
weighted_mass = train_counts * applied_weights
weighted_share = weighted_mass / weighted_mass.sum() * 100
sqrt_mass = train_counts * sqrt_weights
sqrt_share = sqrt_mass / sqrt_mass.sum() * 100
weighting_summary = pd.DataFrame({
    "class_id": class_ids,
    "class_name": class_names,
    "training_images": train_counts.values.astype(int),
    "class_weight_balanced": applied_weights.values,
    "class_weight_sqrt": sqrt_weights.values,
    "class_weight_mode_in_training": Config.CLASS_WEIGHT_MODE if Config.USE_CLASS_WEIGHTS else "none",
    "unweighted_loss_mass_percent": unweighted_share.values,
    "balanced_loss_mass_percent": weighted_share.values,
    "sqrt_loss_mass_percent": sqrt_share.values,
})
weighting_summary.to_csv(os.path.join(Config.REPORTS_DIR, "class_weighting_effect.csv"), index=False)

# Headline imbalance ratio (largest class / smallest class) before and after weighting.
full_counts = df_labels["label"].value_counts().reindex(class_ids, fill_value=0).astype(float)
imbalance_summary = pd.DataFrame([
    {"stage": "Full dataset (image counts)", "imbalance_ratio": full_counts.max() / full_counts.min()},
    {"stage": "Training split (image counts)", "imbalance_ratio": train_counts.max() / train_counts.min()},
    {"stage": "Training split with balanced class weights (effective loss mass)",
     "imbalance_ratio": weighted_mass.max() / weighted_mass.min()},
    {"stage": "Training split with sqrt class weights (effective loss mass)",
     "imbalance_ratio": sqrt_mass.max() / sqrt_mass.min()},
])
imbalance_summary["class_weight_mode_in_training"] = Config.CLASS_WEIGHT_MODE if Config.USE_CLASS_WEIGHTS else "none"
imbalance_summary.to_csv(os.path.join(Config.REPORTS_DIR, "imbalance_ratio_summary.csv"), index=False)
raw_imbalance_ratio = float(imbalance_summary.loc[1, "imbalance_ratio"])
weighted_imbalance_ratio = float(imbalance_summary.loc[2, "imbalance_ratio"])
sqrt_imbalance_ratio = float(imbalance_summary.loc[3, "imbalance_ratio"])
print(
    "[Imbalance] Largest/smallest class ratio: "
    + " -> ".join(f"{ratio:.2f}x" for ratio in imbalance_summary["imbalance_ratio"])
    + "  (full dataset -> training split -> balanced weights -> sqrt weights)"
)
if not Config.USE_CLASS_WEIGHTS:
    print("[Imbalance] Class weights are disabled for this run, so training uses the unweighted training ratio.")
else:
    print(f"[Imbalance] Training uses the '{Config.CLASS_WEIGHT_MODE}' weights.")

positions = np.arange(Config.NUM_CLASSES)
bar_width = 0.27
fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(positions - bar_width, unweighted_share, bar_width, label="Without class weights", color="#6E8792")
ax.bar(positions, weighted_share, bar_width, label="Balanced class weights", color="#D8784B")
ax.bar(positions + bar_width, sqrt_share, bar_width, label="Sqrt class weights (softer)", color="#7A8F55")
ax.set_title(
    "Class Weights Rebalance Expected Training-Loss Contribution\n"
    f"Imbalance ratio {raw_imbalance_ratio:.2f}x -> {weighted_imbalance_ratio:.2f}x (balanced) / "
    f"{sqrt_imbalance_ratio:.2f}x (sqrt)"
)
ax.set_xlabel("Diabetic retinopathy stage")
ax.set_ylabel("Share of summed per-example loss weight (%)")
ax.set_xticks(positions, class_names, rotation=18, ha="right")
ax.legend()
ax.grid(axis="y", alpha=0.25)
ax.text(
    0.01,
    -0.27,
    "Weighted contribution assumes equal per-example losses; it is not an image-count change or an accuracy result.\n"
    + (f"Training uses the '{Config.CLASS_WEIGHT_MODE}' weights (CLASS_WEIGHT_MODE); the other bars show the alternatives."
       if Config.USE_CLASS_WEIGHTS
       else "Class weights are NOT applied during training (CLASS_WEIGHT_MODE='none'); weighted bars show their potential effect."),
    transform=ax.transAxes,
    fontsize=9,
)
fig.tight_layout()
weighting_plot_path = os.path.join(Config.REPORTS_DIR, "class_weighting_effect.png")
fig.savefig(weighting_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"[Class-Weight Figures] Saved weighted-loss comparison -> {weighting_plot_path}")

In [ ]:
import hashlib
from concurrent.futures import ThreadPoolExecutor

if Path("/kaggle/temp").exists():
    _cache_root = Path("/kaggle/temp")
elif Path("/kaggle/working").exists():
    _cache_root = Path("/kaggle/working")
else:
    _cache_root = Path(_RUN_ROOT) / "cache"
CACHE_DIR = _cache_root / "preproc_cache" / str(Config.IMG_SIZE)
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _cache_path(fp):
    return str(CACHE_DIR / (hashlib.md5(fp.encode()).hexdigest() + ".png"))


def _cache_one(fp):
    out = _cache_path(fp)
    if not os.path.exists(out):
        img = preprocess_image(fp, img_size=Config.IMG_SIZE)
        img8 = np.clip(img * 255.0, 0, 255).round().astype(np.uint8)
        if not cv2.imwrite(out, cv2.cvtColor(img8, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_PNG_COMPRESSION, 1]):
            raise OSError(f"Could not write cache file {out} (check disk space and path length).")
    return out


all_paths = pd.concat([train_df, val_df, test_df], ignore_index=True)["filepath"].tolist()
with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
    list(tqdm(executor.map(_cache_one, all_paths), total=len(all_paths), desc="Caching"))


def build_cached_dataset(
    filepaths,
    labels,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    augment_fn=None,
    oversample=False,
):
    """
    Build a batched tf.data pipeline from the preprocessed PNG cache.

    With oversample=True (training split only) the stream is class-balanced and
    infinite, so model.fit() must be given steps_per_epoch.
    """
    cached = [_cache_path(fp) for fp in filepaths]

    def _load(path, label):
        image = tf.io.decode_png(tf.io.read_file(path), channels=3)
        image = tf.image.convert_image_dtype(image, tf.float32)
        image.set_shape([Config.IMG_SIZE, Config.IMG_SIZE, 3])
        return image, tf.one_hot(tf.cast(label, tf.int32), Config.NUM_CLASSES)

    if oversample:
        dataset = build_oversampled_dataset(cached, labels, _load)
    else:
        dataset = tf.data.Dataset.from_tensor_slices((cached, labels))
        if shuffle:
            dataset = dataset.shuffle(
                len(cached), seed=Config.SEED, reshuffle_each_iteration=True
            )
        dataset = dataset.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_fn is not None:
        dataset = dataset.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


print(f"[Cache] Preprocessed image cache ready at {CACHE_DIR}")


In [ ]:
# ============================================================
# SECTION 4.5 — BUILD tf.data DATASETS
# Wire together the preprocessing cache (Section 4),
# augmentation (Section 3), and the split DataFrames to produce
# three ready-to-use tf.data.Dataset objects.
#
# Key decisions:
#   - TRAINING dataset: shuffle=True, augment_fn applied
#   - VALIDATION dataset: shuffle=False, no augmentation
#   - TEST dataset: shuffle=False, no augmentation
# ============================================================

# Extract file paths and labels as plain Python lists
train_paths = train_df["filepath"].tolist()
train_labels_list = train_df["label"].tolist()

val_paths = val_df["filepath"].tolist()
val_labels_list = val_df["label"].tolist()

test_paths = test_df["filepath"].tolist()
test_labels_list = test_df["label"].tolist()

train_dataset = build_cached_dataset(
    filepaths=train_paths,
    labels=train_labels_list,
    batch_size=Config.BATCH_SIZE,
    shuffle=True,
    augment_fn=augment_fn,
    oversample=Config.USE_OVERSAMPLING,
)
val_dataset = build_cached_dataset(
    filepaths=val_paths,
    labels=val_labels_list,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    augment_fn=None,
)
test_dataset = build_cached_dataset(
    filepaths=test_paths,
    labels=test_labels_list,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    augment_fn=None,
)

steps_per_epoch = int(np.ceil(len(train_paths) / Config.BATCH_SIZE))  # required when oversampling
validation_steps = len(val_paths) // Config.BATCH_SIZE
test_steps = len(test_paths) // Config.BATCH_SIZE

print("[Datasets] Cached tf.data pipelines built:")
print(f"  train_dataset  : {len(train_paths):,} images  | {steps_per_epoch} batches/epoch")
print(f"  val_dataset    : {len(val_paths):,} images  | {validation_steps} batches/epoch")
print(f"  test_dataset   : {len(test_paths):,} images  | {test_steps} batches (eval)")
print(f"  Batch size     : {Config.BATCH_SIZE}")
print("  Augmentation   : train only")
print("  Shuffle        : train only")
print(f"  Oversampling   : {Config.USE_OVERSAMPLING} (train only)")

In [ ]:
# ============================================================
# SECTION 4.6 — BATCH SHAPE VERIFICATION
# Peek at one batch to confirm shapes and dtypes are correct
# before we pass these datasets to the model.
# ============================================================

def verify_dataset_batch(dataset: tf.data.Dataset, name: str) -> None:
    """
    Inspect one batch from a tf.data.Dataset and print shape/dtype info.

    Args:
        dataset: A batched tf.data.Dataset (image, label) pairs.
        name:    Human-readable name for the dataset (e.g. "train").
    """
    for images, labels in dataset.take(1):
        print(f"[Batch Check] {name} dataset:")
        print(f"  images shape : {images.shape}   dtype: {images.dtype}")
        print(f"  labels shape : {labels.shape}   dtype: {labels.dtype}")
        print(f"  images range : [{images.numpy().min():.3f}, {images.numpy().max():.3f}]")
        print(f"  labels sample: {labels[0].numpy()}  (one-hot, sum={labels[0].numpy().sum():.0f})")


verify_dataset_batch(train_dataset, "train")
verify_dataset_batch(val_dataset,   "val")
verify_dataset_batch(test_dataset,  "test")

print("\n" + "=" * 60)
print("  SECTION 4 COMPLETE — Train / Val / Test Split")
print("=" * 60)
print(f"  train_df       : {len(train_df):,} rows")
print(f"  val_df         : {len(val_df):,} rows")
print(f"  test_df        : {len(test_df):,} rows")
print(f"  class_weight_dict : computed on training labels")
print(f"  train_dataset, val_dataset, test_dataset : ready for model.fit()")
print("  Next -> Section 5: Model — EfficientNetB3 Transfer Learning")
print("=" * 60)

In [ ]:
# ============================================================
# SECTION 4.7 — DATASET AUDIT & CROSS-SPLIT LEAKAGE EVIDENCE
# ============================================================

def section_1_dataset_audit(df, train_df, val_df, test_df):
    """Save a complete, reproducible dataset and patient-level audit."""
    print("\n" + "=" * 90)
    print("DATASET AUDIT")
    print("=" * 90)

    total_images = len(df)
    class_counts = df["label"].value_counts().sort_index()
    train_counts = train_df["label"].value_counts().sort_index()
    val_counts = val_df["label"].value_counts().sort_index()
    test_counts = test_df["label"].value_counts().sort_index()

    # APTOS has no patient IDs (patient_id = id_code), so this is an image and
    # duplicate-group overlap check, not a patient-level one.
    train_images = set(train_df["patient_id"])
    val_images = set(val_df["patient_id"])
    test_images = set(test_df["patient_id"])
    train_groups, val_groups, test_groups = (
        set(frame["duplicate_group"]) for frame in (train_df, val_df, test_df)
    )
    overlaps = {
        "train_validation_image_overlap": len(train_images & val_images),
        "train_test_image_overlap": len(train_images & test_images),
        "validation_test_image_overlap": len(val_images & test_images),
        "train_validation_duplicate_group_overlap": len(train_groups & val_groups),
        "train_test_duplicate_group_overlap": len(train_groups & test_groups),
        "validation_test_duplicate_group_overlap": len(val_groups & test_groups),
    }

    print(f"Total dataset images: {total_images:,}")
    print("\nClass distribution (overall):")
    print(class_counts.to_string())
    print("\nTrain split distribution:")
    print(train_counts.to_string())
    print("\nValidation split distribution:")
    print(val_counts.to_string())
    print("\nTest split distribution:")
    print(test_counts.to_string())

    print("\nImage / duplicate-group overlap check (APTOS has no patient IDs):")
    print(f"Train images: {len(train_images):,} in {len(train_groups):,} duplicate groups")
    print(f"Validation images: {len(val_images):,} in {len(val_groups):,} duplicate groups")
    print(f"Test images: {len(test_images):,} in {len(test_groups):,} duplicate groups")
    for metric, value in overlaps.items():
        print(f"{metric}: {value:,}")

    split_frames = {"train": train_df, "validation": val_df, "test": test_df}
    audit_records = [
        {"metric": "total_images", "value": total_images},
        {"metric": "train_images", "value": len(train_df)},
        {"metric": "validation_images", "value": len(val_df)},
        {"metric": "test_images", "value": len(test_df)},
        {"metric": "train_percentage", "value": len(train_df) / total_images * 100},
        {"metric": "validation_percentage", "value": len(val_df) / total_images * 100},
        {"metric": "test_percentage", "value": len(test_df) / total_images * 100},
        {"metric": "train_unique_images", "value": len(train_images)},
        {"metric": "validation_unique_images", "value": len(val_images)},
        {"metric": "test_unique_images", "value": len(test_images)},
        {"metric": "train_duplicate_groups", "value": len(train_groups)},
        {"metric": "validation_duplicate_groups", "value": len(val_groups)},
        {"metric": "test_duplicate_groups", "value": len(test_groups)},
        *[{"metric": metric, "value": value} for metric, value in overlaps.items()],
    ]

    for class_id in sorted(class_counts.index):
        audit_records.append({
            "metric": f"overall_class_{class_id}_count",
            "value": int(class_counts.get(class_id, 0)),
        })
        for split_name, split_frame in split_frames.items():
            audit_records.append({
                "metric": f"{split_name}_class_{class_id}_count",
                "value": int(split_frame["label"].eq(class_id).sum()),
            })

    audit_records.append({
        "metric": "image_and_duplicate_group_overlap_status",
        "value": "ZERO_OVERLAP" if max(overlaps.values()) == 0 else "OVERLAP_DETECTED",
    })

    summary_path = os.path.join(Config.REPORTS_DIR, "dataset_audit_summary.csv")
    pd.DataFrame(audit_records).to_csv(summary_path, index=False)
    print(f"\nDataset audit summary saved to: {summary_path}")
    print("\n" + "=" * 90)


section_1_dataset_audit(df_labels, train_df, val_df, test_df)


## Section 5: CNN Architecture & Transfer Learning — EfficientNetB3

### Why EfficientNetB3?

The table lists published ImageNet classification figures as context for architecture selection. It is not a DR benchmark, so Section 6 trains every alternative backbone (and a CNN from scratch) on this project's own split with the same head, preprocessing and schedule, to check the choice on APTOS 2019.

| Backbone | ImageNet-1K top-1 | Parameters | GFLOPs | Trade-off |
|---|---:|---:|---:|---|
| VGG-16 | 71.59% | 138.4M | 15.47 | Simple design, but very large classifier and compute cost |
| ResNet-50 | 76.13% | 25.6M | 4.09 | Residual learning; higher compute and parameter cost |
| DenseNet-121 | 74.98% | 8.0M | 2.87 | Dense connections reuse features; widely used in medical imaging |
| MobileNetV2 | 71.88% | 3.5M | 0.30 | Very efficient, with lower capacity than B3 |
| EfficientNetB0 | 77.1% | 5.3M | 0.39 | Same family as B3 at lower resolution and capacity |
| **EfficientNetB3** | **81.6%** | **12.0M** | **1.8** | Compound-scaled balance of depth, width, resolution, and capacity |

EfficientNetB3 was chosen as a practical capacity/compute trade-off for five-stage fundus grading. Published ImageNet figures motivate this choice but do not prove superiority on diabetic-retinopathy images. The ImageNet classifier top is removed; the backbone starts from ImageNet weights.

### Transfer Learning, Classifier Head and Overfitting Controls

APTOS 2019 leaves only 2,561 training images after the split, so the defaults are chosen to limit overfitting: Phase 1 freezes the backbone and trains the new head; Phase 2 restores the best Phase 1 checkpoint and fine-tunes only the **top 120** EfficientNetB3 layers (of 385 in Kaggle's Keras 3.13) (BatchNorm stays frozen) with a low **1e-5** learning rate and **AdamW weight decay 1e-4**. Both phases stop on **validation QWK** and keep the highest-QWK weights (the pilots in Section 6 stop on validation loss). The Phase 2 learning rate either drops on a validation-loss plateau (default) or follows a 2-epoch warm-up and cosine decay (`Config.PHASE2_LR_SCHEDULE="cosine"`).

The head is **global average pooling → BatchNorm → Dense(256, ReLU) → Dropout(0.5) → Dense(5, softmax)**:

- **Global average pooling** aggregates each feature channel over space while avoiding the large parameter count of flattening the feature map.
- **BatchNorm** stabilizes the feature distribution entering the new classifier head.
- **Dense(256, ReLU)** learns a compact DR-specific representation; it is also used for case-retrieval embeddings.
- **Dropout 0.5** regularizes the head during training.
- **Softmax(5)** returns probabilities for ICDR stages 0–4.

The input adapter rescales preprocessed float values from [0,1] back to [0,255] because Keras EfficientNet applies its own internal 1/255 rescaling. Each comparison backbone gets its own ImageNet adapter: Caffe-style BGR mean subtraction for ResNet-50 and VGG-16, [-1, 1] scaling for MobileNetV2, Torch-style mean/std normalisation for DenseNet-121, and ×255 for EfficientNetB0.

```mermaid
flowchart LR
    A[Fundus image<br/>Config.IMG_SIZE x Config.IMG_SIZE x 3<br/>float values in 0 to 1] --> B[Rescaling x255 adapter]
    B --> C[EfficientNetB3<br/>ImageNet weights<br/>include_top=False]
    C --> D[Global average pooling]
    D --> E[Batch normalization]
    E --> F[Dense 256, ReLU]
    F --> G[Dropout Config.DROPOUT_RATE]
    G --> H[Dense 5, softmax]
    H --> I[ICDR stage probabilities 0 to 4]
    J[Phase 1<br/>freeze backbone<br/>Adam, Config.PHASE1_LR] -.-> C
    K[Phase 2<br/>unfreeze top Config.UNFREEZE_TOP_N<br/>AdamW, Config.PHASE2_LR<br/>BatchNorm stays frozen] -.-> C
```

### Pilot Trials (Section 6)

With `RUN_HYPERPARAMETER_TUNING=True`, every pilot trial trains on the **full training split** and is scored on the **full validation split**, for up to **4 Phase 1 + 6 Phase 2 epochs** with validation-loss early stopping (patience 2), so the saved curves are long enough to show overfitting. Each trial saves its history, training curves (accuracy and loss for augmented training, a clean training subset and validation; val_qwk) and validation confusion matrix under `report_images/pilots/<trial_id>/`; all rows go to `hyperparameter_tuning_results.csv` and `model_comparison_table.csv`.

| Pilot trial | Group | Change vs baseline |
|---|---|---|
| `baseline_300_d05_lr1e5_top120` | hyperparameter | Baseline: 300 px, batch 16, dropout 0.5, LR 1e-3 / 1e-5 (plateau), top-120 fine-tuning, weight decay 1e-4, balanced class weights |
| `dropout_040` | hyperparameter | Dropout 0.5 → 0.4 |
| `unfreeze_top60` | hyperparameter | Fine-tune 120 → 60 layers |
| `unfreeze_top240` | hyperparameter | Fine-tune 120 → 240 layers |
| `phase2_lr_3e5` | hyperparameter | Phase 2 LR 1e-5 → 3e-5 |
| `resolution_224` | hyperparameter | Input 300 → 224 |
| `resolution_380` | hyperparameter | Input 300 → 380, batch 16 → 8 |
| `lr_schedule_cosine` | hyperparameter | Phase 2 LR plateau → 2-epoch warm-up + cosine decay |
| `weight_decay_1e3` | hyperparameter | AdamW weight decay 1e-4 → 1e-3 |
| `class_weights_off` | balancing | Class weights on → off |
| `class_weights_sqrt` | balancing | Balanced → sqrt (softer) class weights |
| `oversampling_instead` | balancing | Class weights → on-the-fly oversampling |
| `ablation_no_ben_graham` | preprocessing ablation | Ben Graham off (crop + resize only) |
| `ablation_clahe_instead` | preprocessing ablation | CLAHE instead of Ben Graham |
| `ablation_all_filters` | preprocessing ablation | Denoise + CLAHE + unsharp edges before Ben Graham |
| `ablation_no_augmentation` | augmentation ablation | Augmentation off |
| `backbone_resnet50` | backbone | EfficientNetB3 → ResNet-50 |
| `backbone_mobilenetv2` | backbone | EfficientNetB3 → MobileNetV2 |
| `backbone_efficientnetb0` | backbone | EfficientNetB3 → EfficientNetB0 |
| `backbone_densenet121` | backbone | EfficientNetB3 → DenseNet-121 |
| `backbone_vgg16` | backbone | EfficientNetB3 → VGG-16 |
| `scratch_cnn_4block` | scratch baseline | Small 4-block CNN, random initialisation, trained end to end (transfer-learning baseline) |

That is **22 defined pilot trials**. By default `Config.PILOT_SKIP_TRIALS` skips five of them (`phase2_lr_3e5`, `unfreeze_top240`, `dropout_040`, `ablation_all_filters`, `backbone_vgg16`): they were dropped after an earlier validation-only pilot showed they underperformed the baseline or duplicated other evidence, to fit the compute budget. They stay defined and appear in `hyperparameter_tuning_results.csv` as `skipped_by_config`, so they can be re-enabled. **17 trials run** — 9 selectable (6 hyperparameter, 3 balancing) and 8 comparison trials. The final configuration is selected by the **highest validation QWK (tie-break: lower validation loss)** among the **hyperparameter** and **balancing** rows only. The notebook also prints, without using them for selection, the selectable trial with the highest validation accuracy and the one with the smallest clean-train vs validation accuracy gap, so the report can discuss the trade-offs. Ablation, backbone and scratch rows are evidence: the deployed pipeline (Grad-CAM layer, embeddings, app) is built around EfficientNetB3 with Ben Graham preprocessing, so those rows are reported and discussed instead of silently changing the final architecture. The held-out test split is not used for any selection. Pilot runs are shorter than the final fit, so rankings may not perfectly predict full-run rankings.

In [ ]:
# ============================================================
# SECTION 5.1 — EFFICIENTNETB3 CLASSIFIER CONSTRUCTION
# Recreate the classifier architecture used by the shared application model.
# ============================================================

_gpu_available = bool(tf.config.list_physical_devices("GPU"))
keras.mixed_precision.set_global_policy("mixed_float16" if _gpu_available else "float32")
print(f"[Precision] Global policy: {keras.mixed_precision.global_policy().name}")

base_model = EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(Config.IMG_SIZE, Config.IMG_SIZE, 3),
)
base_model.trainable = False

inputs = keras.Input(shape=(Config.IMG_SIZE, Config.IMG_SIZE, 3))
x = layers.Rescaling(255.0, name="restore_efficientnet_input_range")(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.BatchNormalization(name="head_bn")(x)
x = layers.Dense(Config.DENSE_UNITS, activation="relu", name="head_dense")(x)
x = layers.Dropout(Config.DROPOUT_RATE, name="head_dropout")(x)
outputs = layers.Dense(
    Config.NUM_CLASSES,
    activation="softmax",
    dtype="float32",
    name="predictions",
)(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="RetinaGuard_EfficientNetB3")

print("[Model] EfficientNetB3 initialized with ImageNet weights.")
print(f"[Model] Classifier head: GAP -> BatchNorm -> Dense({Config.DENSE_UNITS}) -> Dropout({Config.DROPOUT_RATE}) -> Dense({Config.NUM_CLASSES})")
print("[Model] Base frozen for Phase 1.")

In [ ]:
# ============================================================
# SECTION 5.2 — COMPILE MODEL (PHASE 1 CONFIGURATION)
#
# Loss: CategoricalCrossentropy
#   - Labels are one-hot (output of tf.one_hot in the dataset pipeline)
#   - label_smoothing=0.1 softens the targets slightly (e.g. turns a
#     hard [0,0,1,0,0] into [0.02, 0.02, 0.92, 0.02, 0.02]).
#     This prevents the model from becoming over-confident on the
#     training set and improves calibration on unseen data.
#
# Metric: SparseCategoricalAccuracy
#   - We convert one-hot labels back to sparse integers for the metric
#     so the accuracy printout is human-readable (not over one-hot vectors)
#   - QWK is computed post-training in Section 7 (it is not a differentiable
#     TF metric, so we implement it via sklearn after inference)
# ============================================================

def compile_model_phase1(m: keras.Model, jit_compile: Union[bool, str] = "auto") -> None:
    """
    Compile the model for Phase 1 training (frozen base, head only).

    Uses Adam optimiser with Config.PHASE1_LR (1e-3). A higher learning
    rate is safe in Phase 1 because only the randomly-initialised head
    parameters are updated — the pretrained base weights are frozen.

    Args:
        m: The keras.Model to compile (modified in place).
        jit_compile: Passed to Model.compile. The final model keeps "auto" (XLA
            where supported; compiled once, so the speed-up is worth it). The
            Section 6.2b pilots pass False, because an XLA-compiled cluster for
            every pilot model accumulates host RAM across the trial loop.
    """
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=Config.PHASE1_LR),
        loss=keras.losses.CategoricalCrossentropy(
            label_smoothing=Config.LABEL_SMOOTHING,  # soft targets for better calibration
        ),
        metrics=[
            keras.metrics.CategoricalAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc", multi_label=False),
        ],
        jit_compile=jit_compile,
    )
    print(f"[Compile] Phase 1 — LR={Config.PHASE1_LR}, base frozen.")


compile_model_phase1(model)

In [ ]:
# ============================================================
# SECTION 5.3 — MODEL SUMMARY
# Print the full model summary: layer names, output shapes,
# parameter counts, and trainable vs frozen parameter totals.
# ============================================================

print("\n" + "=" * 70)
print("  FULL MODEL SUMMARY")
print("=" * 70)
model.summary(line_length=70, show_trainable=True)

# Separately report base model stats for clarity
total_params     = model.count_params()
trainable_params = sum(
    tf.size(v).numpy() for v in model.trainable_variables
)
frozen_params    = total_params - trainable_params

print("\n" + "-" * 70)
print(f"  Total parameters     : {total_params:>12,}")
print(f"  Trainable (Phase 1)  : {trainable_params:>12,}  (head only)")
print(f"  Frozen (Phase 1)     : {frozen_params:>12,}  (EfficientNetB3 base)")
print(f"  Base model layers    : {len(base_model.layers):>12,}")
print(f"  Layers to unfreeze   : {Config.UNFREEZE_TOP_N:>12,}  (Phase 2)")
print("-" * 70)
print(f"  [MULTI-STAGE NOTE] Final Dense layer outputs {Config.NUM_CLASSES} classes")
print(f"  covering all ICDR stages 0-4 (No DR through Proliferative DR).")
print(f"  Stage-level output; binary DR / referable-DR results are derived in Section 7.4d.")
print("=" * 70)

In [ ]:
# ============================================================
# SECTION 5.4 — PHASE 2 UNFREEZE CONFIGURATION
# Unfreeze the backbone while keeping BatchNorm layers frozen.
# ============================================================

def configure_phase2(m: keras.Model, base: keras.Model, jit_compile: Union[bool, str] = "auto") -> None:
    """Enable fine-tuning for the backbone, keeping BatchNorm stable.

    jit_compile is passed to Model.compile: the final model keeps "auto" (XLA
    where supported), while the Section 6.2b pilots pass False so an XLA
    cluster is not compiled and kept in host RAM for every pilot model.
    """
    n_layers = len(base.layers)
    if not 0 < Config.UNFREEZE_TOP_N <= n_layers:
        raise ValueError(
            f"UNFREEZE_TOP_N must be between 1 and {n_layers}; "
            f"received {Config.UNFREEZE_TOP_N}."
        )

    base.trainable = True
    freeze_until = n_layers - Config.UNFREEZE_TOP_N

    for i, layer in enumerate(base.layers):
        if i < freeze_until or isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True

    newly_trainable = sum(
        tf.size(v).numpy() for v in m.trainable_variables
    )

    m.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=Config.PHASE2_LR,
            weight_decay=Config.WEIGHT_DECAY,  # decoupled weight decay for fine-tuning
        ),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=Config.LABEL_SMOOTHING),
        metrics=[
            keras.metrics.CategoricalAccuracy(name="accuracy"),
            keras.metrics.AUC(name="auc", multi_label=False),
        ],
        jit_compile=jit_compile,
    )

    print(f"[Phase 2] Unfroze the top {Config.UNFREEZE_TOP_N} of {n_layers} backbone layers except BatchNorm.")
    print(f"[Phase 2] Trainable parameters now: {newly_trainable:,}")
    print(f"[Phase 2] Learning rate: {Config.PHASE2_LR}")


print("[Section 5] configure_phase2() defined — BatchNorm layers remain frozen.")

print("\n" + "=" * 60)
print("  SECTION 5 COMPLETE — Model Architecture")
print("=" * 60)
print("  model            : EfficientNetB3 + custom head (5-class)")
print("  base_model       : EfficientNetB3 (frozen for Phase 1)")
print("  compile_model_phase1() : Phase 1 compilation defined")
print("  configure_phase2()     : Phase 2 transition defined")
print("  Output           : softmax over 5 ICDR classes (multi-stage)")
print("  Next -> Section 6: Training Strategy")
print("=" * 60)

## Section 6: Training Strategy

**Goal:** Select a configuration with validation-only pilot trials, then train the selected model on the complete training split and honestly report train/validation behaviour.

### Two-Phase Final Training

**Phase 1 — Feature extraction:** The ImageNet-pretrained backbone is frozen while the five-class head trains with `Config.PHASE1_LR` for up to `Config.PHASE1_EPOCHS` (8) epochs.

**Phase 2 — Fine-tuning:** The top `Config.UNFREEZE_TOP_N` (120) backbone layers are fine-tuned while BatchNorm remains frozen, using AdamW (`Config.PHASE2_LR` 1e-5, weight decay `Config.WEIGHT_DECAY` 1e-4) for up to `Config.PHASE2_EPOCHS` (25) epochs. With `Config.PHASE2_LR_SCHEDULE="plateau"` (default) the learning rate halves after 2 epochs without a validation-loss improvement; with `"cosine"` it warms up linearly over 2 epochs from 10% of the peak, then follows a cosine down to 1% of the peak (no plateau reduction in that mode).

In both phases **EarlyStopping and ModelCheckpoint monitor validation QWK** (`Config.STOP_MONITOR="val_qwk"`, mode max, patience 6, highest-QWK weights restored); ReduceLROnPlateau still reacts to validation loss. `ValidationQWK` computes `val_qwk` on the validation split after every epoch and is first in the callback list, so the value exists before stopping and checkpointing read it. The monitoring-only `CleanTrainMetrics` callback logs `clean_train_loss` and `clean_train_accuracy` on a fixed, stratified subset of 500 training images (seed 42) scored without augmentation or class weights.

**Why validation QWK (run A → run B).** The first full final fit (run A) stopped and checkpointed on validation loss. With balanced class weights and label smoothing, validation loss and validation QWK disagreed in both phases of run A: loss restored epoch 1 of Phase 1 and epoch 3 of Phase 2 while validation QWK and accuracy were still rising. The stopping rule was therefore changed to validation QWK on validation evidence only (run A's Phase 1 and Phase 2 curves), and the final fit was rerun. Run B was fixed as the reported model before its test results were produced, and run A's test metrics were not used in any decision. The pilot trials still stop on validation loss.

### Pilot Selection, Balancing, Ablations and Backbone Comparison

When `RUN_HYPERPARAMETER_TUNING=True`, all pilot trials use the full training and validation splits with up to 4 + 6 epochs and validation-loss early stopping (patience 2). Six hyperparameter trials (baseline, `unfreeze_top60`, `resolution_224`, `resolution_380`, `lr_schedule_cosine`, `weight_decay_1e3`) and three balancing trials (`class_weights_off`, `class_weights_sqrt`, `oversampling_instead`) are selectable; the winner (highest validation QWK, lower validation loss as tie-break) is used for the final full-data training run. With `RUN_COMPARISON_TRIALS=True`, the same runner also trains two preprocessing ablations (no Ben Graham, CLAHE instead), an augmentation ablation, four alternative ImageNet backbones (ResNet-50, MobileNetV2, EfficientNetB0, DenseNet-121) and a small CNN trained from scratch as the transfer-learning baseline. Set `RUN_COMPARISON_TRIALS=False` to skip those eight trials if GPU time is short. Pilot results and final full-run histories are kept separately so pilot scores are not presented as final model performance.

**Compute limits.** Five further trials (`phase2_lr_3e5`, `unfreeze_top240`, `dropout_040`, `ablation_all_filters`, `backbone_vgg16`) are listed in `Config.PILOT_SKIP_TRIALS`: they were dropped after an earlier validation-only pilot showed they underperformed the baseline or duplicated other evidence, to fit the compute budget. They remain defined, are recorded as `skipped_by_config` and are left out of selection and of the 6.2c/6.2d tables and plots. `Config.PILOT_TIME_BUDGET_HOURS` (5 h) stops new trials from starting once the budget is used, recording the rest as `skipped_time_budget`; selection then uses the completed selectable trials. `Config.PILOT_TRIAL_LIMIT` runs only the first N non-skipped trials (for a quick memory test). Setting `Config.RESUME_RUN_ID` to an earlier run ID reuses that run folder: completed trials whose split fingerprint matches the current split are not retrained, and the selection is rebuilt from their saved rows with the same rule. Pilot models are compiled without XLA and fully released after each trial (process RAM and GPU memory are logged per trial) so a long pilot run does not exhaust host memory.

### Overfitting Assessment

Every pilot row records the train–validation accuracy gap and the **clean-train–validation** accuracy gap at the lowest-validation-loss epoch, and how far validation loss rose after its minimum. After the final full-data fit, `overfitting_diagnostics.csv` records the same quantities for each phase. **Why training accuracy can sit below validation accuracy.** Keras' training accuracy is averaged over *augmented* batches (rotations, zoom, brightness/contrast changes), with **dropout** switching off half of the head's units, and — with class weights — a loss that emphasises the hard minority stages. Validation images are scored unaugmented, with dropout off and an unweighted loss, so the "Train (augmented)" curve is measured on a harder task and can legitimately lie *below* validation without meaning the model generalises better than it fits. The **"Train (clean subset)"** curve removes those differences: 500 fixed training images scored exactly like validation. The gap between the clean curve and validation is therefore the like-for-like overfitting signal; both are plotted next to the original augmented curve, unsmoothed.

Read these metrics alongside the saved learning curves; describe overfitting only if the observed curves support it.

Dropout 0.5, partial fine-tuning (top 120 layers), frozen BatchNorm, AdamW weight decay, training-only augmentation, label smoothing and validation-QWK early stopping are the implemented regularization controls. QWK threshold calibration and TTA selection use validation predictions only. The held-out test set is not used to select settings, thresholds, or checkpoints and is evaluated only after the final configuration is fixed.

In [ ]:
# ============================================================
# SECTION 6.1 — EXPERIMENT LOG TABLE
#
# WHY: The coursework requires an "experiments tried" record.
# We maintain a simple DataFrame that logs every training run's
# hyperparameters and final results. This lets us compare Phase 1
# vs Phase 2 (and any additional runs) at a glance.
# ============================================================

# Initialise the experiment log as a global DataFrame
# Each row = one call to model.fit() with its configuration + results
experiment_log = pd.DataFrame(columns=[
    "phase",
    "learning_rate",
    "batch_size",
    "epochs_planned",
    "epochs_run",        # actual epochs before early stopping
    "val_accuracy",
    "val_loss",
    "val_auc",
    "val_qwk",
    "notes",
])


def log_experiment(
    phase: str,
    lr: float,
    batch_size: int,
    epochs_planned: int,
    history: keras.callbacks.History,
    notes: str = "",
) -> None:
    """
    Append a completed training run's results to the experiment log.

    Args:
        phase:          Human-readable phase name (e.g. "Phase 1 - Frozen Base").
        lr:             Learning rate used in this run.
        batch_size:     Batch size used.
        epochs_planned: Maximum epochs configured (before early stopping).
        history:        The History object returned by model.fit().
        notes:          Optional free-text notes about this run.
    """
    global experiment_log

    epochs_run   = len(history.history["val_loss"])
    best_val_acc = max(history.history.get("val_accuracy", [0]))
    best_val_loss = min(history.history["val_loss"])
    best_val_auc  = max(history.history.get("val_auc", [0]))
    best_val_qwk  = max(history.history.get("val_qwk", [float("nan")]))

    new_row = pd.DataFrame([{
        "phase":           phase,
        "learning_rate":   lr,
        "batch_size":      batch_size,
        "epochs_planned":  epochs_planned,
        "epochs_run":      epochs_run,
        "val_accuracy":    round(best_val_acc,  4),
        "val_loss":        round(best_val_loss, 4),
        "val_auc":         round(best_val_auc,  4),
        "val_qwk":         round(best_val_qwk,  4),
        "notes":           notes,
    }])

    experiment_log = pd.concat([experiment_log, new_row], ignore_index=True)
    print(f"[Experiment Log] Logged: {phase}")
    print(experiment_log.to_string(index=False))


print("[Section 6] Experiment log initialised.")

In [ ]:
# ============================================================
# SECTION 6.2 — CALLBACKS
#
# Final-fit stopping/checkpointing rule (run B):
# EarlyStopping and ModelCheckpoint monitor validation QWK
# (Config.STOP_MONITOR, mode max) and restore the highest-QWK weights after
# Config.STOP_PATIENCE epochs without improvement. Run A monitored validation
# loss; under balanced class weights and label smoothing, validation loss and
# QWK disagreed in both final phases (loss chose epoch 1 of Phase 1 and
# epoch 3 of Phase 2 while QWK/accuracy were still rising). Run A had
# already been evaluated on the test split before this change (see the
# Section 6 notes); run B is the reported model.
# Learning-rate control depends on the schedule:
#   "plateau" -> ReduceLROnPlateau halves the LR after Config.RLROP_PATIENCE
#                epochs without a validation-loss improvement (unchanged).
#   "cosine"  -> LearningRateScheduler: linear warm-up over
#                Config.LR_WARMUP_EPOCHS epochs from 10% of the peak LR,
#                then cosine decay to 1% of the peak by the last epoch.
#                (No ReduceLROnPlateau in this mode.)
# ValidationQWK is first in the callback list, so val_qwk is in the epoch
# logs before EarlyStopping / ModelCheckpoint read it.
# Monitoring only:
#   CleanTrainMetrics -> clean_train_loss / clean_train_accuracy on a fixed,
#                        stratified 500-image TRAINING subset, evaluated
#                        without augmentation, dropout or class weights.
# Note: the Section 6.2b pilots use their own EarlyStopping (val_loss) and
# are unaffected by this rule.
# ============================================================

import math


class LearningRateLogger(keras.callbacks.Callback):
    """Record the optimizer learning rate each epoch so history CSVs always contain it."""

    def on_epoch_end(self, epoch, logs=None):
        if logs is not None and "learning_rate" not in logs:
            logs["learning_rate"] = float(np.array(self.model.optimizer.learning_rate))


class ValidationQWK(keras.callbacks.Callback):
    """Log validation quadratic weighted kappa as 'val_qwk' after every epoch (used by the final-fit stopping rule)."""

    def __init__(self, validation_dataset: tf.data.Dataset, validation_labels: np.ndarray):
        super().__init__()
        self.validation_dataset = validation_dataset
        self.validation_labels = np.asarray(validation_labels, dtype=int)

    def on_epoch_end(self, epoch, logs=None):
        probabilities = self.model.predict(self.validation_dataset, verbose=0)
        predictions = np.argmax(probabilities, axis=1)
        if len(predictions) != len(self.validation_labels):
            raise ValueError("ValidationQWK: prediction count does not match validation labels.")
        qwk = float(cohen_kappa_score(self.validation_labels, predictions, weights="quadratic"))
        if logs is not None:
            logs["val_qwk"] = qwk
        print(f" — val_qwk: {qwk:.4f}")


class CleanTrainMetrics(keras.callbacks.Callback):
    """
    Log clean_train_loss / clean_train_accuracy on a fixed training subset (monitoring only).

    Keras' own training accuracy is measured on augmented batches, with dropout
    active and (optionally) class-weighted loss, so it is not comparable with
    validation. This callback re-scores a fixed stratified subset of TRAINING
    images in inference mode, without augmentation or class weights, using the
    same label-smoothed loss as val_loss — a like-for-like train/validation view.
    """

    def __init__(self, clean_dataset: tf.data.Dataset, clean_labels: np.ndarray):
        super().__init__()
        self.clean_dataset = clean_dataset
        self.clean_labels = np.asarray(clean_labels, dtype=int)
        self.loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=Config.LABEL_SMOOTHING)

    def on_epoch_end(self, epoch, logs=None):
        probabilities = self.model.predict(self.clean_dataset, verbose=0)
        if len(probabilities) != len(self.clean_labels):
            raise ValueError("CleanTrainMetrics: prediction count does not match subset labels.")
        one_hot = tf.one_hot(self.clean_labels, Config.NUM_CLASSES)
        clean_loss = float(self.loss_fn(one_hot, probabilities))
        clean_accuracy = float(np.mean(np.argmax(probabilities, axis=1) == self.clean_labels))
        if logs is not None:
            logs["clean_train_loss"] = clean_loss
            logs["clean_train_accuracy"] = clean_accuracy
        print(f" — clean_train_accuracy: {clean_accuracy:.4f} — clean_train_loss: {clean_loss:.4f}")


def make_clean_train_subset(
    train_frame: pd.DataFrame,
    n_images: int = Config.CLEAN_TRAIN_SUBSET_SIZE,
    seed: int = Config.SEED,
) -> pd.DataFrame:
    """Fixed, stratified subset of the training split used by CleanTrainMetrics."""
    if len(train_frame) <= n_images:
        return train_frame.reset_index(drop=True)
    subset, _ = train_test_split(
        train_frame, train_size=n_images, stratify=train_frame["label"], random_state=seed
    )
    return subset.sort_index().reset_index(drop=True)


CLEAN_TRAIN_SUBSET_DF = make_clean_train_subset(train_df)
assert set(CLEAN_TRAIN_SUBSET_DF["split"]) == {"train"}, "The clean-train subset must come from the training split."
# Legend explanation shared by the final-fit curves (6.4) and pilot evidence plots (6.2b);
# the subset size is taken from the subset actually built.
CURVE_LEGEND_NOTE = (
    "augmented = training batches with augmentation, dropout and class weights; "
    f"clean subset = {len(CLEAN_TRAIN_SUBSET_DF):,} fixed training images scored like validation"
)
print(f"[Callbacks] Clean-train subset: {len(CLEAN_TRAIN_SUBSET_DF):,} training images "
      f"{CLEAN_TRAIN_SUBSET_DF['label'].value_counts().sort_index().to_dict()} (seed {Config.SEED})")


def warmup_cosine_lr(epoch: int, total_epochs: int, peak_lr: float,
                     warmup_epochs: int = Config.LR_WARMUP_EPOCHS) -> float:
    """
    Learning rate for one epoch of the warm-up + cosine schedule.

    Epochs 0..warmup-1 rise linearly from 10% of peak_lr towards peak_lr;
    the remaining epochs follow a cosine from peak_lr down to 1% of peak_lr
    at the final epoch.
    """
    warmup_epochs = max(0, min(warmup_epochs, total_epochs - 1))
    if epoch < warmup_epochs:
        return float(peak_lr * (0.1 + 0.9 * epoch / warmup_epochs))
    decay_epochs = max(1, total_epochs - warmup_epochs - 1)
    progress = min(1.0, (epoch - warmup_epochs) / decay_epochs)
    minimum_lr = 0.01 * peak_lr
    return float(minimum_lr + 0.5 * (peak_lr - minimum_lr) * (1.0 + math.cos(math.pi * progress)))


def lr_schedule_callbacks(schedule: str, total_epochs: int, peak_lr: float) -> list:
    """Return the learning-rate callback for 'plateau' (ReduceLROnPlateau) or 'cosine' (warm-up + cosine)."""
    if schedule == "cosine":
        return [keras.callbacks.LearningRateScheduler(
            lambda epoch, lr: warmup_cosine_lr(epoch, total_epochs, peak_lr), verbose=0
        )]
    if schedule == "plateau":
        return [ReduceLROnPlateau(
            monitor="val_loss",
            factor=Config.RLROP_FACTOR,
            patience=Config.RLROP_PATIENCE,
            min_lr=1e-7,
            verbose=1,
            mode="min",
        )]
    raise ValueError(f"Unknown learning-rate schedule: {schedule!r} (use 'plateau' or 'cosine').")


def clean_train_callback(image_size: int, preprocessing: Optional[Dict[str, bool]] = None) -> CleanTrainMetrics:
    """CleanTrainMetrics on the fixed training subset, built at the current image size and preprocessing."""
    clean_dataset = _build_tuning_dataset(
        CLEAN_TRAIN_SUBSET_DF, image_size, shuffle=False, augment=False, preprocessing=preprocessing,
    )
    return CleanTrainMetrics(clean_dataset, CLEAN_TRAIN_SUBSET_DF["label"].to_numpy())


def restored_epoch_index(history: keras.callbacks.History) -> int:
    """0-based epoch whose weights EarlyStopping restores (best Config.STOP_MONITOR value)."""
    values = np.asarray(history.history[Config.STOP_MONITOR], dtype=float)
    return int(np.nanargmax(values) if Config.STOP_MODE == "max" else np.nanargmin(values))


def build_callbacks(
    phase_name: str,
    validation_dataset: Optional[tf.data.Dataset] = None,
    validation_labels: Optional[np.ndarray] = None,
    lr_schedule: str = "plateau",
    total_epochs: Optional[int] = None,
    peak_lr: Optional[float] = None,
) -> list:
    """Build val_qwk-driven stopping/checkpointing, the LR schedule, and monitoring callbacks."""
    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
    checkpoint_path = os.path.join(
        Config.CHECKPOINT_DIR,
        f"best_{phase_name}.weights.h5",
    )
    validation_dataset = val_dataset if validation_dataset is None else validation_dataset
    validation_labels = val_df["label"].to_numpy() if validation_labels is None else validation_labels

    callbacks = [
        ValidationQWK(validation_dataset, validation_labels),  # must stay first: provides val_qwk
        clean_train_callback(Config.IMG_SIZE),                 # monitoring only
        LearningRateLogger(),  # logs the LR used for this epoch
        EarlyStopping(
            monitor=Config.STOP_MONITOR,
            patience=Config.STOP_PATIENCE,
            restore_best_weights=True,
            verbose=1,
            mode=Config.STOP_MODE,
        ),
        *lr_schedule_callbacks(lr_schedule, total_epochs or Config.PHASE2_EPOCHS, peak_lr or Config.PHASE2_LR),
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor=Config.STOP_MONITOR,
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
            mode=Config.STOP_MODE,
        ),
    ]

    print(f"[Callbacks] Built for {phase_name}:")
    print(f"  EarlyStopping     : patience={Config.STOP_PATIENCE} ({Config.STOP_MONITOR}, mode {Config.STOP_MODE}, restore best weights)")
    if lr_schedule == "cosine":
        print(f"  LR schedule       : warm-up {Config.LR_WARMUP_EPOCHS} epochs + cosine to 1% of {peak_lr or Config.PHASE2_LR}")
    else:
        print(f"  ReduceLROnPlateau : factor={Config.RLROP_FACTOR}, patience={Config.RLROP_PATIENCE} (val_loss)")
    print(f"  ModelCheckpoint   : -> {checkpoint_path} (best {Config.STOP_MONITOR}, mode {Config.STOP_MODE})")
    print("  ValidationQWK     : val_qwk logged each epoch (drives stopping and checkpointing)")
    print(f"  CleanTrainMetrics : clean_train_loss/accuracy on {len(CLEAN_TRAIN_SUBSET_DF)} training images (monitoring only)")
    print("  LearningRateLogger: learning_rate saved in the training history")
    return callbacks


In [ ]:
# Fingerprint the fixed manifests so every tuning row proves it used the same splits.
_tuning_manifest_rows = pd.concat(
    [
        frame[["patient_id", "duplicate_group", "label"]].assign(split=split_name)
        for split_name, frame in (
            ("train", train_df),
            ("validation", val_df),
            ("test", test_df),
        )
    ],
    ignore_index=True,
)
_tuning_manifest_rows = _tuning_manifest_rows.astype("string").fillna("").sort_values(
    ["split", "patient_id", "duplicate_group", "label"]
)
_split_fingerprint = hashlib.sha256(
    _tuning_manifest_rows.to_csv(index=False, lineterminator="\n").encode("utf-8")
).hexdigest()
print("[Tuning] Fixed manifest fingerprint:", _split_fingerprint)

### Pilot Trials and Final Fit

The pilot runner trains every configuration listed in Section 5 on the **full** training split and scores it on the **full** validation split (up to 4 Phase 1 + 6 Phase 2 epochs, validation-loss early stopping with patience 2). For each trial it saves `pilots/<trial_id>/history.csv`, a training-curve PNG (augmented-train, clean-train-subset and validation lines; val_qwk) and a validation confusion-matrix PNG, and records trainable parameters, epochs run, the train–validation and clean-train–validation accuracy gaps at the best epoch, the validation-loss rise after its minimum and the training time. It then selects among the completed hyperparameter and balancing trials (9 with the default skip list) by validation QWK, breaking ties with lower validation loss, and additionally prints (for discussion only) the trial with the highest validation accuracy and the one with the smallest clean-train gap. These are pilot results, not final model metrics.

After selection, the notebook rebuilds the datasets at the winning resolution and batch size (and with oversampling, the chosen class-weight mode, LR schedule and weight decay if that trial won) and runs the normal Phase 1 and Phase 2 epoch limits on the complete training split. The resulting `best_phase1.weights.h5` and `best_phase2.weights.h5` are the final model checkpoints. The held-out test split is excluded from all of this and is evaluated only after this final fit. Set `RUN_HYPERPARAMETER_TUNING=False` to skip the pilots and use the configured single-run settings.

Each pilot output row also includes the reason for the change and the hypothesis being tested. These are experiment rationales, not assumptions that a setting will improve performance; the validation metrics determine the observed result.

In [ ]:
# ============================================================
# SECTION 6.2b — VALIDATION-ONLY PILOT TRIALS: HYPERPARAMETERS,
#                BALANCING, ABLATIONS AND BACKBONE COMPARISON
#
# Every trial trains on the FULL training split and is scored on the FULL
# validation split (TUNING_*_FRACTION = 1.0), for up to 4 Phase 1 + 6
# Phase 2 epochs with val_loss early stopping (patience 2), so the curves
# are long enough to show overfitting and the rows are directly comparable.
#
# Trial groups:
#   hyperparameter          -> selectable; tunes the final EfficientNetB3
#   balancing               -> selectable; class weights vs none vs oversampling
#   preprocessing_ablation  -> evidence only; is Ben Graham actually helping?
#   augmentation_ablation   -> evidence only; is augmentation actually helping?
#   backbone                -> evidence only; other ImageNet backbones
#   scratch_baseline        -> evidence only; small CNN with no pretraining
#
# Selection: highest validation QWK (tie-break: lower validation loss)
# among the hyperparameter and balancing groups only. The deployed pipeline
# (Grad-CAM layer, embeddings, Gradio app) is built around EfficientNetB3
# with Ben Graham preprocessing, so ablation/backbone rows are evidence.
# The held-out test split is never used to train, stop or rank any trial.
#
# Proof saved per trial in report_images/pilots/<trial_id>/:
#   history.csv, training_curves.png (accuracy and loss with augmented-train,
#   clean-train-subset and validation lines; val_qwk) and
#   validation_confusion_matrix.png.
# Trial keys batch_size, weight_decay, phase2_lr_schedule and
# class_weight_mode are applied to Config, so a selected trial (e.g.
# resolution_380 with batch 8) carries through to the final fit.
#
# Compute limits (Config, Section 1.2):
#   PILOT_SKIP_TRIALS       -> kept defined but not trained ("skipped_by_config")
#   PILOT_TRIAL_LIMIT       -> only the first N non-skipped trials ("skipped_trial_limit")
#   PILOT_TIME_BUDGET_HOURS -> no new trial starts after the budget ("skipped_time_budget")
#   RESUME_RUN_ID           -> reuse completed rows of hyperparameter_tuning_results.csv
#                              whose split_fingerprint matches the current split
# Only rows with status "completed" are selectable or shown in 6.2c/6.2d.
# After every trial the model, datasets, callbacks, augmentation layer,
# histories and predictions are deleted, Keras is cleared and process RAM
# and GPU memory are recorded (ram_gb_after_trial, gpu_mem_gb_after_trial).
# ============================================================

import gc
import time
import inspect
import psutil
import hashlib
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

RUN_HYPERPARAMETER_TUNING = True
RUN_COMPARISON_TRIALS = True   # set False to skip the ablation, backbone and scratch rows
TUNING_TRAIN_FRACTION = 1.0
TUNING_VALIDATION_FRACTION = 1.0
TUNING_PHASE1_EPOCHS = 4
TUNING_PHASE2_EPOCHS = 6
PILOT_ES_PATIENCE = 2
SELECTABLE_GROUPS = {"hyperparameter", "balancing"}
PILOT_DIR = Path(Config.REPORTS_DIR) / "pilots"

# Baseline settings shared by every trial unless a trial overrides them
# (matches the Config defaults in Section 1.2).
PILOT_BASELINE = {
    "image_size": 300,
    "dropout_rate": 0.5,
    "phase1_lr": 1e-3,
    "phase2_lr": 1e-5,
    "unfreeze_top_n": 120,
    "backbone": "efficientnetb3",
    "preprocessing": {},        # {} = default pipeline: crop -> resize -> Ben Graham
    "augment": True,
    "class_weight_mode": "balanced",   # "balanced", "sqrt" or "none"
    "oversample": False,
    "batch_size": 16,
    "weight_decay": 1e-4,
    "phase2_lr_schedule": "plateau",   # "plateau" or "cosine"
}


def _pilot_trial(trial_id: str, group: str, change: str, rationale: str, **overrides) -> Dict[str, Any]:
    """Build one pilot trial definition from the baseline plus overrides."""
    trial = {**PILOT_BASELINE, **overrides}
    trial["class_weights"] = trial["class_weight_mode"] != "none"
    check_balancing_choice(trial["class_weights"], trial["oversample"])
    trial.update({
        "trial_id": trial_id,
        "group": group,
        "change_vs_baseline": change,
        "trial_rationale": rationale,
    })
    return trial


TUNING_TRIALS = [
    _pilot_trial(
        "baseline_300_d05_lr1e5_top120", "hyperparameter", "Baseline",
        "Reference configuration: dropout 0.5, top-120 fine-tuning, Phase 2 LR 1e-5 with plateau "
        "reduction, balanced class weights, weight decay 1e-4, batch 16.",
    ),
    _pilot_trial(
        "dropout_040", "hyperparameter", "Dropout 0.5 -> 0.4",
        "Test whether lighter head regularization learns more without widening the train-validation gap.",
        dropout_rate=0.4,
    ),
    _pilot_trial(
        "unfreeze_top60", "hyperparameter", "Fine-tune 120 -> 60 layers",
        "Test whether adapting fewer late layers reduces overfitting on about 2,560 training images.",
        unfreeze_top_n=60,
    ),
    _pilot_trial(
        "unfreeze_top240", "hyperparameter", "Fine-tune 120 -> 240 layers",
        "Test whether adapting more of the backbone helps despite the higher overfitting risk.",
        unfreeze_top_n=240,
    ),
    _pilot_trial(
        "phase2_lr_3e5", "hyperparameter", "Phase 2 LR 1e-5 -> 3e-5",
        "Test whether a larger fine-tuning step adapts faster or destabilises the pretrained features.",
        phase2_lr=3e-5,
    ),
    _pilot_trial(
        "resolution_224", "hyperparameter", "Input resolution 300 -> 224",
        "Test whether lower compute and memory cost can retain useful grading performance; "
        "smaller inputs may lose fine retinal detail.",
        image_size=224,
    ),
    _pilot_trial(
        "resolution_380", "hyperparameter", "Input resolution 300 -> 380 (batch 16 -> 8)",
        "Test whether finer retinal detail (microaneurysms) helps; batch size halved to fit GPU memory.",
        image_size=380, batch_size=8,
    ),
    _pilot_trial(
        "lr_schedule_cosine", "hyperparameter", "Phase 2 LR plateau -> warm-up + cosine",
        "Test whether a short warm-up followed by smooth cosine decay fine-tunes more stably than "
        "step reductions on a validation-loss plateau.",
        phase2_lr_schedule="cosine",
    ),
    _pilot_trial(
        "weight_decay_1e3", "hyperparameter", "AdamW weight decay 1e-4 -> 1e-3",
        "Test whether stronger weight decay in Phase 2 narrows the train-validation gap.",
        weight_decay=1e-3,
    ),
    _pilot_trial(
        "class_weights_off", "balancing", "Class weights ON -> OFF",
        "Measure how much inverse-frequency weighting changes minority-stage recall and overall QWK.",
        class_weight_mode="none",
    ),
    _pilot_trial(
        "class_weights_sqrt", "balancing", "Balanced -> sqrt (softer) class weights",
        "Test whether a softer correction keeps minority-stage recall without over-weighting the "
        "rare Severe images.",
        class_weight_mode="sqrt",
    ),
    _pilot_trial(
        "oversampling_instead", "balancing", "Class weights -> on-the-fly oversampling",
        "Balance batches by sampling each stage equally instead of re-weighting the loss.",
        class_weight_mode="none", oversample=True,
    ),
]

COMPARISON_TRIALS = [
    _pilot_trial(
        "ablation_no_ben_graham", "preprocessing_ablation", "Ben Graham ON -> OFF (crop + resize only)",
        "Measure the model-level contribution of Ben Graham illumination normalisation.",
        preprocessing={"apply_ben_graham": False},
    ),
    _pilot_trial(
        "ablation_clahe_instead", "preprocessing_ablation", "Ben Graham -> CLAHE (LAB L-channel)",
        "Compare histogram-based local contrast (CLAHE) against Gaussian background subtraction.",
        preprocessing={"apply_ben_graham": False, "apply_clahe_enhancement": True},
    ),
    _pilot_trial(
        "ablation_all_filters", "preprocessing_ablation", "Add bilateral denoise + CLAHE + unsharp edges before Ben Graham",
        "Test whether explicit noise removal and edge enhancement add information beyond Ben Graham.",
        preprocessing={"apply_denoise": True, "apply_clahe_enhancement": True, "apply_edge_enhancement": True},
    ),
    _pilot_trial(
        "ablation_no_augmentation", "augmentation_ablation", "Augmentation ON -> OFF",
        "Measure the contribution of training-only geometric and photometric augmentation.",
        augment=False,
    ),
    _pilot_trial(
        "backbone_resnet50", "backbone", "EfficientNetB3 -> ResNet50",
        "Residual backbone with more parameters and FLOPs; tests whether the EfficientNet choice is justified.",
        backbone="resnet50",
    ),
    _pilot_trial(
        "backbone_mobilenetv2", "backbone", "EfficientNetB3 -> MobileNetV2",
        "Lightweight mobile backbone; quantifies the accuracy cost of a deployable low-compute model.",
        backbone="mobilenetv2",
    ),
    _pilot_trial(
        "backbone_efficientnetb0", "backbone", "EfficientNetB3 -> EfficientNetB0",
        "Smaller member of the same family; tests whether B3's extra capacity is needed on 3.6k images.",
        backbone="efficientnetb0",
    ),
    _pilot_trial(
        "backbone_densenet121", "backbone", "EfficientNetB3 -> DenseNet121",
        "Densely connected backbone widely used in medical imaging.",
        backbone="densenet121",
    ),
    _pilot_trial(
        "backbone_vgg16", "backbone", "EfficientNetB3 -> VGG16",
        "Classic plain convolutional backbone; a large, older architecture for reference.",
        backbone="vgg16",
    ),
    _pilot_trial(
        "scratch_cnn_4block", "scratch_baseline", "EfficientNetB3 (ImageNet) -> small CNN from scratch",
        "Transfer-learning baseline: 4 conv blocks with random initialisation, trained end to end for the "
        "same total epoch budget, to show what ImageNet pretraining contributes.",
        backbone="scratch_cnn",
    ),
]


def _preprocessing_tag(preprocessing: Optional[Dict[str, bool]]) -> str:
    """Cache-folder suffix for non-default preprocessing (empty for the default pipeline)."""
    if not preprocessing:
        return ""
    return "_" + "_".join(
        f"{key.replace('apply_', '')}-{int(bool(value))}"
        for key, value in sorted(preprocessing.items())
    )


def _tuning_cache_path(filepath: str, image_size: int, preprocessing: Optional[Dict[str, bool]] = None) -> Path:
    cache_dir = Path(_cache_root) / "preproc_cache" / f"{image_size}{_preprocessing_tag(preprocessing)}"
    return cache_dir / (hashlib.md5(str(filepath).encode("utf-8")).hexdigest() + ".png")


def _cache_tuning_images(
    filepaths: List[str],
    image_size: int,
    preprocessing: Optional[Dict[str, bool]] = None,
) -> List[str]:
    """Preprocess each image once per (size, preprocessing variant) and cache it as PNG."""
    preprocessing = preprocessing or {}
    cache_dir = _tuning_cache_path("x", image_size, preprocessing).parent
    cache_dir.mkdir(parents=True, exist_ok=True)

    def _cache_one(filepath: str) -> str:
        output_path = _tuning_cache_path(filepath, image_size, preprocessing)
        if not output_path.is_file():
            image = preprocess_image(filepath, img_size=image_size, **preprocessing)
            image_uint8 = np.clip(image * 255.0, 0, 255).round().astype(np.uint8)
            written = cv2.imwrite(
                str(output_path),
                cv2.cvtColor(image_uint8, cv2.COLOR_RGB2BGR),
                [cv2.IMWRITE_PNG_COMPRESSION, 1],
            )
            if not written:
                raise OSError(f"Could not write cache file {output_path} (check disk space and path length).")
        return str(output_path)

    workers = min(os.cpu_count() or 1, 8)
    with ThreadPoolExecutor(max_workers=workers) as executor:
        return list(
            tqdm(
                executor.map(_cache_one, filepaths),
                total=len(filepaths),
                desc=f"Caching {image_size}px{_preprocessing_tag(preprocessing)} images",
            )
        )


def _build_tuning_dataset(
    dataframe: pd.DataFrame,
    image_size: int,
    shuffle: bool,
    augment: bool,
    preprocessing: Optional[Dict[str, bool]] = None,
    oversample: bool = False,
) -> tf.data.Dataset:
    """Batched dataset from cached PNGs; oversample=True gives an infinite class-balanced stream."""
    cached_paths = _cache_tuning_images(dataframe["filepath"].tolist(), image_size, preprocessing)
    labels = dataframe["label"].to_numpy(dtype=np.int32)

    def _load_cached_image(path: tf.Tensor, label: tf.Tensor) -> tuple:
        image = tf.io.decode_png(tf.io.read_file(path), channels=3)
        image = tf.image.convert_image_dtype(image, tf.float32)
        image.set_shape([image_size, image_size, 3])
        label = tf.one_hot(tf.cast(label, tf.int32), Config.NUM_CLASSES)
        return image, label

    if oversample:
        dataset = build_oversampled_dataset(cached_paths, labels, _load_cached_image)
    else:
        dataset = tf.data.Dataset.from_tensor_slices((cached_paths, labels))
        if shuffle:
            dataset = dataset.shuffle(
                buffer_size=len(cached_paths),
                seed=Config.SEED,
                reshuffle_each_iteration=True,
            )
        dataset = dataset.map(_load_cached_image, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        dataset = dataset.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(Config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def _stratified_pilot_sample(
    dataframe: pd.DataFrame,
    fraction: float,
    seed: int,
) -> pd.DataFrame:
    if not 0 < fraction <= 1:
        raise ValueError("Pilot sample fractions must be in (0, 1].")
    if fraction == 1:
        return dataframe.reset_index(drop=True)
    sampled_groups = []
    for label, group in dataframe.groupby("label", sort=True):
        sample_count = min(len(group), max(1, int(np.ceil(len(group) * fraction))))
        sampled_groups.append(group.sample(sample_count, random_state=seed + int(label)))
    return pd.concat(sampled_groups).sort_index().reset_index(drop=True)


# Backbone registry. Every backbone receives images in [0, 1]; the adapter
# converts them to the input convention that backbone was pretrained with.
_BACKBONE_CONSTRUCTORS = {
    "efficientnetb3": keras.applications.EfficientNetB3,   # expects [0, 255], rescales internally
    "efficientnetb0": keras.applications.EfficientNetB0,   # expects [0, 255], rescales internally
    "resnet50": keras.applications.ResNet50,               # Caffe-style BGR mean-subtracted [0, 255]
    "vgg16": keras.applications.VGG16,                     # Caffe-style BGR mean-subtracted [0, 255]
    "mobilenetv2": keras.applications.MobileNetV2,         # expects [-1, 1]
    "densenet121": keras.applications.DenseNet121,         # Torch-style ImageNet mean/std normalised
}
_CAFFE_BGR_MEAN = (103.939, 116.779, 123.68)
_TORCH_RGB_MEAN = (0.485, 0.456, 0.406)
_TORCH_RGB_STD = (0.229, 0.224, 0.225)


def _backbone_input_adapter(backbone_name: str, inputs: keras.KerasTensor) -> keras.KerasTensor:
    """Convert [0, 1] images to the numeric range each ImageNet backbone was trained on."""
    if backbone_name in ("efficientnetb3", "efficientnetb0"):
        return layers.Rescaling(255.0, name="restore_efficientnet_input_range")(inputs)
    if backbone_name == "mobilenetv2":
        return layers.Rescaling(2.0, offset=-1.0, name="mobilenetv2_scale_to_pm1")(inputs)
    if backbone_name in ("resnet50", "vgg16"):
        scaled = layers.Rescaling(255.0, name=f"{backbone_name}_scale_to_0_255")(inputs)
        return layers.Lambda(
            lambda t: keras.ops.flip(t, axis=-1)
            - keras.ops.convert_to_tensor(_CAFFE_BGR_MEAN, dtype=t.dtype),
            name=f"{backbone_name}_caffe_bgr_mean_subtract",
        )(scaled)
    if backbone_name == "densenet121":
        return layers.Normalization(
            mean=list(_TORCH_RGB_MEAN),
            variance=[std ** 2 for std in _TORCH_RGB_STD],
            name="densenet121_torch_mean_std",
        )(inputs)
    if backbone_name == "scratch_cnn":
        return inputs  # trained from scratch on [0, 1] inputs; no pretrained convention
    raise ValueError(f"Unknown backbone: {backbone_name}")


def build_scratch_cnn(image_size: int) -> keras.Model:
    """Small 4-block CNN (Conv-BN-ReLU-MaxPool, 32/64/128/256 filters) with random initialisation."""
    inputs = keras.Input(shape=(image_size, image_size, 3))
    x = inputs
    for block, filters in enumerate((32, 64, 128, 256), start=1):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"scratch_block{block}_conv")(x)
        x = layers.BatchNormalization(name=f"scratch_block{block}_bn")(x)
        x = layers.Activation("relu", name=f"scratch_block{block}_relu")(x)
        x = layers.MaxPooling2D(2, name=f"scratch_block{block}_pool")(x)
    return keras.Model(inputs, x, name="scratch_cnn_backbone")


def _build_tuning_classifier(backbone_name: str = "efficientnetb3") -> tuple:
    """Build the shared classification head on top of the requested backbone."""
    if backbone_name == "scratch_cnn":
        backbone = build_scratch_cnn(Config.IMG_SIZE)
        backbone.trainable = True  # no pretrained weights to protect
    elif backbone_name in _BACKBONE_CONSTRUCTORS:
        backbone = _BACKBONE_CONSTRUCTORS[backbone_name](
            include_top=False,
            weights="imagenet",
            input_shape=(Config.IMG_SIZE, Config.IMG_SIZE, 3),
        )
        backbone.trainable = False
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    inputs = keras.Input(shape=(Config.IMG_SIZE, Config.IMG_SIZE, 3))
    features = _backbone_input_adapter(backbone_name, inputs)
    if backbone_name == "scratch_cnn":
        features = backbone(features)  # BatchNorm follows the fit/predict mode
    else:
        features = backbone(features, training=False)  # keep pretrained BatchNorm in inference mode
    features = layers.GlobalAveragePooling2D(name="gap")(features)
    features = layers.BatchNormalization(name="head_bn")(features)
    features = layers.Dense(Config.DENSE_UNITS, activation="relu", name="head_dense")(features)
    features = layers.Dropout(Config.DROPOUT_RATE, name="head_dropout")(features)
    outputs = layers.Dense(
        Config.NUM_CLASSES,
        activation="softmax",
        dtype="float32",
        name="predictions",
    )(features)
    model_name = "RetinaGuard_EfficientNetB3" if backbone_name == "efficientnetb3" else f"Pilot_{backbone_name}"
    classifier = keras.Model(inputs=inputs, outputs=outputs, name=model_name)
    return classifier, backbone


def _apply_trial_to_config(trial: Dict[str, Any]) -> None:
    check_balancing_choice(bool(trial["class_weights"]), bool(trial["oversample"]))
    Config.IMG_SIZE = int(trial["image_size"])
    Config.DROPOUT_RATE = float(trial["dropout_rate"])
    Config.PHASE1_LR = float(trial["phase1_lr"])
    Config.PHASE2_LR = float(trial["phase2_lr"])
    Config.UNFREEZE_TOP_N = int(trial["unfreeze_top_n"])
    Config.USE_CLASS_WEIGHTS = bool(trial["class_weights"])
    Config.CLASS_WEIGHT_MODE = str(trial["class_weight_mode"])
    Config.USE_OVERSAMPLING = bool(trial["oversample"])
    Config.BATCH_SIZE = int(trial["batch_size"])
    Config.WEIGHT_DECAY = float(trial["weight_decay"])
    Config.PHASE2_LR_SCHEDULE = str(trial["phase2_lr_schedule"])


def _count_trainable_parameters(m: keras.Model) -> int:
    return int(sum(int(np.prod(v.shape)) for v in m.trainable_variables))


def _combine_histories(histories: List[Tuple[str, keras.callbacks.History]]) -> pd.DataFrame:
    """Concatenate per-phase Keras histories into one epoch-indexed table."""
    frames = []
    for phase_name, history in histories:
        frame = pd.DataFrame(history.history)
        frame.insert(0, "phase", phase_name)
        frame.insert(1, "phase_epoch", np.arange(1, len(frame) + 1))
        frames.append(frame)
    combined = pd.concat(frames, ignore_index=True)
    combined.insert(0, "epoch", np.arange(1, len(combined) + 1))
    return combined


def _save_pilot_evidence(trial_id: str, history_df: pd.DataFrame, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    """Save history.csv, training-curve PNG and validation confusion-matrix PNG for one trial."""
    trial_dir = PILOT_DIR / trial_id
    trial_dir.mkdir(parents=True, exist_ok=True)
    history_df.to_csv(trial_dir / "history.csv", index=False)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    epochs = history_df["epoch"]
    panels = (
        ("accuracy", "clean_train_accuracy", "val_accuracy", "Accuracy"),
        ("loss", "clean_train_loss", "val_loss", "Loss (label-smoothed CE)"),
        (None, None, "val_qwk", "Validation QWK"),
    )
    boundaries = history_df.index[history_df["phase"] != history_df["phase"].shift()].tolist()[1:]
    for axis, (train_key, clean_key, val_key, title) in zip(axes, panels):
        if train_key and train_key in history_df:
            axis.plot(epochs, history_df[train_key], "b-o", markersize=3, label="Train (augmented)")
        if clean_key and clean_key in history_df:
            axis.plot(epochs, history_df[clean_key], "c--s", markersize=3, label="Train (clean subset)")
        if val_key in history_df:
            axis.plot(epochs, history_df[val_key], "r-o", markersize=3, label="Validation")
        for boundary in boundaries:
            axis.axvline(history_df.loc[boundary, "epoch"] - 0.5, color="grey", linestyle="--", linewidth=1)
        axis.set_title(title)
        axis.set_xlabel("Epoch (Phase 1 then Phase 2)")
        axis.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        axis.grid(alpha=0.3)
        axis.legend(fontsize=8)
    fig.suptitle(f"Pilot trial {trial_id} — train vs validation (dashed line = phase change)", fontsize=12)
    fig.text(0.5, -0.02, CURVE_LEGEND_NOTE, ha="center", fontsize=9, color="#475569")
    fig.tight_layout()
    fig.savefig(trial_dir / "training_curves.png", dpi=130, bbox_inches="tight")
    plt.close(fig)

    matrix = confusion_matrix(y_true, y_pred, labels=range(Config.NUM_CLASSES))
    fig, axis = plt.subplots(figsize=(6.5, 5.5))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", ax=axis,
                xticklabels=Config.CLASS_NAMES, yticklabels=Config.CLASS_NAMES)
    axis.set_title(f"{trial_id}\nValidation confusion matrix")
    axis.set_xlabel("Predicted stage")
    axis.set_ylabel("True stage")
    axis.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(trial_dir / "validation_confusion_matrix.png", dpi=130, bbox_inches="tight")
    plt.close(fig)


def _process_memory_gb() -> Tuple[float, float]:
    """Process RAM (resident set size) and GPU:0 memory in use, in GB (GPU is NaN when unavailable)."""
    ram_gb = psutil.Process().memory_info().rss / 1024 ** 3
    try:
        gpu_gb = tf.config.experimental.get_memory_info("GPU:0")["current"] / 1024 ** 3
    except Exception:  # no GPU, or memory info not supported on this device
        gpu_gb = float("nan")
    return float(ram_gb), float(gpu_gb)


def _release_keras_memory() -> None:
    """Clear Keras' global state (freeing memory where supported), then run the garbage collector."""
    # clear_session() also resets the global dtype policy to float32, so the
    # policy set in Section 5.1 (mixed_float16 on GPU) is restored afterwards.
    precision_policy = keras.mixed_precision.global_policy().name
    if "free_memory" in inspect.signature(keras.backend.clear_session).parameters:
        keras.backend.clear_session(free_memory=True)
    else:
        keras.backend.clear_session()
    keras.mixed_precision.set_global_policy(precision_policy)
    gc.collect()


def _pilot_selection_key(validation_qwk: float, validation_loss: float) -> Tuple[float, float]:
    """Selection rule: highest validation QWK, tie-break lower validation loss."""
    return (validation_qwk if np.isfinite(validation_qwk) else -1.0, -validation_loss)


def _select_best_trial(
    trial_records: List[Dict[str, Any]],
    trials_by_id: Dict[str, Dict[str, Any]],
) -> Optional[Dict[str, Any]]:
    """Best completed selectable trial (new or resumed rows alike), in trial order."""
    best_trial, best_key = None, (-np.inf, -np.inf)
    for record in trial_records:
        trial = trials_by_id[str(record["trial_id"])]
        if record["status"] != "completed" or trial["group"] not in SELECTABLE_GROUPS:
            continue
        key = _pilot_selection_key(float(record["pilot_validation_qwk"]), float(record["pilot_validation_loss"]))
        if key > best_key:
            best_key, best_trial = key, trial.copy()
    return best_trial


def _not_run_record(trial: Dict[str, Any], status: str) -> Dict[str, Any]:
    """Tuning-CSV row for a trial that was not trained in this run (metrics left empty)."""
    return {
        "run_id": _RUN_ID,
        "trial_id": str(trial["trial_id"]),
        "group": trial["group"],
        "status": status,
        "selectable_for_final_model": trial["group"] in SELECTABLE_GROUPS,
        "change_vs_baseline": trial["change_vs_baseline"],
        "trial_rationale": trial["trial_rationale"],
        "backbone": trial["backbone"],
        "split_fingerprint": _split_fingerprint,
    }


def _load_resumable_records(
    tuning_results_path: Path,
    trials_by_id: Dict[str, Dict[str, Any]],
) -> Dict[str, Dict[str, Any]]:
    """Completed rows from an earlier session of this run folder, reusable only with the same split."""
    if not tuning_results_path.is_file():
        return {}
    previous = pd.read_csv(tuning_results_path, dtype={"trial_id": str, "split_fingerprint": str})
    if "status" not in previous.columns:
        # Files written before the status column existed only ever held completed trials.
        previous["status"] = "completed"
    resumable = {}
    for row in previous.to_dict("records"):
        trial_id = str(row["trial_id"])
        if row["status"] != "completed" or trial_id not in trials_by_id:
            continue
        if str(row.get("split_fingerprint")) != _split_fingerprint:
            print(f"[Resume] {trial_id}: saved with a different split fingerprint; it will be retrained.")
            continue
        resumable[trial_id] = row
    return resumable


def _write_tuning_results(trial_records: List[Dict[str, Any]], pending_resumed: List[Dict[str, Any]],
                          tuning_results_path: Path) -> pd.DataFrame:
    """Write every row so far (plus resumed rows not yet reached, so a crash never loses them)."""
    results = pd.DataFrame(trial_records + pending_resumed)
    front = ["run_id", "trial_id", "group", "status"]
    results = results[front + [column for column in results.columns if column not in front]]
    results.to_csv(tuning_results_path, index=False)
    return results


def _run_pilot_trial(
    trial: Dict[str, Any],
    pilot_train_df: pd.DataFrame,
    pilot_val_df: pd.DataFrame,
    pilot_true: np.ndarray,
) -> Dict[str, Any]:
    """Train and score one pilot trial, save its evidence, and free everything it built."""
    global augmentation_layer

    set_all_seeds(Config.SEED)
    trial_id = str(trial["trial_id"])
    _apply_trial_to_config(trial)

    print("\n" + "=" * 76)
    print(f"[Pilot] {trial_id}  (group: {trial['group']})")
    print(f"        change: {trial['change_vs_baseline']}")
    print("=" * 76)

    augmentation_layer = build_augmentation_layer()
    pilot_train_dataset = _build_tuning_dataset(
        pilot_train_df, Config.IMG_SIZE, shuffle=True,
        augment=bool(trial["augment"]), preprocessing=trial["preprocessing"],
        oversample=Config.USE_OVERSAMPLING,
    )
    pilot_val_dataset = _build_tuning_dataset(
        pilot_val_df, Config.IMG_SIZE, shuffle=False,
        augment=False, preprocessing=trial["preprocessing"],
    )
    pilot_steps = int(np.ceil(len(pilot_train_df) / Config.BATCH_SIZE)) if Config.USE_OVERSAMPLING else None
    pilot_class_weights = active_class_weights()
    pilot_clean_callback = clean_train_callback(Config.IMG_SIZE, trial["preprocessing"])

    def _pilot_callbacks(schedule: str, total_epochs: int, peak_lr: float) -> list:
        return [
            ValidationQWK(pilot_val_dataset, pilot_true),   # monitoring only
            pilot_clean_callback,                           # monitoring only
            LearningRateLogger(),
            EarlyStopping(monitor="val_loss", patience=PILOT_ES_PATIENCE,
                          restore_best_weights=True, mode="min", verbose=1),
            *lr_schedule_callbacks(schedule, total_epochs, peak_lr),
        ]

    pilot_model, pilot_base_model = _build_tuning_classifier(trial["backbone"])
    # Different backbones have different depths; never unfreeze more layers than exist.
    Config.UNFREEZE_TOP_N = min(Config.UNFREEZE_TOP_N, len(pilot_base_model.layers))
    # XLA off for pilots: each trial would otherwise keep its own compiled XLA cluster in host RAM.
    compile_model_phase1(pilot_model, jit_compile=False)

    fit_start = time.perf_counter()
    if trial["backbone"] == "scratch_cnn":
        # No pretrained features to freeze or fine-tune: train end to end for the same total budget.
        scratch_callbacks = _pilot_callbacks("plateau", TUNING_PHASE1_EPOCHS + TUNING_PHASE2_EPOCHS, Config.PHASE1_LR)
        histories = [("Scratch (end to end)", pilot_model.fit(
            pilot_train_dataset,
            epochs=TUNING_PHASE1_EPOCHS + TUNING_PHASE2_EPOCHS,
            steps_per_epoch=pilot_steps,
            validation_data=pilot_val_dataset,
            callbacks=scratch_callbacks,
            class_weight=pilot_class_weights,
            verbose=1,
        ))]
        del scratch_callbacks
    else:
        phase1_callbacks = _pilot_callbacks("plateau", TUNING_PHASE1_EPOCHS, Config.PHASE1_LR)
        history_phase1_pilot = pilot_model.fit(
            pilot_train_dataset,
            epochs=TUNING_PHASE1_EPOCHS,
            steps_per_epoch=pilot_steps,
            validation_data=pilot_val_dataset,
            callbacks=phase1_callbacks,
            class_weight=pilot_class_weights,
            verbose=1,
        )
        configure_phase2(pilot_model, pilot_base_model, jit_compile=False)
        phase2_callbacks = _pilot_callbacks(Config.PHASE2_LR_SCHEDULE, TUNING_PHASE2_EPOCHS, Config.PHASE2_LR)
        history_phase2_pilot = pilot_model.fit(
            pilot_train_dataset,
            epochs=TUNING_PHASE2_EPOCHS,
            steps_per_epoch=pilot_steps,
            validation_data=pilot_val_dataset,
            callbacks=phase2_callbacks,
            class_weight=pilot_class_weights,
            verbose=1,
        )
        histories = [("Phase 1", history_phase1_pilot), ("Phase 2", history_phase2_pilot)]
        del phase1_callbacks, phase2_callbacks, history_phase1_pilot, history_phase2_pilot
    training_seconds = time.perf_counter() - fit_start
    trainable_parameters = _count_trainable_parameters(pilot_model)

    pilot_probabilities = pilot_model.predict(pilot_val_dataset, verbose=0)
    pilot_pred = np.argmax(pilot_probabilities, axis=1)
    if len(pilot_true) != len(pilot_pred):
        raise ValueError(f"Pilot validation prediction count mismatch for {trial_id}.")

    history_df = _combine_histories(histories)
    best_index = int(np.nanargmin(history_df["val_loss"]))
    final_phase_val_loss = float(np.nanmin(histories[-1][1].history["val_loss"]))  # restored weights
    _save_pilot_evidence(trial_id, history_df, pilot_true, pilot_pred)

    pilot_accuracy = float(np.mean(pilot_true == pilot_pred))
    pilot_qwk = float(cohen_kappa_score(pilot_true, pilot_pred, weights="quadratic"))
    pilot_report = classification_report(
        pilot_true, pilot_pred,
        labels=list(range(Config.NUM_CLASSES)),
        target_names=Config.CLASS_NAMES,
        output_dict=True, zero_division=0,
    )
    record = {
        "run_id": _RUN_ID,
        "trial_id": trial_id,
        "group": trial["group"],
        "status": "completed",
        "selectable_for_final_model": trial["group"] in SELECTABLE_GROUPS,
        "change_vs_baseline": trial["change_vs_baseline"],
        "trial_rationale": trial["trial_rationale"],
        "result_stage": "pilot_screen",
        "backbone": trial["backbone"],
        "backbone_parameters": int(pilot_base_model.count_params()),
        "trainable_parameters": trainable_parameters,
        "preprocessing_variant": _preprocessing_tag(trial["preprocessing"]).lstrip("_") or "default_ben_graham",
        "augmentation": bool(trial["augment"]),
        "image_size": Config.IMG_SIZE,
        "dropout_rate": Config.DROPOUT_RATE,
        "phase1_learning_rate": Config.PHASE1_LR,
        "phase2_learning_rate": Config.PHASE2_LR,
        "phase2_unfreeze_top_n": Config.UNFREEZE_TOP_N,
        "class_weights_enabled": bool(Config.USE_CLASS_WEIGHTS),
        "class_weight_mode": Config.CLASS_WEIGHT_MODE,
        "oversampling_enabled": bool(Config.USE_OVERSAMPLING),
        "batch_size": Config.BATCH_SIZE,
        "weight_decay": Config.WEIGHT_DECAY,
        "phase2_lr_schedule": Config.PHASE2_LR_SCHEDULE,
        "seed": Config.SEED,
        "split_fingerprint": _split_fingerprint,
        "pilot_train_images": len(pilot_train_df),
        "pilot_validation_images": len(pilot_val_df),
        "pilot_train_fraction": TUNING_TRAIN_FRACTION,
        "pilot_validation_fraction": TUNING_VALIDATION_FRACTION,
        "epochs_run": len(history_df),
        "pilot_phase1_epochs_run": len(histories[0][1].history["val_loss"]),
        "pilot_phase2_epochs_run": len(histories[1][1].history["val_loss"]) if len(histories) > 1 else 0,
        "best_val_loss_epoch": best_index + 1,
        "train_val_accuracy_gap_at_best_epoch": float(
            history_df.loc[best_index, "accuracy"] - history_df.loc[best_index, "val_accuracy"]
        ),
        "clean_train_minus_validation_accuracy_gap_at_best_epoch": float(
            history_df.loc[best_index, "clean_train_accuracy"] - history_df.loc[best_index, "val_accuracy"]
        ),
        "val_loss_rise_after_min": float(history_df["val_loss"].iloc[-1] - history_df["val_loss"].iloc[best_index]),
        "training_time_seconds": round(training_seconds, 1),
        "pilot_validation_accuracy": pilot_accuracy,
        "pilot_validation_qwk": pilot_qwk,
        "pilot_validation_macro_f1": float(pilot_report["macro avg"]["f1-score"]),
        "pilot_validation_loss": final_phase_val_loss,
        **{
            f"pilot_recall_class_{class_id}": float(pilot_report[class_name]["recall"])
            for class_id, class_name in enumerate(Config.CLASS_NAMES)
        },
    }
    print(
        f"[Pilot] {trial_id}: validation accuracy={pilot_accuracy:.4f}, "
        f"QWK={pilot_qwk:.4f}, macro-F1={record['pilot_validation_macro_f1']:.4f}, "
        f"val loss={final_phase_val_loss:.4f}, {record['epochs_run']} epochs in {training_seconds:.0f}s"
    )

    # Free everything this trial built before the next one starts.
    del pilot_model, pilot_base_model
    del pilot_train_dataset, pilot_val_dataset, pilot_clean_callback, _pilot_callbacks
    del augmentation_layer
    del histories, history_df, pilot_probabilities, pilot_pred
    return record


def run_hyperparameter_tuning() -> pd.DataFrame:
    """Run the pilot trials (skip list, resume, limits), then prepare the winning selectable config."""
    global model, base_model, augmentation_layer
    global train_dataset, val_dataset, test_dataset
    global steps_per_epoch, validation_steps, test_steps
    global selected_tuning_trial, tuning_results_df, EXECUTED_TRIALS

    if not RUN_HYPERPARAMETER_TUNING:
        print("[Tuning] Disabled; the ordinary single-configuration path will be used.")
        return pd.DataFrame()

    EXECUTED_TRIALS = TUNING_TRIALS + (COMPARISON_TRIALS if RUN_COMPARISON_TRIALS else [])
    trials_by_id = {str(trial["trial_id"]): trial for trial in EXECUTED_TRIALS}
    defined_ids = {str(trial["trial_id"]) for trial in TUNING_TRIALS + COMPARISON_TRIALS}
    skip_ids = set(Config.PILOT_SKIP_TRIALS)
    unknown_skips = sorted(skip_ids - defined_ids)
    if unknown_skips:
        raise ValueError(f"Config.PILOT_SKIP_TRIALS lists unknown trial IDs: {unknown_skips}")
    # The first N non-skipped trials (all of them when PILOT_TRIAL_LIMIT is None).
    allowed_ids = [trial_id for trial_id in trials_by_id if trial_id not in skip_ids]
    if Config.PILOT_TRIAL_LIMIT is not None:
        allowed_ids = allowed_ids[: int(Config.PILOT_TRIAL_LIMIT)]
    if not any(trials_by_id[trial_id]["group"] in SELECTABLE_GROUPS for trial_id in allowed_ids):
        raise ValueError("At least one selectable (hyperparameter/balancing) trial must be allowed to run.")

    pilot_train_df = _stratified_pilot_sample(train_df, TUNING_TRAIN_FRACTION, Config.SEED)
    pilot_val_df = _stratified_pilot_sample(val_df, TUNING_VALIDATION_FRACTION, Config.SEED + 1000)
    assert set(pilot_train_df["split"]) == {"train"} and set(pilot_val_df["split"]) == {"validation"}, \
        "Pilot trials may only use the training and validation splits."
    pilot_true = pilot_val_df["label"].to_numpy(dtype=int)

    tuning_results_path = Path(Config.REPORTS_DIR) / "hyperparameter_tuning_results.csv"
    PILOT_DIR.mkdir(parents=True, exist_ok=True)
    resumable_records = _load_resumable_records(tuning_results_path, trials_by_id)
    resumed_ids = [trial_id for trial_id in allowed_ids if trial_id in resumable_records]
    if resumed_ids:
        print(f"[Resume] Reusing {len(resumed_ids)} completed trial(s) from {tuning_results_path}:")
        for trial_id in resumed_ids:
            row = resumable_records[trial_id]
            print(f"         {trial_id}: validation QWK={float(row['pilot_validation_qwk']):.4f}, "
                  f"val loss={float(row['pilot_validation_loss']):.4f}")
    elif Config.RESUME_RUN_ID:
        print("[Resume] No completed trials with the current split fingerprint were found.")

    print(
        f"[Tuning] Pilot data: {len(pilot_train_df):,}/{len(train_df):,} training and "
        f"{len(pilot_val_df):,}/{len(val_df):,} validation images. "
        f"{len(allowed_ids) - len(resumed_ids)} trials to train, {len(resumed_ids)} resumed, "
        f"{len(skip_ids & set(trials_by_id))} skipped by config, "
        f"{len(trials_by_id) - len(skip_ids & set(trials_by_id)) - len(allowed_ids)} beyond PILOT_TRIAL_LIMIT "
        f"(up to {TUNING_PHASE1_EPOCHS} Phase 1 + {TUNING_PHASE2_EPOCHS} Phase 2 epochs each; "
        f"time budget {Config.PILOT_TIME_BUDGET_HOURS} h)."
    )

    trial_records = []
    time_budget_exceeded = False
    pilot_loop_start = time.perf_counter()
    for trial in EXECUTED_TRIALS:
        trial_id = str(trial["trial_id"])
        if trial_id in skip_ids:
            trial_records.append(_not_run_record(trial, "skipped_by_config"))
            continue
        if trial_id not in allowed_ids:
            trial_records.append(_not_run_record(trial, "skipped_trial_limit"))
            continue
        if trial_id in resumable_records:
            trial_records.append(resumable_records[trial_id])
            continue

        elapsed_hours = (time.perf_counter() - pilot_loop_start) / 3600
        if (not time_budget_exceeded and Config.PILOT_TIME_BUDGET_HOURS is not None
                and elapsed_hours > Config.PILOT_TIME_BUDGET_HOURS):
            time_budget_exceeded = True
            print(f"\n[Tuning] Time budget reached ({elapsed_hours:.2f} h > {Config.PILOT_TIME_BUDGET_HOURS} h); "
                  "remaining trials are not started.")
        if time_budget_exceeded:
            trial_records.append(_not_run_record(trial, "skipped_time_budget"))
            continue

        record = _run_pilot_trial(trial, pilot_train_df, pilot_val_df, pilot_true)
        _release_keras_memory()
        ram_gb, gpu_gb = _process_memory_gb()
        record["ram_gb_after_trial"] = round(ram_gb, 3)
        record["gpu_mem_gb_after_trial"] = round(gpu_gb, 3)
        print(f"[Memory] After {trial_id}: process RAM {ram_gb:.2f} GB, GPU memory in use {gpu_gb:.2f} GB "
              f"({(time.perf_counter() - pilot_loop_start) / 3600:.2f} h since the pilot loop began)")
        trial_records.append(record)
        reached_ids = {str(row["trial_id"]) for row in trial_records}
        _write_tuning_results(
            trial_records,
            [resumable_records[i] for i in resumed_ids if i not in reached_ids],
            tuning_results_path,
        )

    tuning_results_df = _write_tuning_results(trial_records, [], tuning_results_path)
    print("\n[Pilot] Trial status counts:", tuning_results_df["status"].value_counts().to_dict())

    best_trial = _select_best_trial(trial_records, trials_by_id)
    if best_trial is None:
        raise RuntimeError("No valid selectable pilot trial completed.")

    # Rebuild the final EfficientNetB3 with the selected settings and default preprocessing.
    selected_tuning_trial = best_trial
    _apply_trial_to_config(best_trial)

    augmentation_layer = build_augmentation_layer()
    train_dataset = _build_tuning_dataset(
        train_df, Config.IMG_SIZE, shuffle=True, augment=True, oversample=Config.USE_OVERSAMPLING,
    )
    val_dataset = _build_tuning_dataset(val_df, Config.IMG_SIZE, shuffle=False, augment=False)
    test_dataset = _build_tuning_dataset(test_df, Config.IMG_SIZE, shuffle=False, augment=False)
    steps_per_epoch = int(np.ceil(len(train_df) / Config.BATCH_SIZE))
    validation_steps = len(val_df) // Config.BATCH_SIZE
    test_steps = len(test_df) // Config.BATCH_SIZE

    model, base_model = _build_tuning_classifier("efficientnetb3")
    Config.UNFREEZE_TOP_N = min(Config.UNFREEZE_TOP_N, len(base_model.layers))
    compile_model_phase1(model)  # final model: default jit_compile="auto"

    print("\n[Pilot] Selected settings for the final full-data fit:",
          {key: best_trial[key] for key in ("trial_id", "image_size", "dropout_rate", "phase1_lr",
                                            "phase2_lr", "unfreeze_top_n", "class_weight_mode", "oversample",
                                            "batch_size", "weight_decay", "phase2_lr_schedule")})
    print("[Pilot] Selection: highest validation QWK (tie-break: lower validation loss) among completed "
          f"trials in groups {sorted(SELECTABLE_GROUPS)}.")
    # Trade-off evidence for the report — printed only, NOT used for selection.
    selectable_rows = tuning_results_df[
        (tuning_results_df["status"] == "completed") & tuning_results_df["group"].isin(SELECTABLE_GROUPS)
    ]
    best_accuracy_row = selectable_rows.loc[selectable_rows["pilot_validation_accuracy"].idxmax()]
    smallest_gap_row = selectable_rows.loc[
        selectable_rows["clean_train_minus_validation_accuracy_gap_at_best_epoch"].abs().idxmin()
    ]
    print(f"[Pilot] (Not used for selection) Highest validation accuracy: {best_accuracy_row['trial_id']} "
          f"({best_accuracy_row['pilot_validation_accuracy']:.4f})")
    print(f"[Pilot] (Not used for selection) Smallest clean-train vs validation accuracy gap: "
          f"{smallest_gap_row['trial_id']} "
          f"({smallest_gap_row['clean_train_minus_validation_accuracy_gap_at_best_epoch']:+.4f})")
    print("[Pilot] All pilot rows saved to:", tuning_results_path)
    print("[Pilot] Per-trial history, curves and confusion matrices saved under:", PILOT_DIR)
    return tuning_results_df


if RUN_HYPERPARAMETER_TUNING:
    hyperparameter_tuning_results = run_hyperparameter_tuning()
else:
    print("[Tuning] Disabled; use the ordinary single-configuration training path.")


In [ ]:
# ============================================================
# SECTION 6.2c — MODEL COMPARISON TABLE AND DIFFERENCE FROM BASELINE
# One row per trial with validation metrics and overfitting evidence,
# plus each non-hyperparameter trial's difference from the baseline.
# All values are computed from the pilot runs above; validation split only.
# ============================================================

if RUN_HYPERPARAMETER_TUNING:
    # Skipped trials (config, trial limit, time budget) stay in the tuning CSV but not in these tables.
    completed_pilot_results = hyperparameter_tuning_results[
        hyperparameter_tuning_results["status"] == "completed"
    ].reset_index(drop=True)
    comparison_columns = [
        "trial_id", "group", "change_vs_baseline", "backbone",
        "trainable_parameters", "epochs_run", "training_time_seconds",
        "pilot_validation_qwk", "pilot_validation_macro_f1", "pilot_validation_accuracy",
        "pilot_validation_loss", "train_val_accuracy_gap_at_best_epoch",
        "clean_train_minus_validation_accuracy_gap_at_best_epoch", "val_loss_rise_after_min",
        *[f"pilot_recall_class_{class_id}" for class_id in range(Config.NUM_CLASSES)],
    ]
    model_comparison_table = completed_pilot_results[comparison_columns].copy()
    model_comparison_path = Path(Config.REPORTS_DIR) / "model_comparison_table.csv"
    model_comparison_table.to_csv(model_comparison_path, index=False)
    print("[Pilot] Final model comparison table (full validation split):")
    display(model_comparison_table.round(4))
    print(f"[Pilot] Saved -> {model_comparison_path}")
    print(f"[Pilot] Selected for the final fit: {selected_tuning_trial['trial_id']}")

    baseline_rows = completed_pilot_results.loc[
        completed_pilot_results["trial_id"] == TUNING_TRIALS[0]["trial_id"]
    ]
    if baseline_rows.empty:
        raise RuntimeError("The baseline pilot trial did not complete, so differences from it cannot be computed.")
    baseline_row = baseline_rows.iloc[0]
    delta_metrics = [
        "pilot_validation_accuracy",
        "pilot_validation_qwk",
        "pilot_validation_macro_f1",
        *[f"pilot_recall_class_{class_id}" for class_id in range(Config.NUM_CLASSES)],
    ]
    comparison_rows = completed_pilot_results[
        completed_pilot_results["group"] != "hyperparameter"
    ].copy()
    for metric in delta_metrics:
        comparison_rows[f"delta_{metric.replace('pilot_', '')}"] = (
            comparison_rows[metric] - float(baseline_row[metric])
        )
    comparison_path = Path(Config.REPORTS_DIR) / "comparison_trials_results.csv"
    comparison_rows.to_csv(comparison_path, index=False)

    if not comparison_rows.empty:
        print("\n[Comparison] Difference from the EfficientNetB3 + Ben Graham baseline "
              "(positive = better than baseline):")
        display(
            comparison_rows[
                ["trial_id", "group", "change_vs_baseline",
                 "delta_validation_qwk", "delta_validation_macro_f1", "delta_validation_accuracy",
                 "delta_recall_class_1", "delta_recall_class_3"]
            ].round(4)
        )
        print(f"The validation split has only {len(val_df):,} images "
              f"({int((val_df['label'] == 3).sum())} Severe), so differences of a few hundredths "
              "can be noise; discuss them as indicative, not conclusive.")
    print(f"[Comparison] Saved -> {comparison_path}")
else:
    print("[Pilot] Skipped because hyperparameter tuning is disabled.")


In [ ]:
# ============================================================
# SECTION 6.2d — PILOT COMPARISON GRAPHS
# (a) Validation QWK and macro-F1 per trial, grouped by trial group.
# (b) Per-stage validation recall heatmap.
# These are pilot runs (up to 4 + 6 epochs), not final model results.
# ============================================================

if RUN_HYPERPARAMETER_TUNING:
    group_order = ["hyperparameter", "balancing", "preprocessing_ablation",
                   "augmentation_ablation", "backbone", "scratch_baseline"]
    group_colors = {
        "hyperparameter": "#367C8A",
        "balancing": "#7A8F55",
        "preprocessing_ablation": "#D8784B",
        "augmentation_ablation": "#B5655E",
        "backbone": "#6E6A9E",
        "scratch_baseline": "#8C8C8C",
    }
    # Only completed trials are plotted; skipped rows have no metrics.
    pilot_plot_df = hyperparameter_tuning_results[hyperparameter_tuning_results["status"] == "completed"].copy()
    pilot_plot_df["group"] = pd.Categorical(pilot_plot_df["group"], categories=group_order, ordered=True)
    pilot_plot_df = pilot_plot_df.sort_values(["group", "pilot_validation_qwk"], ascending=[True, False])
    trial_labels = pilot_plot_df["trial_id"].astype(str).tolist()
    positions = np.arange(len(trial_labels))
    bar_colors = [group_colors.get(str(group), "#888888") for group in pilot_plot_df["group"]]
    baseline_values = pilot_plot_df.set_index("trial_id").loc[TUNING_TRIALS[0]["trial_id"]]

    fig, axis = plt.subplots(figsize=(max(12, 0.75 * len(trial_labels) + 4), 6))
    bar_width = 0.4
    axis.bar(positions - bar_width / 2, pilot_plot_df["pilot_validation_qwk"], bar_width,
             color=bar_colors, edgecolor="black", linewidth=0.5, label="Validation QWK")
    axis.bar(positions + bar_width / 2, pilot_plot_df["pilot_validation_macro_f1"], bar_width,
             color=bar_colors, edgecolor="black", linewidth=0.5, hatch="//", alpha=0.75, label="Validation macro-F1")
    axis.axhline(float(baseline_values["pilot_validation_qwk"]), color="black", linestyle="--",
                 linewidth=1, label="Baseline QWK")
    group_edges = pilot_plot_df["group"].astype(str).ne(pilot_plot_df["group"].astype(str).shift()).to_numpy()
    for edge in np.flatnonzero(group_edges)[1:]:
        axis.axvline(edge - 0.5, color="grey", linewidth=0.8, alpha=0.6)
    axis.set_xticks(positions, trial_labels, rotation=60, ha="right", fontsize=8)
    axis.set_ylabel("Score (validation split)")
    axis.set_title("Pilot trials — validation QWK (solid) and macro-F1 (hatched), grouped by trial group")
    handles = [plt.Rectangle((0, 0), 1, 1, color=group_colors[group]) for group in group_order
               if group in set(pilot_plot_df["group"].astype(str))]
    labels = [group for group in group_order if group in set(pilot_plot_df["group"].astype(str))]
    legend_groups = axis.legend(handles, labels, title="Trial group", loc="upper right", fontsize=8)
    axis.add_artist(legend_groups)
    axis.legend(loc="upper center", fontsize=8)
    axis.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    pilot_comparison_plot_path = Path(Config.REPORTS_DIR) / "pilot_comparison.png"
    fig.savefig(pilot_comparison_plot_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"[Pilot Comparison] Saved QWK / macro-F1 chart -> {pilot_comparison_plot_path}")

    recall_columns = [f"pilot_recall_class_{class_id}" for class_id in range(Config.NUM_CLASSES)]
    recall_matrix = pilot_plot_df.set_index("trial_id")[recall_columns].copy()
    recall_matrix.columns = Config.CLASS_NAMES
    fig, axis = plt.subplots(figsize=(9, max(5, 0.45 * len(recall_matrix) + 1.5)))
    sns.heatmap(recall_matrix, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="YlGnBu",
                ax=axis, cbar_kws={"label": "Recall"})
    axis.set_title("Per-stage validation recall by pilot trial")
    axis.set_xlabel("Dataset grade")
    axis.set_ylabel("")
    fig.tight_layout()
    recall_plot_path = Path(Config.REPORTS_DIR) / "pilot_recall_heatmap.png"
    fig.savefig(recall_plot_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"[Pilot Comparison] Saved recall heatmap -> {recall_plot_path}")
else:
    print("[Pilot Comparison] Skipped because hyperparameter tuning is disabled.")


In [ ]:
# ============================================================
# SECTION 6.3 — FINAL PHASE 1 TRAINING (FROZEN BACKBONE)
# After the optional pilot selects settings, train the head on the
# complete training split using the selected configuration.
# ============================================================

print("=" * 60)
print("  FINAL PHASE 1 — Frozen Backbone (Head Only)")
print(f"  Image size   : {Config.IMG_SIZE}")
print(f"  Dropout      : {Config.DROPOUT_RATE}")
print(f"  Max epochs   : {Config.PHASE1_EPOCHS}")
print(f"  Learning rate: {Config.PHASE1_LR}")
print(f"  Batch size   : {Config.BATCH_SIZE}")
print("=" * 60)

callbacks_phase1 = build_callbacks("phase1", lr_schedule="plateau",
                                   total_epochs=Config.PHASE1_EPOCHS, peak_lr=Config.PHASE1_LR)
_phase1_fit_start = time.perf_counter()
history_phase1 = model.fit(
    train_dataset,
    epochs=Config.PHASE1_EPOCHS,
    validation_data=val_dataset,
    callbacks=callbacks_phase1,
    steps_per_epoch=steps_per_epoch if Config.USE_OVERSAMPLING else None,
    class_weight=active_class_weights(),
    verbose=1,
)

FINAL_FIT_SECONDS = {"phase1": round(time.perf_counter() - _phase1_fit_start, 1)}

log_experiment(
    phase="Phase 1 - Final Full-Data Fit",
    lr=Config.PHASE1_LR,
    batch_size=Config.BATCH_SIZE,
    epochs_planned=Config.PHASE1_EPOCHS,
    history=history_phase1,
    notes=(
        f"Selected pilot trial={selected_tuning_trial['trial_id'] if RUN_HYPERPARAMETER_TUNING else 'single_config'}; "
        f"image_size={Config.IMG_SIZE}; dropout={Config.DROPOUT_RATE}; "
        f"class weights enabled={Config.USE_CLASS_WEIGHTS}; class weight mode={Config.CLASS_WEIGHT_MODE}; "
        f"oversampling={Config.USE_OVERSAMPLING}; batch={Config.BATCH_SIZE}"
    ),
)

pd.DataFrame(history_phase1.history).to_csv(
    os.path.join(Config.REPORTS_DIR, "phase1_training_history.csv"), index=False
)
print("[Phase 1] Final full-data training complete.")

In [ ]:
# ============================================================
# SECTION 6.4 — PHASE 1 LEARNING CURVES
# Plot and save training vs validation accuracy and loss curves, plus
# validation QWK. Each accuracy/loss panel shows three lines:
#   Train (augmented)    -> Keras training metric on augmented batches,
#                           dropout on, class-weighted loss
#   Train (clean subset) -> 500 fixed training images, no augmentation,
#                           inference mode, unweighted loss (like-for-like)
#   Validation           -> validation split, inference mode, unweighted
# These give an early visual indicator of overfitting (validation
# diverging from the clean training curve) or underfitting (all
# curves plateauing at a poor value) before fine-tuning begins.
# ============================================================

def plot_training_curves(
    history: keras.callbacks.History,
    phase_name: str,
    save_dir: str = Config.REPORTS_DIR,
) -> None:
    """
    Plot and save training vs validation accuracy, loss and validation QWK.

    Three subplots side by side:
    - Left: accuracy — train (augmented), train (clean subset), validation
    - Middle: loss — train (augmented), train (clean subset), validation
    - Right: val_qwk (logged by the ValidationQWK callback)

    The clean-subset line is the like-for-like comparison with validation:
    a widening gap between it and validation indicates overfitting; all
    curves plateauing at a poor value indicates underfitting. A vertical
    line marks the epoch whose weights early stopping restores (best
    Config.STOP_MONITOR). Curves are plotted exactly as logged (no smoothing).

    Args:
        history:    Keras History object from model.fit().
        phase_name: Label for the figure title and filename.
        save_dir:   Directory to save the PNG.
    """
    acc     = history.history.get("accuracy",     history.history.get("categorical_accuracy", []))
    val_acc = history.history.get("val_accuracy", history.history.get("val_categorical_accuracy", []))
    loss     = history.history["loss"]
    val_loss = history.history["val_loss"]
    clean_acc = history.history.get("clean_train_accuracy", [])
    clean_loss = history.history.get("clean_train_loss", [])
    epochs   = range(1, len(loss) + 1)

    val_qwk = history.history.get("val_qwk", [])
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(19, 5.5))
    fig.suptitle(f"Training Curves — {phase_name}", fontsize=13, fontweight="bold")

    # ---- Accuracy subplot ----
    ax1.plot(epochs, acc,     "b-o", markersize=4, label="Train (augmented)")
    if clean_acc:
        ax1.plot(epochs, clean_acc, "c--s", markersize=4, label="Train (clean subset)")
    ax1.plot(epochs, val_acc, "r-o", markersize=4, label="Validation")
    ax1.set_title("Accuracy")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy")
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.3)
    ax1.set_ylim(0, 1)

    # ---- Loss subplot ----
    ax2.plot(epochs, loss,     "b-o", markersize=4, label="Train (augmented)")
    if clean_loss:
        ax2.plot(epochs, clean_loss, "c--s", markersize=4, label="Train (clean subset)")
    ax2.plot(epochs, val_loss, "r-o", markersize=4, label="Validation")
    ax2.set_title("Loss (label-smoothed cross-entropy)")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Loss")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)

    # ---- Validation QWK subplot (drives final-fit stopping and checkpointing) ----
    if val_qwk:
        ax3.plot(epochs, val_qwk, "g-o", markersize=4, label="Validation QWK")
    ax3.set_title("Validation Quadratic Weighted Kappa")
    ax3.set_xlabel("Epoch")
    ax3.set_ylabel("QWK")
    ax3.legend(fontsize=8)
    ax3.grid(alpha=0.3)

    # Mark the epoch whose weights early stopping restores (best Config.STOP_MONITOR)
    best_epoch = restored_epoch_index(history) + 1
    for ax in (ax1, ax2, ax3):
        ax.axvline(
            x=best_epoch,
            color="green",
            linestyle="--",
            linewidth=1.2,
            alpha=0.7,
            label=f"Restored epoch ({best_epoch}, best {Config.STOP_MONITOR})",
        )
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    fig.text(0.5, -0.02, CURVE_LEGEND_NOTE, ha="center", fontsize=9, color="#475569")
    plt.tight_layout()
    save_path = os.path.join(save_dir, f"curves_{phase_name.replace(' ', '_')}.png")
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"[Curves] Saved -> {save_path}")
    plt.show()


plot_training_curves(history_phase1, "Phase_1_Frozen_Base")


In [ ]:
# ============================================================
# SECTION 6.5 — FINAL PHASE 2 TRAINING (FINE-TUNING)
# Fine-tune the selected configuration on the complete
# training split; these are the weights used by final evaluation.
# ============================================================

print("=" * 60)
print("  FINAL PHASE 2 — Fine-Tuning (BatchNorm Frozen)")
print(f"  Selected pilot trial: {selected_tuning_trial['trial_id'] if RUN_HYPERPARAMETER_TUNING else 'single_config'}")
print(f"  Image size   : {Config.IMG_SIZE}")
print(f"  Dropout      : {Config.DROPOUT_RATE}")
print(f"  Unfreeze top : {Config.UNFREEZE_TOP_N} layers")
print(f"  Max epochs   : {Config.PHASE2_EPOCHS}")
print(f"  Learning rate: {Config.PHASE2_LR} ({Config.PHASE2_LR_SCHEDULE} schedule)")
print(f"  Weight decay : {Config.WEIGHT_DECAY}")
print(f"  Batch size   : {Config.BATCH_SIZE}")
print("=" * 60)

model.load_weights(os.path.join(Config.CHECKPOINT_DIR, "best_phase1.weights.h5"))
configure_phase2(model, base_model)
callbacks_phase2 = build_callbacks("phase2", lr_schedule=Config.PHASE2_LR_SCHEDULE,
                                   total_epochs=Config.PHASE2_EPOCHS, peak_lr=Config.PHASE2_LR)
_phase2_fit_start = time.perf_counter()
history_phase2 = model.fit(
    train_dataset,
    epochs=Config.PHASE2_EPOCHS,
    validation_data=val_dataset,
    callbacks=callbacks_phase2,
    steps_per_epoch=steps_per_epoch if Config.USE_OVERSAMPLING else None,
    class_weight=active_class_weights(),
    verbose=1,
)

FINAL_FIT_SECONDS["phase2"] = round(time.perf_counter() - _phase2_fit_start, 1)
print(f"[Timing] Final fit: Phase 1 {FINAL_FIT_SECONDS['phase1']:.0f}s, Phase 2 {FINAL_FIT_SECONDS['phase2']:.0f}s")

log_experiment(
    phase="Phase 2 - Final Full-Data Fine-Tuning",
    lr=Config.PHASE2_LR,
    batch_size=Config.BATCH_SIZE,
    epochs_planned=Config.PHASE2_EPOCHS,
    history=history_phase2,
    notes=(
        f"Top {Config.UNFREEZE_TOP_N} layers trainable except BatchNorm; "
        f"dropout={Config.DROPOUT_RATE}; class weights={Config.USE_CLASS_WEIGHTS}; "
        f"oversampling={Config.USE_OVERSAMPLING}; class weight mode={Config.CLASS_WEIGHT_MODE}; "
        f"lr schedule={Config.PHASE2_LR_SCHEDULE}; weight decay={Config.WEIGHT_DECAY}; batch={Config.BATCH_SIZE}; "
        f"image_size={Config.IMG_SIZE}; "
        f"selected pilot trial={selected_tuning_trial['trial_id'] if RUN_HYPERPARAMETER_TUNING else 'single_config'}"
    ),
)

pd.DataFrame(history_phase2.history).to_csv(
    os.path.join(Config.REPORTS_DIR, "phase2_training_history.csv"), index=False
)
plot_training_curves(history_phase2, "Phase_2_Final_Full_Data_Fine_Tuning")
experiment_log.to_csv(os.path.join(Config.REPORTS_DIR, "experiment_log.csv"), index=False)
print("[Phase 2] Final full-data training complete.")
model.load_weights(os.path.join(Config.CHECKPOINT_DIR, "best_phase2.weights.h5"))

In [ ]:
# ============================================================
# SECTION 6.5b — POST-TRAINING OVERFITTING DIAGNOSTICS
# Summarize observed train/validation gaps and validation-loss movement.
# These are diagnostic signals, not a clinical or statistical proof.
# ============================================================


def _summarize_overfitting_signals(history, phase_name: str) -> Dict[str, Any]:
    accuracy = history.history.get("accuracy", history.history.get("categorical_accuracy"))
    validation_accuracy = history.history.get(
        "val_accuracy", history.history.get("val_categorical_accuracy")
    )
    training_loss = history.history.get("loss")
    validation_loss = history.history.get("val_loss")
    if any(values is None for values in (accuracy, validation_accuracy, training_loss, validation_loss)):
        raise ValueError(f"Training history is missing required accuracy/loss metrics for {phase_name}.")

    best_accuracy_index = int(np.nanargmax(validation_accuracy))
    best_loss_index = int(np.nanargmin(validation_loss))
    restored_index = restored_epoch_index(history)  # best Config.STOP_MONITOR; weights early stopping restores
    accuracy_gap = float(accuracy[best_loss_index] - validation_accuracy[best_loss_index])
    clean_accuracy = history.history.get("clean_train_accuracy")
    clean_gap = (
        float(clean_accuracy[best_loss_index] - validation_accuracy[best_loss_index])
        if clean_accuracy else float("nan")
    )
    late_validation_loss_rise = float(validation_loss[-1] - validation_loss[best_loss_index])
    if late_validation_loss_rise > 1e-6:
        loss_observation = (
            f"Validation loss rose by {late_validation_loss_rise:.4f} after its minimum; "
            "inspect the curves and accuracy gap for a possible overfitting signal."
        )
    else:
        loss_observation = (
            "No late validation-loss rise was observed; this alone does not rule out overfitting."
        )

    return {
        "phase": phase_name,
        "epochs_run": len(validation_loss),
        "best_val_accuracy_epoch": best_accuracy_index + 1,
        "train_accuracy_at_best_val_accuracy": float(accuracy[best_accuracy_index]),
        "best_validation_accuracy": float(validation_accuracy[best_accuracy_index]),
        "train_minus_validation_accuracy_gap_at_best_val_loss": accuracy_gap,
        "clean_train_minus_validation_accuracy_gap": clean_gap,
        "best_val_qwk": float(np.nanmax(history.history["val_qwk"])) if history.history.get("val_qwk") else float("nan"),
        "best_val_loss_epoch": best_loss_index + 1,
        "restored_epoch": restored_index + 1,
        "validation_accuracy_at_restored_epoch": float(validation_accuracy[restored_index]),
        "clean_train_minus_validation_accuracy_gap_at_restored_epoch": (
            float(clean_accuracy[restored_index] - validation_accuracy[restored_index])
            if clean_accuracy else float("nan")
        ),
        "minimum_validation_loss": float(validation_loss[best_loss_index]),
        "final_validation_loss": float(validation_loss[-1]),
        "final_minus_min_validation_loss": late_validation_loss_rise,
        "curve_observation": loss_observation,
        "overfitting_controls": (
            f"dropout={Config.DROPOUT_RATE}; training-only augmentation; "
            f"{Config.STOP_MONITOR} early stopping (mode {Config.STOP_MODE}, patience {Config.STOP_PATIENCE}); "
            f"top {Config.UNFREEZE_TOP_N} layers fine-tuned; "
            + (f"AdamW weight_decay={Config.WEIGHT_DECAY}" if phase_name == "Phase 2" else "Phase 1 backbone frozen")
        ),
    }


overfitting_diagnostics = pd.DataFrame(
    [
        _summarize_overfitting_signals(history_phase1, "Phase 1"),
        _summarize_overfitting_signals(history_phase2, "Phase 2"),
    ]
)
overfitting_diagnostics_path = Path(Config.REPORTS_DIR) / "overfitting_diagnostics.csv"
overfitting_diagnostics.to_csv(overfitting_diagnostics_path, index=False)
print("Observed train/validation diagnostics (not proof of overfitting):")
display(overfitting_diagnostics)
print("Saved curve interpretation inputs to:", overfitting_diagnostics_path)

In [ ]:
# ============================================================
# SECTION 6.5c - LEARNING-RATE CURVE
# Plots the learning rate used at each epoch of the final Phase 1 and
# Phase 2 runs. Phase 1 uses ReduceLROnPlateau; Phase 2 uses either
# ReduceLROnPlateau or warm-up + cosine (Config.PHASE2_LR_SCHEDULE).
# Read from the saved history CSVs so it also works after a restart.
# ============================================================

def _load_lr_history(csv_name: str, history=None) -> pd.Series:
    frame = pd.read_csv(os.path.join(Config.REPORTS_DIR, csv_name))
    for key in ("learning_rate", "lr"):
        if key in frame.columns:
            return frame[key].astype(float)
    if history is not None:
        for key in ("learning_rate", "lr"):
            if key in history.history:
                return pd.Series(history.history[key], dtype=float)
    raise KeyError(f"No learning-rate column found in {csv_name}; rerun training with build_callbacks().")


_lr_phase1 = _load_lr_history("phase1_training_history.csv", globals().get("history_phase1"))
_lr_phase2 = _load_lr_history("phase2_training_history.csv", globals().get("history_phase2"))
learning_rate_history = pd.DataFrame({
    "phase": ["Phase 1"] * len(_lr_phase1) + ["Phase 2"] * len(_lr_phase2),
    "phase_epoch": list(range(1, len(_lr_phase1) + 1)) + list(range(1, len(_lr_phase2) + 1)),
    "overall_epoch": list(range(1, len(_lr_phase1) + len(_lr_phase2) + 1)),
    "learning_rate": pd.concat([_lr_phase1, _lr_phase2], ignore_index=True),
})
learning_rate_history.to_csv(os.path.join(Config.REPORTS_DIR, "learning_rate_history.csv"), index=False)

fig, ax = plt.subplots(figsize=(10, 4.5))
for _phase, _color in (("Phase 1", "#367C8A"), ("Phase 2", "#D8784B")):
    _rows = learning_rate_history[learning_rate_history["phase"] == _phase]
    ax.step(_rows["overall_epoch"], _rows["learning_rate"], where="mid",
            color=_color, marker="o", markersize=4, label=_phase)
ax.axvline(len(_lr_phase1) + 0.5, color="grey", linestyle="--", linewidth=1, alpha=0.7)
ax.set_yscale("log")
ax.set_title(f"Learning Rate per Epoch - Final Training Run (Phase 2: {Config.PHASE2_LR_SCHEDULE})")
ax.set_xlabel("Epoch (Phase 1 then Phase 2)")
ax.set_ylabel("Learning rate (log scale)")
ax.legend()
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
lr_curve_path = os.path.join(Config.REPORTS_DIR, "lr_curve_final.png")
fig.savefig(lr_curve_path, dpi=150, bbox_inches="tight")
plt.show()

for _phase, _rows in learning_rate_history.groupby("phase"):
    _reductions = int((_rows["learning_rate"].diff() < 0).sum())
    _schedule = "plateau" if _phase == "Phase 1" else Config.PHASE2_LR_SCHEDULE
    print(
        f"[LR Curve] {_phase} ({_schedule}): start {_rows['learning_rate'].iloc[0]:.2e} -> "
        f"peak {_rows['learning_rate'].max():.2e} -> end {_rows['learning_rate'].iloc[-1]:.2e} "
        f"({_reductions} epoch-to-epoch decreases)"
    )
print(f"[LR Curve] Saved -> {lr_curve_path}")


In [ ]:
# ============================================================
# SECTION 6.6 — FINAL RUN CONFIGURATION AUDIT
# Confirms the full-data final fit used the selected pilot settings.
# ============================================================

phase2_rows = experiment_log[
    experiment_log["phase"].str.contains("Phase 2", case=False, na=False)
].copy()
if phase2_rows.empty:
    raise RuntimeError("No completed Phase 2 record is available.")

final_phase2_record = phase2_rows.iloc[-1]
recorded_notes = str(final_phase2_record["notes"])
phase2_config_matches = all([
    np.isclose(float(final_phase2_record["learning_rate"]), Config.PHASE2_LR),
    int(final_phase2_record["batch_size"]) == Config.BATCH_SIZE,
    f"Top {Config.UNFREEZE_TOP_N} layers trainable" in recorded_notes,
    f"dropout={Config.DROPOUT_RATE}" in recorded_notes,
    f"class weights={Config.USE_CLASS_WEIGHTS}" in recorded_notes,
    f"image_size={Config.IMG_SIZE}" in recorded_notes,
])
if RUN_HYPERPARAMETER_TUNING:
    phase2_config_matches = phase2_config_matches and (
        f"selected pilot trial={selected_tuning_trial['trial_id']}" in recorded_notes
    )

best_val_accuracy_epoch = int(np.argmax(history_phase2.history["val_accuracy"])) + 1
best_val_loss_epoch = int(np.argmin(history_phase2.history["val_loss"])) + 1
restored_phase2_epoch = restored_epoch_index(history_phase2) + 1
final_run_config = {
    "image_size": Config.IMG_SIZE,
    "batch_size": Config.BATCH_SIZE,
    "phase1_learning_rate": Config.PHASE1_LR,
    "phase2_learning_rate": Config.PHASE2_LR,
    "phase2_unfreeze_top_n": Config.UNFREEZE_TOP_N,
    "dropout_rate": Config.DROPOUT_RATE,
    "class_weights": Config.USE_CLASS_WEIGHTS,
    "oversampling": Config.USE_OVERSAMPLING,
    "class_weight_mode": Config.CLASS_WEIGHT_MODE,
    "phase2_lr_schedule": Config.PHASE2_LR_SCHEDULE,
    "weight_decay": Config.WEIGHT_DECAY,
    "early_stopping_monitor": Config.STOP_MONITOR,
    "early_stopping_mode": Config.STOP_MODE,
    "early_stopping_patience": Config.STOP_PATIENCE,
    "selected_pilot_trial": selected_tuning_trial["trial_id"] if RUN_HYPERPARAMETER_TUNING else None,
}

print("Final full-data model configuration:")
for key, value in final_run_config.items():
    print(f"  {key}: {value}")
print(f"Best final Phase 2 validation-accuracy epoch: {best_val_accuracy_epoch}")
print(f"Best final Phase 2 validation-loss epoch    : {best_val_loss_epoch}")
print(f"Restored final Phase 2 epoch (best {Config.STOP_MONITOR}): {restored_phase2_epoch}")
print(f"Final Phase 2 record matches selected settings: {phase2_config_matches}")
print("Final weights loaded from:", os.path.join(Config.CHECKPOINT_DIR, "best_phase2.weights.h5"))

## Section 7: Evaluation

**Goal:** Conduct a rigorous clinical and statistical evaluation on the held-out test set
(15% of the total dataset, never seen during training or validation).

> **Headline numbers.** Section 7.2 reports both decision rules on the same test predictions: **"Argmax (plain)"** is the headline **accuracy**, and **"Validation-QWK thresholds"** (cut points chosen on the validation split) is the headline **QWK**. Both rows, with macro-F1, are saved to `test_decision_rule_summary.csv`; thresholding can raise QWK while lowering accuracy, so both are always reported.

> **Small test set.** The APTOS 2019 test split holds 550 images, with only 29 Severe cases (per-class counts are printed in Section 4.3). Treat per-class test metrics — especially Severe recall — as having wide uncertainty; a handful of images changes them by several percentage points.

### Clinical Evaluation Metric: Why Quadratic Weighted Kappa (QWK)?

Standard **classification accuracy** treats all errors identically:
- Misclassifying **No DR (Class 0)** as **Mild NPDR (Class 1)** is penalized by 1 error.
- Misclassifying **No DR (Class 0)** as **Proliferative DR (Class 4)** is also penalized by 1 error.

Clinically, this is fundamentally flawed. Diabetic Retinopathy staging is an **ordinal scale**
(ordered severity levels from 0 to 4):
1. Confusing adjacent stages (e.g., Mild vs. Moderate) has minor clinical consequence because both
   require clinical follow-up and monitoring.
2. Confusing distant stages (e.g., predicting "No DR" when the patient has sight-threatening
   "Severe" or "Proliferative DR") is catastrophic, delaying emergency laser photocoagulation or
   anti-VEGF injections and risking irreversible vision loss.

**Quadratic Weighted Kappa (QWK)** measures inter-rater agreement between the ground truth and
model predictions on an ordinal scale, penalizing misclassifications quadratically according to
the distance $|i - j|^2$:
$$\kappa = 1 - \frac{\sum_{i,j} w_{ij} O_{ij}}{\sum_{i,j} w_{ij} E_{ij}}, \quad \text{where } w_{ij} = \frac{(i - j)^2}{(N - 1)^2}$$
- An error between Class 0 and Class 1 carries a penalty weight of $(0-1)^2 = 1$.
- An error between Class 0 and Class 4 carries a penalty weight of $(0-4)^2 = 16$.

This mirrors the official metric used in the landmark Kaggle Diabetic Retinopathy and APTOS 2019
competitions, providing the gold-standard benchmark for automated retinal image grading.

In [ ]:
def predict_probs_tta(eval_model, images):
    probabilities = eval_model(images, training=False)
    probabilities += eval_model(tf.image.flip_left_right(images), training=False)
    probabilities += eval_model(tf.image.flip_up_down(images), training=False)
    probabilities += eval_model(
        tf.image.flip_left_right(tf.image.flip_up_down(images)), training=False
    )
    return (probabilities / 4.0).numpy()


def _val_metrics(use_tta):  # CHANGED: compare both accuracy and ordinal QWK
    y_true_batches = []  # CHANGED: collect validation truth alongside predictions
    y_pred_batches = []  # CHANGED: collect validation predictions
    for images, labels in val_dataset:
        if use_tta:
            probabilities = predict_probs_tta(model, images)
        else:
            probabilities = model(images, training=False).numpy()
        y_true_batches.append(np.argmax(labels.numpy(), axis=1))  # CHANGED: use this batch's labels
        y_pred_batches.append(np.argmax(probabilities, axis=1))  # CHANGED: collect predictions
    y_true = np.concatenate(y_true_batches)  # CHANGED: aggregate validation labels
    y_pred = np.concatenate(y_pred_batches)  # CHANGED: aggregate validation predictions
    accuracy = float(np.mean(y_true == y_pred))  # CHANGED: validation accuracy
    qwk = float(cohen_kappa_score(y_true, y_pred, weights="quadratic"))  # CHANGED: validation QWK
    return accuracy, qwk  # CHANGED: return both selection metrics


plain_val_accuracy, plain_val_qwk = _val_metrics(False)  # CHANGED: score plain validation inference
tta_val_accuracy, tta_val_qwk = _val_metrics(True)  # CHANGED: score four-view TTA on validation only
USE_TTA_FOR_EVALUATION = tta_val_qwk > plain_val_qwk  # CHANGED: select TTA only when validation QWK improves
print(
    f"Validation plain — accuracy: {plain_val_accuracy:.4f}, QWK: {plain_val_qwk:.4f} | "
    f"TTA — accuracy: {tta_val_accuracy:.4f}, QWK: {tta_val_qwk:.4f} | "
    f"Selected: {'TTA' if USE_TTA_FOR_EVALUATION else 'plain'}"
)  # CHANGED: report both metrics and validation-only selection

In [ ]:
# ============================================================
# SECTION 7.0b — VALIDATION-ONLY QWK THRESHOLD CALIBRATION
# Thresholds are selected from validation predictions, never test labels.
# ============================================================

from itertools import combinations


def apply_qwk_thresholds(probabilities: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    """Map expected ordinal grades to classes using increasing cut points."""
    probabilities = np.asarray(probabilities, dtype=np.float64)
    thresholds = np.asarray(thresholds, dtype=np.float64)
    if probabilities.ndim != 2 or probabilities.shape[1] != Config.NUM_CLASSES:
        raise ValueError("probabilities must have shape (n_samples, NUM_CLASSES).")
    if thresholds.size != Config.NUM_CLASSES - 1 or np.any(np.diff(thresholds) <= 0):
        raise ValueError("Provide NUM_CLASSES - 1 strictly increasing thresholds.")
    expected_grade = probabilities @ np.arange(Config.NUM_CLASSES, dtype=np.float64)
    return np.digitize(expected_grade, thresholds).astype(int)


def tune_qwk_thresholds(
    y_true: np.ndarray,
    probabilities: np.ndarray,
) -> tuple[np.ndarray, float]:
    """Choose ordinal cut points by maximizing QWK over validation predictions."""
    probabilities = np.asarray(probabilities, dtype=np.float64)
    y_true = np.asarray(y_true, dtype=int)
    expected_grade = probabilities @ np.arange(Config.NUM_CLASSES, dtype=np.float64)
    quantile_candidates = np.quantile(expected_grade, np.linspace(0.05, 0.95, 10))
    default_candidates = np.arange(0.5, Config.NUM_CLASSES - 0.5, 1.0)
    candidates = np.unique(np.concatenate([quantile_candidates, default_candidates]))

    best_thresholds = None
    best_qwk = -np.inf
    for trial_thresholds in combinations(candidates, Config.NUM_CLASSES - 1):
        predictions = apply_qwk_thresholds(probabilities, np.asarray(trial_thresholds))
        score = cohen_kappa_score(y_true, predictions, weights="quadratic")
        if np.isfinite(score) and score > best_qwk:
            best_thresholds = np.asarray(trial_thresholds, dtype=np.float64)
            best_qwk = float(score)

    if best_thresholds is None:
        raise RuntimeError("Could not find valid QWK thresholds from the validation predictions.")
    return best_thresholds, best_qwk


val_probability_batches = []
for images, _ in val_dataset:
    if USE_TTA_FOR_EVALUATION:
        val_probabilities = predict_probs_tta(model, images)
    else:
        val_probabilities = model(images, training=False).numpy()
    val_probability_batches.append(val_probabilities)

val_probabilities = np.concatenate(val_probability_batches, axis=0)
val_true = val_df["label"].to_numpy(dtype=int)
if len(val_true) != len(val_probabilities):
    raise ValueError("Validation labels and predictions have different lengths.")

QWK_THRESHOLDS, calibrated_val_qwk = tune_qwk_thresholds(val_true, val_probabilities)
argmax_val_predictions = np.argmax(val_probabilities, axis=1)
calibrated_val_predictions = apply_qwk_thresholds(val_probabilities, QWK_THRESHOLDS)
print(f"Validation argmax QWK      : {cohen_kappa_score(val_true, argmax_val_predictions, weights='quadratic'):.4f}")
print(f"Validation calibrated QWK  : {calibrated_val_qwk:.4f}")
print(f"Validation argmax accuracy : {np.mean(val_true == argmax_val_predictions):.4f}")
print(f"Validation calibrated acc. : {np.mean(val_true == calibrated_val_predictions):.4f}")
print("Validation-selected QWK thresholds:", np.round(QWK_THRESHOLDS, 4).tolist())
print("Freeze these thresholds before evaluating the held-out test set.")

In [ ]:
# ============================================================
# SECTION 7.0c — CLASS-WEIGHT VALIDATION EVIDENCE
# For each arm, restart the kernel and rerun training through this cell:
# once with USE_CLASS_WEIGHTS=False, then once with it True.
# Keep the seed, manifests, and other settings fixed. Stop before
# Section 7.1 and evaluate the held-out test set only once after selection.
# ============================================================

import hashlib

_expected_weight_note = f"class weights enabled={Config.USE_CLASS_WEIGHTS}"
_phase1_records = experiment_log[
    experiment_log["phase"].str.contains("Phase 1", case=False, na=False)
]
_phase2_records = experiment_log[
    experiment_log["phase"].str.contains("Phase 2", case=False, na=False)
]
if _phase1_records.empty or _phase2_records.empty:
    raise RuntimeError("Complete both classifier training phases before recording evidence.")
if _expected_weight_note not in str(_phase1_records.iloc[-1]["notes"]):
    raise RuntimeError("Phase 1 was not trained with the currently configured class-weight setting.")
if f"class weights={Config.USE_CLASS_WEIGHTS}" not in str(_phase2_records.iloc[-1]["notes"]):
    raise RuntimeError("Phase 2 was not trained with the currently configured class-weight setting.")

_validation_true = val_df["label"].to_numpy(dtype=int)
_validation_probability_batches = []
for _images, _ in val_dataset:
    _validation_probability_batches.append(model(_images, training=False).numpy())
_validation_probabilities = np.concatenate(_validation_probability_batches, axis=0)
_validation_pred = np.argmax(_validation_probabilities, axis=1)
if len(_validation_true) != len(_validation_pred):
    raise ValueError("Validation labels and predictions have different lengths.")

_validation_report = classification_report(
    _validation_true,
    _validation_pred,
    labels=list(range(Config.NUM_CLASSES)),
    target_names=Config.CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
_validation_accuracy = float(np.mean(_validation_true == _validation_pred))
_validation_qwk = float(
    cohen_kappa_score(_validation_true, _validation_pred, weights="quadratic")
)

_manifest_rows = pd.concat(
    [
        frame[["patient_id", "duplicate_group", "label"]].assign(split=split_name)
        for split_name, frame in (
            ("train", train_df),
            ("validation", val_df),
            ("test", test_df),
        )
    ],
    ignore_index=True,
)
_manifest_rows = _manifest_rows.astype("string").fillna("").sort_values(
    ["split", "patient_id", "duplicate_group", "label"]
)
_split_fingerprint = hashlib.sha256(
    _manifest_rows.to_csv(index=False, lineterminator="\n").encode("utf-8")
).hexdigest()

_validation_records = []
for _class_id, _class_name in enumerate(Config.CLASS_NAMES):
    _class_metrics = _validation_report[_class_name]
    _validation_records.append(
        {
            "run_id": _RUN_ID,
            "split_fingerprint": _split_fingerprint,
            "class_weights_enabled": bool(Config.USE_CLASS_WEIGHTS),
            "seed": Config.SEED,
            "image_size": Config.IMG_SIZE,
            "batch_size": Config.BATCH_SIZE,
            "phase1_learning_rate": Config.PHASE1_LR,
            "phase2_learning_rate": Config.PHASE2_LR,
            "phase1_epochs": Config.PHASE1_EPOCHS,
            "phase2_epochs": Config.PHASE2_EPOCHS,
            "dropout_rate": Config.DROPOUT_RATE,
            "phase2_unfreeze_top_n": Config.UNFREEZE_TOP_N,
            "class_id": _class_id,
            "class_name": _class_name,
            "validation_accuracy": _validation_accuracy,
            "validation_qwk": _validation_qwk,
            "precision": float(_class_metrics["precision"]),
            "recall": float(_class_metrics["recall"]),
            "f1": float(_class_metrics["f1-score"]),
            "support": int(_class_metrics["support"]),
        }
    )

_validation_evidence = pd.DataFrame(_validation_records)
_run_evidence_path = Path(Config.REPORTS_DIR) / "class_weight_validation_evidence.csv"
_validation_evidence.to_csv(_run_evidence_path, index=False)

_comparison_path = Path(Config.REPORTS_DIR).parent.parent / "class_weight_validation_runs.csv"
if _comparison_path.is_file():
    _all_validation_evidence = pd.read_csv(_comparison_path)
    _all_validation_evidence = _all_validation_evidence[
        _all_validation_evidence["run_id"] != _RUN_ID
    ]
    _all_validation_evidence = pd.concat(
        [_all_validation_evidence, _validation_evidence], ignore_index=True
    )
else:
    _all_validation_evidence = _validation_evidence.copy()
_all_validation_evidence.to_csv(_comparison_path, index=False)
_all_validation_evidence.to_csv(
    Path(Config.REPORTS_DIR) / "class_weight_validation_comparison.csv", index=False
)

print(
    f"Validation only | class weights={Config.USE_CLASS_WEIGHTS} | "
    f"accuracy={_validation_accuracy:.4f} | QWK={_validation_qwk:.4f}"
)
print("Per-class recall and support:")
print(
    _validation_evidence[["class_id", "class_name", "recall", "support"]]
    .to_string(index=False)
)
print(f"Saved this run's metrics -> {_run_evidence_path}")
print(f"Saved accumulated comparison -> {_comparison_path}")

_latest_run_ids = (
    _all_validation_evidence.sort_values("run_id")
    .groupby("class_weights_enabled")["run_id"]
    .last()
)
if {False, True}.issubset(set(_latest_run_ids.index)):
    _off_run_id = _latest_run_ids.loc[False]
    _on_run_id = _latest_run_ids.loc[True]
    _run_summaries = _all_validation_evidence.drop_duplicates("run_id").set_index("run_id")
    _off_summary = _run_summaries.loc[_off_run_id]
    _on_summary = _run_summaries.loc[_on_run_id]
    _matched_fields = [
        "split_fingerprint",
        "seed",
        "image_size",
        "batch_size",
        "phase1_learning_rate",
        "phase2_learning_rate",
        "phase1_epochs",
        "phase2_epochs",
        "dropout_rate",
        "phase2_unfreeze_top_n",
    ]
    _matched = all(_off_summary[field] == _on_summary[field] for field in _matched_fields)
    if _matched:
        print("Matched validation comparison (class weights ON minus OFF):")
        print(
            f"  Accuracy delta: {_on_summary['validation_accuracy'] - _off_summary['validation_accuracy']:+.4f}"
        )
        print(
            f"  QWK delta     : {_on_summary['validation_qwk'] - _off_summary['validation_qwk']:+.4f}"
        )
        _recall_comparison = _all_validation_evidence[
            _all_validation_evidence["run_id"].isin([_off_run_id, _on_run_id])
        ].pivot(index=["class_id", "class_name"], columns="class_weights_enabled", values="recall")
        _recall_comparison["recall_delta_on_minus_off"] = (
            _recall_comparison[True] - _recall_comparison[False]
        )
        print(_recall_comparison.to_string())
    else:
        print(
            "Both weight settings are logged, but their split fingerprint, seed, or "
            "training settings differ; this is not a matched comparison."
        )
else:
    print(
        "Paired comparison pending: run the other class-weight setting with the same "
        "saved splits, seed, and training settings."
    )

In [ ]:
# ============================================================
# SECTION 7.1 — TEST INFERENCE, SAVED PREDICTIONS & BASELINE
# Run one model inference pass on test; all test metrics derive from this NPZ.
# The majority-class baseline predicts the most common TRAIN label.
# ============================================================

def generate_test_predictions(
    eval_model: keras.Model,
    test_ds: tf.data.Dataset,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return test labels, raw argmax predictions, and class probabilities."""
    print("[Evaluation] Running inference on the held-out test set once...")
    y_true_batches = []
    y_prob_batches = []

    for images, labels in tqdm(test_ds, desc="Evaluating test set"):
        if USE_TTA_FOR_EVALUATION:
            probabilities = predict_probs_tta(eval_model, images)
        else:
            probabilities = eval_model(images, training=False).numpy()
        y_true_batches.append(np.argmax(labels.numpy(), axis=-1))
        y_prob_batches.append(probabilities)

    probabilities = np.concatenate(y_prob_batches, axis=0)
    y_true = np.concatenate(y_true_batches, axis=0)
    y_pred_argmax = np.argmax(probabilities, axis=1)
    print(f"[Evaluation] Inference complete for {len(y_true):,} test samples.")
    return y_true, y_pred_argmax, probabilities


# Define the naive baseline from training-label frequencies, never test-label frequencies.
_train_class_counts = train_df["label"].value_counts().sort_index()
MAJORITY_CLASS = int(_train_class_counts.idxmax())
print(
    f"[Baseline] Majority class from training split: {MAJORITY_CLASS} "
    f"({Config.CLASS_NAMES[MAJORITY_CLASS]}), "
    f"{int(_train_class_counts.loc[MAJORITY_CLASS]):,}/{len(train_df):,} training images."
)

# Preserve raw argmax metrics, then apply thresholds fixed using validation only.
y_test_true, y_test_pred_argmax, y_test_prob = generate_test_predictions(model, test_dataset)
y_test_pred = apply_qwk_thresholds(y_test_prob, QWK_THRESHOLDS)
y_test_pred_majority = np.full(y_test_true.shape, MAJORITY_CLASS, dtype=int)
test_predictions_path = os.path.join(Config.REPORTS_DIR, "test_predictions.npz")
np.savez_compressed(
    test_predictions_path,
    y_true=y_test_true,
    y_pred=y_test_pred,
    y_pred_argmax=y_test_pred_argmax,
    y_pred_majority=y_test_pred_majority,
    majority_class=np.asarray(MAJORITY_CLASS),
    softmax_probabilities=y_test_prob,
    qwk_thresholds=QWK_THRESHOLDS,
    use_tta=np.asarray(USE_TTA_FOR_EVALUATION),
    filepaths=np.asarray(test_df["filepath"].tolist()),
)

with np.load(test_predictions_path) as saved_test_predictions:
    y_test_true = saved_test_predictions["y_true"]
    y_test_pred = saved_test_predictions["y_pred"]
    y_test_pred_argmax = saved_test_predictions["y_pred_argmax"]
    y_test_pred_majority = saved_test_predictions["y_pred_majority"]
    y_test_prob = saved_test_predictions["softmax_probabilities"]
    MAJORITY_CLASS = int(saved_test_predictions["majority_class"].item())
    test_filepaths = saved_test_predictions["filepaths"].astype(str)

if not np.array_equal(y_test_pred_argmax, np.argmax(y_test_prob, axis=1)):
    raise AssertionError("Saved argmax predictions do not match saved softmax probabilities.")
if not np.array_equal(y_test_pred_majority, np.full_like(y_test_true, MAJORITY_CLASS)):
    raise AssertionError("Saved majority-baseline predictions do not match the saved baseline class.")
if len(test_filepaths) != len(y_test_true):
    raise ValueError("Saved test paths and prediction arrays have different lengths.")

# Compare all decision rules from the same persisted truth/prediction arrays.
_test_prediction_sets = {
    "majority_class_baseline": y_test_pred_majority,
    "efficientnet_argmax": y_test_pred_argmax,
    "efficientnet_validation_qwk_thresholds": y_test_pred,
}
_test_comparison_records = []
for _method, _predictions in _test_prediction_sets.items():
    _method_report = classification_report(
        y_test_true,
        _predictions,
        labels=list(range(Config.NUM_CLASSES)),
        target_names=Config.CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    _test_comparison_records.append({
        "method": _method,
        "majority_class": MAJORITY_CLASS,
        "accuracy": float(np.mean(y_test_true == _predictions)),
        "quadratic_weighted_kappa": float(
            cohen_kappa_score(y_test_true, _predictions, weights="quadratic")
        ),
        "macro_f1": float(_method_report["macro avg"]["f1-score"]),
        "weighted_f1": float(_method_report["weighted avg"]["f1-score"]),
        **{
            f"recall_{class_name.replace(' ', '_')}": float(_method_report[class_name]["recall"])
            for class_name in Config.CLASS_NAMES
        },
    })

_test_comparison_df = pd.DataFrame(_test_comparison_records)
_test_comparison_path = Path(Config.REPORTS_DIR) / "model_vs_majority_baseline.csv"
_test_comparison_df.to_csv(_test_comparison_path, index=False)
print("[Evaluation] Metrics derived from the saved one-pass test prediction artifact:")
display(_test_comparison_df)
print(f"[Evaluation] Saved model/baseline comparison -> {_test_comparison_path}")
print(f"[Evaluation] Saved one-pass test predictions -> {test_predictions_path}")

In [ ]:
# ============================================================
# SECTION 7.2 — STANDARD ACCURACY & QUADRATIC WEIGHTED KAPPA
# ============================================================

def calculate_clinical_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> Dict[str, float]:
    """Calculate accuracy and quadratic weighted kappa for ordinal grades."""
    accuracy = float(np.mean(y_true == y_pred))
    qwk = float(cohen_kappa_score(y_true, y_pred, weights="quadratic"))

    print("\n" + "=" * 65)
    print("  CLINICAL & STATISTICAL METRICS SUMMARY")
    print("=" * 65)
    print(f"  Total Test Samples          : {len(y_true):,}")
    print(f"  Standard Accuracy           : {accuracy:.4f}  ({accuracy * 100:.2f}%)")
    print(f"  Quadratic Weighted Kappa    : {qwk:.4f}")
    print("-" * 65)

    if qwk >= 0.80:
        agreement_desc = "Very strong ordinal agreement; not evidence of specialist equivalence"
    elif qwk >= 0.60:
        agreement_desc = "Substantial ordinal agreement"
    elif qwk >= 0.40:
        agreement_desc = "Moderate ordinal agreement"
    else:
        agreement_desc = "Fair or slight ordinal agreement"

    print(f"  QWK interpretation          : {agreement_desc}")
    print("=" * 65 + "\n")
    return {"accuracy": accuracy, "quadratic_weighted_kappa": qwk}


argmax_eval_metrics = calculate_clinical_metrics(y_test_true, y_test_pred_argmax)  # CHANGED: report plain argmax metrics
print("Thresholded predictions:")  # CHANGED: separate calibrated decision rule
eval_metrics = calculate_clinical_metrics(y_test_true, y_test_pred)  # CHANGED: keep calibrated metrics for existing downstream cells
print("Thresholding may raise QWK while lowering plain accuracy.")  # CHANGED: disclose calibration trade-off

# Headline table: both decision rules on the same saved test predictions.
from sklearn.metrics import f1_score

_labels = list(range(Config.NUM_CLASSES))
headline_metrics = pd.DataFrame([
    {
        "decision_rule": "Argmax (plain)",
        "accuracy": argmax_eval_metrics["accuracy"],
        "qwk": argmax_eval_metrics["quadratic_weighted_kappa"],
        "macro_f1": float(f1_score(y_test_true, y_test_pred_argmax, labels=_labels, average="macro", zero_division=0)),
        "headline_for": "accuracy",
    },
    {
        "decision_rule": "Validation-QWK thresholds",
        "accuracy": eval_metrics["accuracy"],
        "qwk": eval_metrics["quadratic_weighted_kappa"],
        "macro_f1": float(f1_score(y_test_true, y_test_pred, labels=_labels, average="macro", zero_division=0)),
        "headline_for": "QWK",
    },
])
headline_metrics_path = os.path.join(Config.REPORTS_DIR, "test_decision_rule_summary.csv")
headline_metrics.to_csv(headline_metrics_path, index=False)
print("\nTEST SET — both decision rules (thresholds chosen on validation only):")
for _row in headline_metrics.itertuples():
    print(f"  {_row.decision_rule:<27} — accuracy {_row.accuracy:.4f} / QWK {_row.qwk:.4f} / "
          f"macro-F1 {_row.macro_f1:.4f}   (headline {_row.headline_for})")
print(f"[Evaluation] Saved -> {headline_metrics_path}")


# ------------------------------------------------------------
# TEST LOSS - one number, computed from the saved one-pass test
# probabilities with the same loss the model was trained on
# (categorical cross-entropy, label_smoothing=0.1). This equals what
# model.evaluate() reports for plain inference, without a second test pass.
# ------------------------------------------------------------
_y_test_one_hot = tf.one_hot(y_test_true, Config.NUM_CLASSES)
TEST_LOSS = float(
    keras.losses.CategoricalCrossentropy(label_smoothing=0.1)(_y_test_one_hot, y_test_prob)
)
TEST_LOSS_UNSMOOTHED = float(
    keras.losses.CategoricalCrossentropy()(_y_test_one_hot, y_test_prob)
)
print(f"Test Loss: {TEST_LOSS:.4f}  (training loss definition, label smoothing 0.1)")
print(f"Test cross-entropy without label smoothing: {TEST_LOSS_UNSMOOTHED:.4f}")
if USE_TTA_FOR_EVALUATION:
    print("Note: computed on the validation-selected 4-view TTA probabilities.")
pd.DataFrame([{
    "test_loss": TEST_LOSS,
    "test_cross_entropy_unsmoothed": TEST_LOSS_UNSMOOTHED,
    "label_smoothing": 0.1,
    "use_tta": bool(USE_TTA_FOR_EVALUATION),
    "test_samples": int(len(y_test_true)),
    "argmax_accuracy": argmax_eval_metrics["accuracy"],
    "argmax_qwk": argmax_eval_metrics["quadratic_weighted_kappa"],
    "thresholded_accuracy": eval_metrics["accuracy"],
    "thresholded_qwk": eval_metrics["quadratic_weighted_kappa"],
}]).to_csv(os.path.join(Config.REPORTS_DIR, "test_loss_and_metrics.csv"), index=False)
print(f"[Evaluation] Saved -> {os.path.join(Config.REPORTS_DIR, 'test_loss_and_metrics.csv')}")


In [ ]:
# ============================================================
# SECTION 7.3 — DETAILED CLASSIFICATION REPORT
#
# Generates per-class Precision, Recall, and F1-Score along with
# Macro and Weighted Averages.
#
# Note on Imbalanced Datasets:
#   - Macro Average: Unweighted mean across all 5 classes — treats
#     rare Proliferative DR equally with frequent No DR.
#   - Weighted Average: Weighted by class support — dominated by No DR.
# In clinical disease staging, the Macro Average and minority class
# recalls (classes 3 & 4) are the most critical safety indicators.
# ============================================================

def generate_classification_summary(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: List[str] = Config.CLASS_NAMES,
    save_path: Optional[str] = None,
) -> pd.DataFrame:
    """
    Generate and display a structured classification report DataFrame.

    Args:
        y_true: True integer labels.
        y_pred: Predicted integer labels.
        class_names: List of class names ordered by class index 0-4.
        save_path: Optional path to save CSV report.

    Returns:
        Formatted pandas DataFrame containing per-class and aggregated metrics.
    """
    report_dict = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report_dict).transpose()

    print("\n" + "=" * 70)
    print("  PER-STAGE CLASSIFICATION REPORT (PRECISION, RECALL, F1-SCORE)")
    print("=" * 70)
    print(report_df.round(4).to_string())
    print("=" * 70)

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        report_df.to_csv(save_path)
        print(f"[Evaluation] Saved classification report -> {save_path}")

    return report_df


report_df = generate_classification_summary(
    y_test_true,
    y_test_pred,
    save_path=os.path.join(Config.REPORTS_DIR, "classification_report.csv"),
)

In [ ]:
# ============================================================
# SECTION 7.4 — CONFUSION MATRIX & CLINICAL ERROR ANALYSIS
#
# Generates normalized and raw confusion matrices as a heatmap
# and produces an automated clinical interpretation of misclassifications:
# - Identifies adjacent stage confusions (e.g. Mild vs Moderate)
# - Flags high-risk distant confusions (e.g. No DR vs Severe/PDR)
# - Provides clinical hypotheses for boundary classification ambiguities.
# ============================================================

def plot_confusion_matrix_heatmap(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: List[str] = Config.CLASS_NAMES,
    save_path: Optional[str] = None,
) -> np.ndarray:
    """
    Plot and save dual confusion matrices (raw counts and normalized percentages).

    Args:
        y_true: Ground truth integer class labels.
        y_pred: Predicted integer class labels.
        class_names: Human-readable class names.
        save_path: Optional path to save PNG.

    Returns:
        Raw confusion matrix 2D NumPy array.
    """
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    # Normalized by true row support (recall per class)
    cm_norm = cm.astype("float") / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Subplot 1: Absolute Counts
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[0],
        cbar=True,
    )
    axes[0].set_title("Confusion Matrix (Counts)", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Predicted Stage", fontsize=11)
    axes[0].set_ylabel("True Stage", fontsize=11)
    axes[0].tick_params(axis="x", rotation=30)

    # Subplot 2: Normalized (Recall) Proportions
    sns.heatmap(
        cm_norm,
        annot=True,
        fmt=".2%",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=axes[1],
        cbar=True,
    )
    axes[1].set_title("Confusion Matrix (Normalized by True Class)", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Predicted Stage", fontsize=11)
    axes[1].set_ylabel("True Stage", fontsize=11)
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[Confusion Matrix] Saved figure -> {save_path}")

    plt.show()
    return cm


def interpret_confusion_matrix(
    cm: np.ndarray,
    class_names: List[str] = Config.CLASS_NAMES,
    save_path: Optional[str] = None,
) -> Optional[pd.DataFrame]:
    """
    Analyze the confusion matrix to identify primary error patterns and print clinical hypotheses.

    Args:
        cm: 2D integer confusion matrix array (rows=true, cols=pred).
        class_names: Class labels.
        save_path: Optional CSV path for every misclassification pair, ranked by count.
    """
    print("\n" + "=" * 70)
    print("  CLINICAL CONFUSION INTERPRETATION & HYPOTHESIS ANALYSIS")
    print("=" * 70)

    # Collect all off-diagonal confusion counts
    off_diagonals = []
    n_classes = len(class_names)
    for i in range(n_classes):
        for j in range(n_classes):
            if i != j and cm[i, j] > 0:
                off_diagonals.append((cm[i, j], i, j, abs(i - j)))

    # Sort by error count descending
    off_diagonals.sort(key=lambda x: x[0], reverse=True)

    total_errors = sum(count for count, *_ in off_diagonals)
    pairs_df = pd.DataFrame([
        {
            "rank": rank,
            "true_label": true_cls,
            "true_class": class_names[true_cls],
            "predicted_label": pred_cls,
            "predicted_class": class_names[pred_cls],
            "count": int(count),
            "percent_of_true_class": float(count / cm[true_cls].sum() * 100),
            "percent_of_all_errors": float(count / total_errors * 100) if total_errors else 0.0,
            "stage_distance": distance,
            "error_type": "adjacent" if distance == 1 else "distant (high clinical risk)",
        }
        for rank, (count, true_cls, pred_cls, distance) in enumerate(off_diagonals, start=1)
    ])
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        pairs_df.to_csv(save_path, index=False)
        print(f"[Confusion Pairs] Saved all {len(pairs_df)} misclassification pairs -> {save_path}")

    print("Top Misclassification Pairs (Observed in Test Data):")
    for count, true_cls, pred_cls, distance in off_diagonals[:5]:
        dist_type = "Adjacent (Distance 1)" if distance == 1 else f"Distant (Distance {distance}) - High Clinical Risk!"
        print(
            f"  - True [{class_names[true_cls]}] misclassified as [{class_names[pred_cls]}]: "
            f"{count:,} cases | {dist_type}"
        )

    print("\nGeneral clinical background (not derived from this run's results):")
    print("Clinical Diagnosis & Boundary Ambiguity Hypotheses:")
    print("1. Mild NPDR (Stage 1) vs. Moderate NPDR (Stage 2):")
    print("   - Clinical criterion for Stage 1 is microaneurysms only. Stage 2 involves early dot")
    print("     blot hemorrhages, venous beading, or hard exudates.")
    print(f"   - In downsampled {Config.IMG_SIZE}x{Config.IMG_SIZE} images, faint microaneurysms and small dot hemorrhages")
    print("     can look identical at single-pixel scales, causing the model to blur this boundary.")

    print("2. Moderate NPDR (Stage 2) vs. Severe NPDR (Stage 3):")
    print("   - Severe NPDR requires fulfilling the '4-2-1 rule' (>20 intraretinal hemorrhages in")
    print("     all 4 quadrants, venous beading in 2+ quadrants, or IRMA in 1+ quadrant).")
    print("   - Without explicit lesion segmentation or full-retina multi-quadrant counting,")
    print("     global average pooled features capture lesion density but cannot formally count")
    print("     quadrant coverage, resulting in adjacent class confusion.")

    print("3. Distant Misclassification Safeguards:")
    print("   - Severe/Proliferative cases misdiagnosed as No DR represent fatal false negatives.")
    print("   - The GovernanceAgent (Section 10) explicitly gates cases where model confidence is")
    print("     uncertain (< 70%), diverting them to human ophthalmologist triage.")

    # Computed from this run's confusion matrix (rows = true stage, columns = predicted).
    def _rate(count: int, total: int) -> str:
        return f"{count / total * 100:.1f}%" if total else "n/a"

    row_totals = cm.sum(axis=1)
    print("\nWhat this run's confusion matrix shows for each hypothesis:")
    if n_classes >= 5:
        print(
            f"1. Mild -> Moderate: {cm[1, 2]:,} of {row_totals[1]:,} Mild ({_rate(cm[1, 2], row_totals[1])}); "
            f"Moderate -> Mild: {cm[2, 1]:,} of {row_totals[2]:,} Moderate ({_rate(cm[2, 1], row_totals[2])})."
        )
        print(
            f"2. Moderate -> Severe: {cm[2, 3]:,} of {row_totals[2]:,} Moderate ({_rate(cm[2, 3], row_totals[2])}); "
            f"Severe -> Moderate: {cm[3, 2]:,} of {row_totals[3]:,} Severe ({_rate(cm[3, 2], row_totals[3])})."
        )
        severe_as_no_dr = int(cm[3, 0] + cm[4, 0])
        severe_total = int(row_totals[3] + row_totals[4])
        print(
            f"3. Severe/Proliferative predicted as No DR: {severe_as_no_dr:,} of {severe_total:,} "
            f"({_rate(severe_as_no_dr, severe_total)}) — Severe {cm[3, 0]:,}, Proliferative {cm[4, 0]:,}."
        )
    print("=" * 70 + "\n")
    return pairs_df


test_cm = plot_confusion_matrix_heatmap(
    y_test_true,
    y_test_pred,
    save_path=os.path.join(Config.REPORTS_DIR, "confusion_matrix.png"),
)
top_confusion_pairs = interpret_confusion_matrix(
    test_cm,
    save_path=os.path.join(Config.REPORTS_DIR, "top_confusion_pairs.csv"),
)
display(top_confusion_pairs.head(10))

In [ ]:
# ============================================================
# SECTION 7.4b — ARGMAX REPORTS AND ROC FROM SAVED PREDICTIONS
# No model inference is performed here; all results come from the NPZ.
# ============================================================

from sklearn.metrics import auc, roc_curve  # CHANGED: compute ROC from persisted softmax scores

with np.load(test_predictions_path) as saved_test_predictions:  # CHANGED: use the one-pass test artifact
    artifact_y_true = saved_test_predictions["y_true"]  # CHANGED: load labels from artifact
    artifact_y_pred = saved_test_predictions["y_pred"]  # CHANGED: load thresholded predictions
    artifact_y_pred_argmax = saved_test_predictions["y_pred_argmax"]  # CHANGED: load plain predictions
    artifact_probabilities = saved_test_predictions["softmax_probabilities"]  # CHANGED: load scores for ROC

if not np.array_equal(artifact_y_pred_argmax, np.argmax(artifact_probabilities, axis=1)):  # CHANGED: verify stored argmax consistency
    raise AssertionError("Saved argmax predictions do not match the saved softmax probabilities.")  # CHANGED: stop on inconsistent evidence

argmax_report_df = generate_classification_summary(  # CHANGED: report raw argmax metrics from saved arrays
    artifact_y_true,
    artifact_y_pred_argmax,
    save_path=os.path.join(Config.REPORTS_DIR, "classification_report_argmax.csv"),
)
argmax_cm = plot_confusion_matrix_heatmap(  # CHANGED: create the complementary raw argmax confusion matrix
    artifact_y_true,
    artifact_y_pred_argmax,
    save_path=os.path.join(Config.REPORTS_DIR, "confusion_matrix_argmax.png"),
)
interpret_confusion_matrix(argmax_cm)  # CHANGED: interpret only predictions stored in the artifact

roc_records = []  # CHANGED: collect per-class ROC summary from saved probabilities
fig, ax = plt.subplots(figsize=(8, 7))  # CHANGED: create one-vs-rest ROC figure
for class_id, class_name in enumerate(Config.CLASS_NAMES):  # CHANGED: calculate one ROC curve per ICDR stage
    binary_truth = (artifact_y_true == class_id).astype(int)  # CHANGED: form one-vs-rest labels
    if np.unique(binary_truth).size != 2:  # CHANGED: require positives and negatives for AUC
        raise ValueError(f"Test artifact is missing one-vs-rest examples for class {class_id}.")  # CHANGED: fail instead of reporting invalid AUC
    false_positive_rate, true_positive_rate, _ = roc_curve(binary_truth, artifact_probabilities[:, class_id])  # CHANGED: use saved softmax column
    class_auc = float(auc(false_positive_rate, true_positive_rate))  # CHANGED: integrate ROC curve
    roc_records.append({"class_id": class_id, "class_name": class_name, "roc_auc": class_auc})  # CHANGED: persist class AUC
    ax.plot(false_positive_rate, true_positive_rate, label=f"{class_name} (AUC={class_auc:.3f})")  # CHANGED: plot per-stage ROC

macro_auc = float(np.mean([record["roc_auc"] for record in roc_records]))  # CHANGED: macro-average one-vs-rest AUC
roc_records.append({"class_id": "macro", "class_name": "Macro average", "roc_auc": macro_auc})  # CHANGED: include aggregate AUC in CSV
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")  # CHANGED: show chance baseline
ax.set(title=f"One-vs-Rest ROC Curves (Macro AUC={macro_auc:.3f})", xlabel="False Positive Rate", ylabel="True Positive Rate")  # CHANGED: label plot
ax.legend(loc="lower right", fontsize=8)  # CHANGED: identify each class curve
ax.grid(alpha=0.25)  # CHANGED: improve plot readability
fig.tight_layout()  # CHANGED: fit labels and legend
roc_plot_path = os.path.join(Config.REPORTS_DIR, "roc_auc_curves.png")  # CHANGED: save report figure in run directory
fig.savefig(roc_plot_path, dpi=150, bbox_inches="tight")  # CHANGED: persist ROC figure
plt.show()  # CHANGED: display ROC figure
pd.DataFrame(roc_records).to_csv(os.path.join(Config.REPORTS_DIR, "roc_auc_summary.csv"), index=False)  # CHANGED: persist per-class and macro AUC values
print(f"[Evaluation] Macro one-vs-rest ROC-AUC: {macro_auc:.4f}")  # CHANGED: report artifact-derived AUC
print(f"[Evaluation] ROC figure saved -> {roc_plot_path}")  # CHANGED: disclose ROC artifact path

In [ ]:
# ============================================================
# SECTION 7.4c — PER-STAGE RECALL COMPARISON
# Reuses the saved one-pass test predictions; performs no new inference.
# ============================================================

with np.load(test_predictions_path) as saved_predictions:
    recall_true = saved_predictions["y_true"].astype(int)
    recall_prediction_sets = {
        "Majority baseline": saved_predictions["y_pred_majority"].astype(int),
        "EfficientNet argmax": saved_predictions["y_pred_argmax"].astype(int),
        "Validation-calibrated": saved_predictions["y_pred"].astype(int),
    }

recall_records = []
for method_name, method_predictions in recall_prediction_sets.items():
    method_report = classification_report(
        recall_true,
        method_predictions,
        labels=list(range(Config.NUM_CLASSES)),
        target_names=Config.CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    for class_id, class_name in enumerate(Config.CLASS_NAMES):
        recall_records.append({
            "class_id": class_id,
            "class_name": class_name,
            "method": method_name,
            "recall": float(method_report[class_name]["recall"]),
            "support": int(method_report[class_name]["support"]),
        })

per_class_recall_comparison = pd.DataFrame(recall_records)
recall_table = per_class_recall_comparison.pivot(
    index=["class_id", "class_name", "support"],
    columns="method",
    values="recall",
).reset_index()
recall_csv_path = Path(Config.REPORTS_DIR) / "per_class_recall_comparison.csv"
per_class_recall_comparison.to_csv(recall_csv_path, index=False)
display(recall_table.round(4))

recall_plot_data = per_class_recall_comparison.pivot(
    index="class_name", columns="method", values="recall"
).reindex(Config.CLASS_NAMES)
fig, ax = plt.subplots(figsize=(12, 6))
recall_plot_data.plot(
    kind="bar",
    ax=ax,
    color=["#7A8F55", "#367C8A", "#D8784B"],
    width=0.78,
)
ax.set_title("Per-Stage Recall from the Saved Held-Out Test Predictions")
ax.set_xlabel("Dataset grade")
ax.set_ylabel("Recall")
ax.set_ylim(0, 1)
ax.legend(title="Decision rule")
ax.grid(axis="y", alpha=0.25)
ax.tick_params(axis="x", rotation=18)
fig.tight_layout()
recall_plot_path = Path(Config.REPORTS_DIR) / "per_class_recall_comparison.png"
fig.savefig(recall_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"[Recall Comparison] Reused saved test predictions; no model inference was run.")
print(f"[Recall Comparison] Saved table -> {recall_csv_path}")
print(f"[Recall Comparison] Saved plot -> {recall_plot_path}")

### 7.4d DR detection and referable-DR screening

The brief asks the model to classify diabetic retinopathy **as well as** its stage. The next cell collapses the five-stage predictions into *any DR* (stage ≥ 1) and *referable DR* (stage ≥ 2), and reports sensitivity, specificity, PPV, NPV and AUC for each.

In [ ]:
# ============================================================
# SECTION 7.4d — DR DETECTION (BINARY) AND REFERABLE-DR SCREENING
#
# The brief asks the model to detect diabetic retinopathy AS WELL AS
# its stage. The 5-stage predictions are collapsed into two screening
# questions, using the saved one-pass test predictions (no new inference):
#   - Any DR       : stage >= 1  (DR present vs No DR)
#   - Referable DR : stage >= 2  (the usual screening-programme cut-off
#                    for referral to an ophthalmologist)
# Sensitivity = share of diseased eyes that are flagged (missed cases hurt);
# specificity = share of healthy eyes correctly cleared (false alarms cost).
# AUC uses the summed softmax probability of the positive stages.
# ============================================================

from sklearn.metrics import roc_auc_score

SCREENING_CUTOFFS = {"Any DR (stage >= 1)": 1, "Referable DR (stage >= 2)": 2}


def screening_metrics(y_true: np.ndarray, y_pred: np.ndarray, cutoff: int) -> Dict[str, float]:
    """Binary confusion counts and rates after collapsing stages at `cutoff`."""
    true_positive_mask = np.asarray(y_true) >= cutoff
    pred_positive_mask = np.asarray(y_pred) >= cutoff
    tp = int(np.sum(true_positive_mask & pred_positive_mask))
    tn = int(np.sum(~true_positive_mask & ~pred_positive_mask))
    fp = int(np.sum(~true_positive_mask & pred_positive_mask))
    fn = int(np.sum(true_positive_mask & ~pred_positive_mask))

    def _ratio(numerator: int, denominator: int) -> float:
        return float(numerator / denominator) if denominator else float("nan")

    return {
        "tp": tp, "fn": fn, "fp": fp, "tn": tn,
        "sensitivity": _ratio(tp, tp + fn),
        "specificity": _ratio(tn, tn + fp),
        "ppv": _ratio(tp, tp + fp),
        "npv": _ratio(tn, tn + fn),
        "accuracy": _ratio(tp + tn, tp + tn + fp + fn),
    }


with np.load(test_predictions_path) as saved_predictions:
    screening_true = saved_predictions["y_true"].astype(int)
    screening_probabilities = saved_predictions["softmax_probabilities"]
    screening_rules = {
        "EfficientNet argmax": saved_predictions["y_pred_argmax"].astype(int),
        "Validation-calibrated": saved_predictions["y_pred"].astype(int),
    }

screening_records = []
for task_name, cutoff in SCREENING_CUTOFFS.items():
    positive_probability = screening_probabilities[:, cutoff:].sum(axis=1)
    task_auc = float(roc_auc_score(screening_true >= cutoff, positive_probability))
    for rule_name, rule_predictions in screening_rules.items():
        screening_records.append({
            "task": task_name,
            "decision_rule": rule_name,
            "positives_in_test": int(np.sum(screening_true >= cutoff)),
            "negatives_in_test": int(np.sum(screening_true < cutoff)),
            **screening_metrics(screening_true, rule_predictions, cutoff),
            "roc_auc": task_auc,
        })

screening_df = pd.DataFrame(screening_records)
screening_csv_path = Path(Config.REPORTS_DIR) / "screening_metrics.csv"
screening_df.to_csv(screening_csv_path, index=False)
display(screening_df.round(4))

# Clinically critical errors under the calibrated rule used for the headline results.
calibrated_predictions = screening_rules["Validation-calibrated"]
missed_referable = int(np.sum((screening_true >= 2) & (calibrated_predictions < 2)))
sight_threatening_as_no_dr = int(np.sum((screening_true >= 3) & (calibrated_predictions == 0)))
print(f"Referable-DR eyes missed (true >= 2, predicted < 2): {missed_referable:,} "
      f"of {int(np.sum(screening_true >= 2)):,}")
print(f"Severe/Proliferative eyes predicted as No DR       : {sight_threatening_as_no_dr:,}")

fig, axes = plt.subplots(1, len(SCREENING_CUTOFFS), figsize=(12, 5))
for axis, (task_name, cutoff) in zip(axes, SCREENING_CUTOFFS.items()):
    row = screening_df[(screening_df["task"] == task_name)
                       & (screening_df["decision_rule"] == "Validation-calibrated")].iloc[0]
    matrix = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]])
    negative_label = "No DR" if cutoff == 1 else "Stage 0-1"
    positive_label = "DR" if cutoff == 1 else "Stage 2-4"
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axis,
                xticklabels=[negative_label, positive_label],
                yticklabels=[negative_label, positive_label])
    axis.set_title(f"{task_name}\nSens {row['sensitivity']:.3f} | Spec {row['specificity']:.3f} "
                   f"| AUC {row['roc_auc']:.3f}", fontsize=11)
    axis.set_xlabel("Predicted")
    axis.set_ylabel("True")
fig.suptitle("Binary screening performance on the held-out test set (validation-calibrated rule)",
             fontsize=12, fontweight="bold")
fig.tight_layout()
screening_plot_path = Path(Config.REPORTS_DIR) / "screening_confusion_matrices.png"
fig.savefig(screening_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"[Screening] Saved table -> {screening_csv_path}")
print(f"[Screening] Saved figure -> {screening_plot_path}")

In [ ]:
# ============================================================
# SECTION 7.5 — SECTION SUMMARY
# ============================================================

print("=" * 60)
print("  SECTION 7 COMPLETE — Evaluation")
print("=" * 60)
print(f"  Accuracy                : {eval_metrics['accuracy']:.4f}")
print(f"  Quadratic Weighted Kappa: {eval_metrics['quadratic_weighted_kappa']:.4f}")
print(f"  Artifacts exported to {Config.REPORTS_DIR}:")
print("    - classification_report.csv")
print("    - confusion_matrix.png, confusion_matrix_argmax.png, roc_auc_curves.png")
print("    - per_class_recall_comparison.png")
print("    - screening_metrics.csv, screening_confusion_matrices.png")
print("  Next -> Section 8: Explainability — Grad-CAM")
print("=" * 60)

## Section 8: Explainability — Grad-CAM

**Goal:** Provide visual and linguistic interpretability for the model's predictions using
**Gradient-weighted Class Activation Mapping (Grad-CAM)** and an automated retinal quadrant describer.

### Clinical & Regulatory Rationale for Explainability in Medical AI

Deep neural networks are frequently criticized as "black boxes" in healthcare. The European Union
AI Act and FDA SaMD (Software as a Medical Device) guidelines require high-risk clinical AI
systems to provide explainability to clinical end-users.

**Grad-CAM** works by:
1. Identifying the last convolutional layer (`top_activation` in EfficientNetB3), which preserves
   both high-level semantic features (lesions, exudates, neovascular tufts) and spatial coordinates.
2. Computing the gradient of the winning class score $y^c$ with respect to feature activation maps $A^k$:
   $$\alpha_k^c = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A_{i,j}^k}$$
3. Computing a weighted combination followed by a rectified linear activation (ReLU):
   $$L_{\text{Grad-CAM}}^c = \text{ReLU}\left(\sum_k \alpha_k^c A^k\right)$$
   The $\text{ReLU}$ ensures the heatmap displays features that **positively contribute** to the
   predicted disease stage, rather than features that suppress it.

### Automated Retinal Quadrant Description
To make explainability immediately actionable for busy clinicians who do not want to inspect raw
heatmaps, we segment the retinal field into anatomical quadrants (Superior-Temporal, Superior-Nasal,
Inferior-Temporal, Inferior-Nasal, and Central Macular) and generate a natural language explanation.

In [ ]:
# ============================================================
# SECTION 8.1 — GRAD-CAM MODEL CONSTRUCTION
#
# Builds a gradient-tappable sub-model using the Keras functional API.
# It returns both:
#   1. The feature activation map from the final conv layer ('top_activation')
#   2. The final softmax prediction probabilities
# ============================================================

def build_gradcam_model(
    full_model: keras.Model,
    base: keras.Model,
    last_conv_layer_name: str = "top_activation",
) -> keras.Model:
    """
    Construct a dual-output Grad-CAM inspection model.

    Args:
        full_model: The complete trained DR_EfficientNetB3 model.
        base: The EfficientNetB3 base model sub-layer.
        last_conv_layer_name: Name of target convolutional layer in base model.
                              Defaults to 'top_activation' (last activation of EfficientNetB3).

    Returns:
        Keras Model with inputs=raw_image, outputs=[conv_features, class_predictions].
    """
    # 1. Access the target conv layer from the base feature extractor
    target_conv_layer = base.get_layer(last_conv_layer_name)

    # 2. Build intermediate sub-model for the base extractor
    base_submodel = keras.Model(
        inputs=base.inputs,
        outputs=[target_conv_layer.output, base.output],
        name="base_submodel",
    )

    # 3. Wire the full computational graph from raw inputs through the classification head
    inputs = keras.Input(shape=(Config.IMG_SIZE, Config.IMG_SIZE, 3), name="gradcam_input")
    scaled_inputs = layers.Rescaling(255.0, name="gradcam_restore_input_range")(inputs)
    conv_features, base_features = base_submodel(scaled_inputs)

    # Pass through head layers (reusing trained weights from full_model)
    x = full_model.get_layer("gap")(base_features)
    x = full_model.get_layer("head_bn")(x)
    x = full_model.get_layer("head_dense")(x)
    x = full_model.get_layer("head_dropout")(x)
    preds = full_model.get_layer("predictions")(x)

    gradcam_model = keras.Model(
        inputs=inputs,
        outputs=[conv_features, preds],
        name="gradcam_extractor",
    )
    return gradcam_model


# Instantiate Grad-CAM inspection model
gradcam_model = build_gradcam_model(model, base_model, last_conv_layer_name="top_activation")
print(f"[Grad-CAM] Model built successfully with target layer: 'top_activation'")

In [ ]:
# ============================================================
# SECTION 8.2 — GRAD-CAM COMPUTATION VIA TF.GRADIENTTAPE
#
# Computes Grad-CAM heatmaps for a target class.
# ============================================================

def compute_gradcam_heatmap(
    img_array: np.ndarray,
    cam_model: keras.Model = gradcam_model,
    pred_index: Optional[int] = None,
) -> Tuple[np.ndarray, int, float]:
    """Return a normalized float32 Grad-CAM map, class index, and confidence."""
    if img_array.ndim == 3:
        img_tensor = tf.expand_dims(img_array, axis=0)
    else:
        img_tensor = tf.convert_to_tensor(img_array)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        conv_outputs, predictions = cam_model(img_tensor, training=False)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0]).numpy()
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs_val = conv_outputs[0]
    heatmap = conv_outputs_val @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)

    max_val = tf.math.reduce_max(heatmap)
    if max_val > 0:
        heatmap = heatmap / max_val

    confidence = float(predictions[0, pred_index].numpy())
    heatmap = np.asarray(heatmap.numpy(), dtype=np.float32)
    return heatmap, int(pred_index), confidence

In [ ]:
# ============================================================
# SECTION 8.3 — HEATMAP SUPERIMPOSITION & QUADRANT GENERATOR
#
# 1. superimpose_gradcam(): Blends Jet colormap with RGB fundus image.
# 2. generate_quadrant_explanation(): Divides heatmap into retinal
#    anatomical regions and generates a natural-language description.
# ============================================================

def superimpose_gradcam(
    original_rgb: np.ndarray,
    heatmap: np.ndarray,
    alpha: float = 0.4,
    colormap: int = cv2.COLORMAP_JET,
) -> np.ndarray:
    """
    Overlay Grad-CAM heatmap onto the RGB fundus image.

    Args:
        original_rgb: RGB image (H, W, 3), float [0,1] or uint8 [0,255].
        heatmap: 2D float array (h, w) with values in [0, 1].
        alpha: Blending weight for heatmap (1-alpha weight for original image).
        colormap: OpenCV colormap constant.

    Returns:
        Superimposed RGB image as uint8 NumPy array of shape (H, W, 3).
    """
    # Standardize image to uint8 [0, 255]
    if original_rgb.dtype != np.uint8:
        base_img = (np.clip(original_rgb, 0.0, 1.0) * 255).astype(np.uint8)
    else:
        base_img = original_rgb.copy()

    h, w = base_img.shape[:2]

    # Resize heatmap to match original image dimensions
    heatmap_resized = cv2.resize(heatmap, (w, h), interpolation=cv2.INTER_LINEAR)
    heatmap_uint8 = (heatmap_resized * 255).astype(np.uint8)

    # Apply color palette (Jet: blue=low, red=peak activation)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, colormap)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    # Alpha blending: overlay = alpha * heatmap + (1 - alpha) * original
    overlay = cv2.addWeighted(heatmap_color, alpha, base_img, 1.0 - alpha, 0)
    return overlay


def generate_quadrant_explanation(
    heatmap: np.ndarray,
    predicted_stage: int,
    confidence: float,
) -> str:
    """
    Analyze the spatial distribution of Grad-CAM activations and generate a clinical description.

    Divides the retinal field into 5 key regions:
      - Superior-Temporal (upper outer quadrant)
      - Superior-Nasal (upper inner quadrant)
      - Inferior-Temporal (lower outer quadrant)
      - Inferior-Nasal (lower inner quadrant)
      - Central Macular (central 50% zone)

    Args:
        heatmap: 2D float array normalized to [0, 1].
        predicted_stage: Predicted DR stage (0 to 4).
        confidence: Prediction confidence score.

    Returns:
        Structured natural-language clinical explanation sentence.
    """
    h, w = heatmap.shape
    mid_y, mid_x = h // 2, w // 2

    # Define quadrant masks
    quadrants = {
        "superior-temporal": heatmap[:mid_y, :mid_x],
        "superior-nasal":    heatmap[:mid_y, mid_x:],
        "inferior-temporal": heatmap[mid_y:, :mid_x],
        "inferior-nasal":    heatmap[mid_y:, mid_x:],
    }

    # Central macular region (central 50% of the image)
    y1, y2 = int(0.25 * h), int(0.75 * h)
    x1, x2 = int(0.25 * w), int(0.75 * w)
    central_region = heatmap[y1:y2, x1:x2]

    # Calculate mean activation density per region
    scores = {name: float(np.mean(q)) for name, q in quadrants.items()}
    scores["central-macular"] = float(np.mean(central_region))

    # Identify primary and secondary regions of interest
    sorted_regions = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    peak_region, peak_score = sorted_regions[0]
    sec_region, sec_score   = sorted_regions[1]

    stage_name = Config.CLASS_NAMES[predicted_stage]

    # Clinical pathology mapping associated with stage
    pathology_notes = {
        0: "absence of diabetic microvascular abnormalities",
        1: "isolated microaneurysms or focal vascular dilatation",
        2: "microaneurysms, dot-and-blot hemorrhages, or hard exudates",
        3: "widespread intraretinal microvascular abnormalities (IRMA) or multi-quadrant hemorrhages",
        4: "neovascularization, vitreous preretinal hemorrhage, or fibrous proliferation",
    }

    explanation = (
        f"The model predicted {stage_name} (Stage {predicted_stage}) with {confidence*100:.1f}% confidence. "
        f"Grad-CAM salience indicates the model focused most strongly on the [{peak_region}] region "
        f"(salience density: {peak_score:.2f}), with secondary attention on the [{sec_region}] region, "
        f"consistent with {pathology_notes[predicted_stage]}."
    )
    return explanation

In [ ]:
# ============================================================
# SECTION 8.4 — GRAD-CAM EXAMPLES: CORRECT AND INCORRECT VALIDATION PREDICTIONS
# Select examples from validation predictions; do not inspect the held-out test set.
# The Grad-CAM target is the model's predicted class, not the ground-truth label.
# ============================================================

def evaluate_and_visualize_gradcam(
    validation_df: pd.DataFrame,
    validation_true: np.ndarray,
    validation_pred: np.ndarray,
    save_dir: str = Config.REPORTS_DIR,
) -> None:
    """Show one correct and one incorrect validation prediction per true stage when available."""
    validation_df = validation_df.reset_index(drop=True)
    validation_true = np.asarray(validation_true, dtype=int)
    validation_pred = np.asarray(validation_pred, dtype=int)
    if len(validation_df) != len(validation_true) or len(validation_true) != len(validation_pred):
        raise ValueError("Validation rows, labels, and predictions must have matching lengths.")

    selected_examples = []
    for true_stage in range(Config.NUM_CLASSES):
        for outcome, mask in (
            ("Correct", validation_pred == validation_true),
            ("Incorrect", validation_pred != validation_true),
        ):
            candidates = np.flatnonzero((validation_true == true_stage) & mask)
            if candidates.size:
                selected_examples.append((int(candidates[0]), outcome))

    if not selected_examples:
        raise ValueError("No validation examples were available for Grad-CAM visualization.")

    os.makedirs(save_dir, exist_ok=True)
    fig, axes = plt.subplots(
        len(selected_examples),
        3,
        figsize=(15, 4 * len(selected_examples)),
        squeeze=False,
    )
    column_headers = ["Validation fundus", "Grad-CAM for predicted class", "Predicted-class overlay"]
    for column, header in enumerate(column_headers):
        axes[0, column].set_title(header, fontsize=12, fontweight="bold")

    for row_index, (sample_index, outcome) in enumerate(selected_examples):
        sample_row = validation_df.iloc[sample_index]
        true_stage = int(validation_true[sample_index])
        predicted_stage = int(validation_pred[sample_index])
        image = preprocess_image(sample_row["filepath"])
        heatmap, _, confidence = compute_gradcam_heatmap(
            image,
            cam_model=gradcam_model,
            pred_index=predicted_stage,
        )
        overlay = superimpose_gradcam(image, heatmap, alpha=0.4)
        axes[row_index, 0].imshow(image)
        axes[row_index, 0].axis("off")
        axes[row_index, 0].set_ylabel(
            f"{outcome}\nTrue: {Config.CLASS_NAMES[true_stage]}\n"
            f"Predicted: {Config.CLASS_NAMES[predicted_stage]} ({confidence:.1%})",
            fontsize=9,
            rotation=0,
            labelpad=65,
            va="center",
        )
        axes[row_index, 1].imshow(heatmap, cmap="jet")
        axes[row_index, 1].axis("off")
        axes[row_index, 2].imshow(overlay)
        axes[row_index, 2].axis("off")
        print(
            f"[{outcome}] True={Config.CLASS_NAMES[true_stage]}, "
            f"predicted={Config.CLASS_NAMES[predicted_stage]}, "
            f"predicted-class confidence={confidence:.4f}. "
            "Grad-CAM indicates influential regions, not verified lesions or causal evidence."
        )

    fig.suptitle(
        "Grad-CAM examples from validation predictions (correct and incorrect where available)",
        fontsize=13,
        fontweight="bold",
    )
    plt.tight_layout()
    save_path = os.path.join(save_dir, "gradcam_correct_and_incorrect_validation.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"[Grad-CAM] Validation examples saved -> {save_path}")
    plt.show()


evaluate_and_visualize_gradcam(
    val_df,
    val_true,
    calibrated_val_predictions,
)

In [ ]:
# ============================================================
# SECTION 8.5 — SECTION SUMMARY
# ============================================================

print("=" * 60)
print("  SECTION 8 COMPLETE — Explainability (Grad-CAM)")
print("=" * 60)
print("  build_gradcam_model()          : dual-output model tapping 'top_activation'")
print("  compute_gradcam_heatmap()      : tf.GradientTape computation + ReLU")
print("  superimpose_gradcam()          : Jet colormap alpha blending")
print("  generate_quadrant_explanation(): anatomical quadrant natural-language generator")
print(f"  Artifact saved                 : {Config.REPORTS_DIR}/gradcam_correct_and_incorrect_validation.png")
print("  Next -> Innovation A: Embedding-Based Similar-Case Retrieval")
print("=" * 60)

## Section 9: Innovation Feature A — Embedding-Based Similar-Case Retrieval

**Purpose:** Show a query fundus image beside visually similar examples from the saved training split. The gallery uses real image files and labels from the supplied dataset; it does not generate synthetic photographs or independently verified clinical records.

### Rationale and Mechanism

The classifier's 256-dimensional `head_dense` representation is extracted before the softmax layer, L2-normalized, and compared with cached training embeddings using cosine similarity. For normalized vectors, cosine similarity is their dot product. The notebook's demonstration selects a validation image as the query and retrieves from the training split, keeping the query out of the reference set.

This was added as a case-comparison aid alongside the five-stage prediction. It is **not** a Siamese or metric-learning model: the embedding was learned through the classification objective. Similarity does not establish pathological equivalence, and dataset labels are not independent clinical confirmation. No retrieval accuracy, clinician-utility study, or controlled evidence that retrieval improves grading has been produced. Treat the gallery as an exploratory interface, not a clinical explanation or diagnosis.

In [ ]:
# ============================================================
# SECTION 9.1 — PENULTIMATE EMBEDDING EXTRACTOR MODEL
#
# Creates a dedicated feature extractor from the trained network.
# It taps the 256-dimensional Dense layer ('head_dense') immediately
# before the final 5-class softmax output layer.
# ============================================================

def build_embedding_extractor(trained_model: keras.Model) -> keras.Model:
    """
    Construct a feature extractor model outputting 256-D penultimate embeddings.

    Args:
        trained_model: Full trained DR_EfficientNetB3 model.

    Returns:
        Keras Model with inputs=raw_images, outputs=256-dimensional penultimate features.
    """
    # Tap the intermediate dense layer before dropout and softmax
    dense_output = trained_model.get_layer("head_dense").output

    embedding_extractor = keras.Model(
        inputs=trained_model.input,
        outputs=dense_output,
        name="penultimate_embedding_extractor",
    )
    return embedding_extractor


# Build embedding extractor
embedding_extractor = build_embedding_extractor(model)
print(f"[Embedding] Extractor built: {embedding_extractor.name}")
print(f"  Input shape : {embedding_extractor.input_shape}")
print(f"  Output shape: {embedding_extractor.output_shape} (256-dimensional latent representation)")

In [ ]:
# ============================================================
# SECTION 9.2 — EXTRACT & CACHE TRAINING SET EMBEDDINGS
#
# Generates 256-D penultimate embeddings for every training image,
# L2-normalises them (so cosine similarity becomes a dot product),
# and saves them to Config.EMBEDDINGS_PATH for the retrieval demo and app.
#
# Images are read from the preprocessed PNG cache built for training,
# so extraction does not repeat the OpenCV preprocessing for the training images.
# ============================================================

def extract_and_cache_training_embeddings(
    extractor: keras.Model,
    train_dataframe: pd.DataFrame,
    cache_path: str = Config.EMBEDDINGS_PATH,
    force_recompute: bool = False,
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """
    Extract L2-normalised embeddings for all training images and persist them as NPZ.

    For unit vectors u and v, cos_sim(u, v) = u . v, and
    ||u - v||^2 = 2 - 2 (u . v), so ranking by dot product is equivalent
    to nearest-neighbour search in Euclidean distance.

    Returns:
        embeddings (N, 256) float32, labels (N,) int, filepaths list[str]
    """
    if os.path.exists(cache_path) and not force_recompute:
        print(f"[Embedding] Loading cached embeddings from {cache_path}...")
        with np.load(cache_path, allow_pickle=True) as data:
            return data["embeddings"], data["labels"].astype(int), list(data["filepaths"])

    filepaths = train_dataframe["filepath"].tolist()
    labels = train_dataframe["label"].to_numpy(dtype=int)
    print(f"[Embedding] Computing embeddings for {len(filepaths):,} training images...")

    # shuffle=False keeps the dataset order identical to train_dataframe row order.
    embedding_dataset = _build_tuning_dataset(
        train_dataframe, Config.IMG_SIZE, shuffle=False, augment=False
    )
    batches = []
    for images, _ in tqdm(embedding_dataset, desc="Extracting embeddings"):
        features = extractor(images, training=False).numpy().astype(np.float32)
        features /= np.linalg.norm(features, axis=1, keepdims=True) + 1e-10
        batches.append(features)
    embeddings = np.concatenate(batches, axis=0)
    if len(embeddings) != len(filepaths):
        raise ValueError("Embedding count does not match the number of training images.")

    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    np.savez_compressed(cache_path, embeddings=embeddings, labels=labels, filepaths=np.asarray(filepaths))
    print(f"[Embedding] Saved {len(embeddings):,} embeddings -> {cache_path}")
    return embeddings, labels, filepaths


train_embeddings, train_labels_arr, train_fps = extract_and_cache_training_embeddings(
    embedding_extractor,
    train_df,
    cache_path=Config.EMBEDDINGS_PATH,
)
print(f"[Embedding] Matrix shape: {train_embeddings.shape}")

In [ ]:
# ============================================================
# SECTION 9.3 — REUSABLE SIMILAR-CASE RETRIEVAL FUNCTION
#
# find_similar_cases(query_image, k=3):
#   - Accepts an image array or file path
#   - Extracts and normalizes the 256-D query vector
#   - Computes cosine similarity to cached training embeddings
#   - Returns real training image paths with their dataset labels
# Similarity is a feature-space ranking, not evidence of clinical equivalence.
# ============================================================


def find_similar_cases(
    query_image: Union[str, np.ndarray],
    k: int = 3,
    extractor: keras.Model = embedding_extractor,
    stored_embeddings: np.ndarray = train_embeddings,
    stored_labels: np.ndarray = train_labels_arr,
    stored_filepaths: List[str] = train_fps,
) -> List[Dict[str, Any]]:
    """Return the top-k nearest dataset-labeled training examples."""
    if isinstance(query_image, str):
        img_arr = preprocess_image(query_image)
    else:
        img_arr = query_image.copy()

    query_tensor = np.expand_dims(img_arr, axis=0) if img_arr.ndim == 3 else img_arr
    query_embedding = extractor(query_tensor, training=False).numpy()
    query_embedding /= np.linalg.norm(query_embedding, axis=1, keepdims=True) + 1e-10
    similarities = np.dot(stored_embeddings, query_embedding.T).squeeze()
    top_indices = np.argsort(similarities)[::-1][:k]

    results = []
    for rank, index in enumerate(top_indices, start=1):
        stage = int(stored_labels[index])
        results.append({
            "rank": rank,
            "filepath": stored_filepaths[index],
            "label": stage,
            "stage_name": Config.CLASS_NAMES[stage],
            "cosine_similarity": float(similarities[index]),
        })
    return results


print("[Section 9] find_similar_cases(query_image, k=3) is defined.")

In [ ]:
# ============================================================
# SECTION 9.4 — VISUAL DEMONSTRATION OF SIMILAR-CASE RETRIEVAL
# The query is validation-only; reference images come from train_df.
# All gallery images are real files. Captions show supplied dataset labels.
# ============================================================


def visualize_similar_cases_demo(
    query_filepath: str,
    k: int = 3,
    save_path: Optional[str] = None,
) -> List[Dict[str, Any]]:
    """Plot a validation fundus image beside its nearest training references."""
    similar_cases = find_similar_cases(query_filepath, k=k)
    query_img = preprocess_image(query_filepath)
    fig, axes = plt.subplots(1, k + 1, figsize=(4 * (k + 1), 4.5))

    axes[0].imshow(query_img)
    axes[0].set_title("QUERY: VALIDATION IMAGE", fontsize=11, fontweight="bold")
    axes[0].axis("off")

    for index, case in enumerate(similar_cases, start=1):
        case_img = preprocess_image(case["filepath"])
        axes[index].imshow(case_img)
        axes[index].set_title(
            f"Training reference #{case['rank']}\n"
            f"Dataset label: {case['stage_name']} ({case['label']})\n"
            f"Cosine similarity: {case['cosine_similarity']:.3f}",
            fontsize=9,
        )
        axes[index].axis("off")

    plt.suptitle(
        "Similar-Case Retrieval Demo — Similarity Is Not Clinical Confirmation",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[Retrieval] Saved demonstration figure -> {save_path}")

    plt.show()
    print("\nNearest training references for validation query:")
    for case in similar_cases:
        print(
            f"  Rank {case['rank']}: dataset label [{case['stage_name']}] | "
            f"cosine similarity={case['cosine_similarity']:.4f}"
        )
        print(f"         Image file: {case['filepath']}")
    return similar_cases


sample_query_row = val_df[val_df["label"] >= 2].iloc[0]
sample_retrievals = visualize_similar_cases_demo(
    sample_query_row["filepath"],
    k=3,
    save_path=os.path.join(Config.REPORTS_DIR, "similar_cases_demo.png"),
)

In [ ]:
# ============================================================
# SECTION 9.4b — VALIDATION RETRIEVAL COMPARISON
# Compare learned-embedding neighbors with random training references.
# Metrics measure dataset-label agreement, not clinical usefulness.
# ============================================================

RETRIEVAL_EVAL_PER_CLASS = 100
RETRIEVAL_RANDOM_REPEATS = 20
RETRIEVAL_K = 3

if not set(val_df["patient_id"].dropna()).isdisjoint(set(train_df["patient_id"].dropna())):
    raise AssertionError("Retrieval validation queries overlap training patients.")
if not set(val_df["duplicate_group"].dropna()).isdisjoint(set(train_df["duplicate_group"].dropna())):
    raise AssertionError("Retrieval validation queries overlap training duplicate groups.")

retrieval_eval_parts = []
for retrieval_class_id, class_rows in val_df.groupby("label", sort=True):
    retrieval_eval_parts.append(
        class_rows.sample(
            min(RETRIEVAL_EVAL_PER_CLASS, len(class_rows)),
            random_state=Config.SEED + int(retrieval_class_id),
        )
    )
retrieval_eval_df = pd.concat(retrieval_eval_parts).reset_index(drop=True)
retrieval_eval_dataset = _build_tuning_dataset(
    retrieval_eval_df,
    Config.IMG_SIZE,
    shuffle=False,
    augment=False,
)

query_embedding_batches = []
query_label_batches = []
for query_images, query_labels in retrieval_eval_dataset:
    query_features = embedding_extractor(query_images, training=False).numpy()
    query_features /= np.linalg.norm(query_features, axis=1, keepdims=True) + 1e-10
    query_embedding_batches.append(query_features)
    query_label_batches.append(np.argmax(query_labels.numpy(), axis=1))
query_embeddings = np.concatenate(query_embedding_batches, axis=0)
query_true_labels = np.concatenate(query_label_batches, axis=0).astype(int)
if len(query_true_labels) != len(retrieval_eval_df):
    raise ValueError("Retrieval query labels and validation rows are misaligned.")

retrieval_similarities = query_embeddings @ train_embeddings.T
nearest_indices = np.argpartition(
    retrieval_similarities,
    -RETRIEVAL_K,
    axis=1,
)[:, -RETRIEVAL_K:]
nearest_scores = np.take_along_axis(retrieval_similarities, nearest_indices, axis=1)
nearest_order = np.argsort(nearest_scores, axis=1)[:, ::-1]
nearest_indices = np.take_along_axis(nearest_indices, nearest_order, axis=1)
nearest_labels = train_labels_arr[nearest_indices]
embedding_top1_agreement = nearest_labels[:, 0] == query_true_labels
embedding_top3_hit = np.any(nearest_labels == query_true_labels[:, np.newaxis], axis=1)
embedding_top3_grade_error = np.min(
    np.abs(nearest_labels - query_true_labels[:, np.newaxis]), axis=1
)

random_generator = np.random.default_rng(Config.SEED)
random_top1_agreement_repeats = []
random_top3_hit_repeats = []
random_top3_grade_error_repeats = []
random_top3_hit_by_stage_repeats = []
for _ in range(RETRIEVAL_RANDOM_REPEATS):
    random_indices = np.stack([
        random_generator.choice(len(train_labels_arr), size=RETRIEVAL_K, replace=False)
        for _ in range(len(query_true_labels))
    ])
    random_labels = train_labels_arr[random_indices]
    random_top1_agreement_repeats.append(random_labels[:, 0] == query_true_labels)
    random_top3_hit_repeats.append(
        np.any(random_labels == query_true_labels[:, np.newaxis], axis=1)
    )
    random_top3_grade_error_repeats.append(
        np.min(np.abs(random_labels - query_true_labels[:, np.newaxis]), axis=1)
    )
    random_top3_hit_by_stage_repeats.append([
        np.mean(
            np.any(random_labels[query_true_labels == stage] == stage, axis=1)
        )
        for stage in range(Config.NUM_CLASSES)
    ])

random_top1_agreement = np.asarray(random_top1_agreement_repeats)
random_top3_hit = np.asarray(random_top3_hit_repeats)
random_top3_grade_error = np.asarray(random_top3_grade_error_repeats)
retrieval_overall_comparison = pd.DataFrame([
    {
        "method": "Embedding nearest neighbors",
        "queries": len(query_true_labels),
        "top1_dataset_label_agreement": float(np.mean(embedding_top1_agreement)),
        "top3_dataset_label_hit_rate": float(np.mean(embedding_top3_hit)),
        "top3_best_absolute_grade_error": float(np.mean(embedding_top3_grade_error)),
        "random_repeat_sd": np.nan,
    },
    {
        "method": f"Random references (mean of {RETRIEVAL_RANDOM_REPEATS} repeats)",
        "queries": len(query_true_labels),
        "top1_dataset_label_agreement": float(np.mean(random_top1_agreement)),
        "top3_dataset_label_hit_rate": float(np.mean(random_top3_hit)),
        "top3_best_absolute_grade_error": float(np.mean(random_top3_grade_error)),
        "random_repeat_sd": float(np.std(np.mean(random_top3_hit, axis=1), ddof=1)),
    },
])
retrieval_stage_records = []
random_stage_hits = np.asarray(random_top3_hit_by_stage_repeats)
for stage_id, stage_name in enumerate(Config.CLASS_NAMES):
    stage_queries = query_true_labels == stage_id
    retrieval_stage_records.extend([
        {
            "class_id": stage_id,
            "class_name": stage_name,
            "method": "Embedding nearest neighbors",
            "queries": int(stage_queries.sum()),
            "top3_dataset_label_hit_rate": float(np.mean(embedding_top3_hit[stage_queries])),
        },
        {
            "class_id": stage_id,
            "class_name": stage_name,
            "method": "Random references",
            "queries": int(stage_queries.sum()),
            "top3_dataset_label_hit_rate": float(np.mean(random_stage_hits[:, stage_id])),
        },
    ])
retrieval_by_stage_comparison = pd.DataFrame(retrieval_stage_records)
retrieval_overall_comparison.to_csv(
    Path(Config.REPORTS_DIR) / "retrieval_validation_metrics.csv", index=False
)
retrieval_by_stage_comparison.to_csv(
    Path(Config.REPORTS_DIR) / "retrieval_validation_by_stage.csv", index=False
)
display(retrieval_overall_comparison.round(4))
display(retrieval_by_stage_comparison.pivot(
    index=["class_id", "class_name", "queries"],
    columns="method",
    values="top3_dataset_label_hit_rate",
).round(4))

retrieval_plot_data = retrieval_by_stage_comparison.pivot(
    index="class_name", columns="method", values="top3_dataset_label_hit_rate"
).reindex(Config.CLASS_NAMES)
fig, ax = plt.subplots(figsize=(11, 6))
retrieval_plot_data.plot(
    kind="bar",
    ax=ax,
    color=["#367C8A", "#D8784B"],
    width=0.78,
)
ax.set_title("Validation Top-3 Dataset-Label Match Rate: Embeddings vs Random References")
ax.set_xlabel("Validation query grade")
ax.set_ylabel("Fraction with at least one matching dataset label")
ax.set_ylim(0, 1)
ax.legend(title="Reference selection")
ax.grid(axis="y", alpha=0.25)
ax.tick_params(axis="x", rotation=18)
fig.tight_layout()
retrieval_comparison_plot_path = Path(Config.REPORTS_DIR) / "retrieval_validation_comparison.png"
fig.savefig(retrieval_comparison_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print("[Retrieval Evaluation] Dataset-label agreement is not evidence of clinical equivalence or improved diagnosis.")
print(f"[Retrieval Evaluation] Metrics saved -> {Config.REPORTS_DIR}/retrieval_validation_metrics.csv")
print(f"[Retrieval Evaluation] Plot saved -> {retrieval_comparison_plot_path}")

In [ ]:
# ============================================================
# SECTION 9.5 — SECTION SUMMARY
# ============================================================

print("=" * 60)
print("  SECTION 9 COMPLETE — Innovation Feature A")
print("=" * 60)
print("  build_embedding_extractor()           : 256-D latent projection")
print("  extract_and_cache_training_embeddings(): cached to embeddings.npz")
print("  find_similar_cases(query_img, k=3)    : cosine-metric CBR engine")
print(f"  Visual demonstration saved            : {Config.REPORTS_DIR}/similar_cases_demo.png")
print("  Next -> Innovation B: Multi-Agent Clinical Decision Pipeline")
print("=" * 60)

## Section 10: Innovation Feature B — Multi-Agent Clinical Decision Pipeline
*(Grounded in Coursework Material: `6. Multi-Agent_AI_Blueprint.pdf` & `7/8. Defense_Multi_Agent_LLM.ipynb`)*

**Goal:** Build a robust, safety-critical multi-agent clinical decision support (CDS) system
composed of four specialized, decoupled agents coordinated by a strict governance layer.

```
                  ┌─────────────────────────────────────┐
                  │          Query Fundus Image         │
                  └──────────────────┬──────────────────┘
                                     │
                                     ▼
                  ┌─────────────────────────────────────┐
                  │           DiagnosisAgent            │
                  │   - EfficientNetB3 CNN Inference    │
                  │   - Stage & Confidence Prediction   │
                  └─────────┬─────────────────┬─────────┘
                            │                 │
              ┌─────────────┘                 └─────────────┐
              ▼                                             ▼
┌─────────────────────────────┐               ┌─────────────────────────────┐
│     ExplainabilityAgent     │               │        AdvisoryAgent        │
│   - Grad-CAM Heatmap        │               │   - Protocol Guidance       │
│   - Quadrant Description    │               │   - Clinical Next Steps     │
│   - Similar-Case Retrieval  │               │   - Medical Disclaimer      │
└─────────────┬───────────────┘               └─────────────┬───────────────┘
              │                                             │
              └─────────────────────┬───────────────────────┘
                                    │
                                    ▼
                  ┌─────────────────────────────────────┐
                  │           GovernanceAgent           │
                  │   - Safety Gating Layer             │
                  │   - Is Confidence >= Threshold?     │
                  └─────────┬─────────────────┬─────────┘
                            │                 │
             NO (Conf < 70%)│                 │YES (Conf >= 70%)
                            ▼                 ▼
          ┌───────────────────────────┐     ┌───────────────────────────┐
          │ OVERRIDE OUTPUT:          │     │ RELEASE OUTPUT:           │
          │ "FLAGGED FOR HUMAN REVIEW"│     │ Validated Diagnosis,      │
          │ Guidance withheld         │     │ Guidance & Explanations   │
          └───────────────────────────┘     └───────────────────────────┘
```

### Architectural & Safety Rationale in a Clinical Context

1. **Why not a monolithic chatbot or single model wrapper?**
   - Single-model wrappers conflate perceptual classification with clinical protocol and risk management.
   - If an unconstrained model outputs both a probability and unstructured text advice, hallucination
     or low-confidence guesswork can be presented as medical guidance with disastrous patient consequences.

2. **Specialized Decoupled Responsibilities:**
   - **`DiagnosisAgent`** has one isolated role: perceptual classification.
   - **`ExplainabilityAgent`** provides verifiable evidentiary backing (Grad-CAM spatial localization +
     historical case retrieval).
   - **`AdvisoryAgent`** references validated clinical protocols (NHS/AAO staging guidelines), ensuring
     recommendations strictly adhere to evidence-based medical consensus.
   - **`GovernanceAgent`** functions as an **autonomous safety officer**: it intercepts predictions and has
     the authority to veto automated recommendations when the model's epistemic uncertainty is too high.

In [ ]:
# ============================================================
# SECTION 10.1 — AGENT 1: DiagnosisAgent
#
# Runs the trained EfficientNetB3 CNN model on an input image,
# extracts softmax probabilities, and returns the discrete predicted
# stage, stage name, and confidence score.
# ============================================================

class DiagnosisAgent:
    """
    Agent responsible for perceptual classification and disease staging.

    Evaluates fundus photographs through the trained EfficientNetB3 network
    and extracts class probabilities across all 5 ICDR stages.
    """

    def __init__(self, trained_model: keras.Model):
        """
        Initialize with the trained DR detection model.

        Args:
            trained_model: Trained Keras model outputting 5-class softmax distribution.
        """
        self.model = trained_model
        self.class_names = Config.CLASS_NAMES

    def process(self, image: Union[str, np.ndarray]) -> Dict[str, Any]:
        """
        Run inference and return structured diagnosis outputs.

        Args:
            image: Image filepath string or preprocessed RGB array of shape (Config.IMG_SIZE, Config.IMG_SIZE, 3).

        Returns:
            Dictionary containing:
                - 'predicted_stage': Integer ICDR stage (0 to 4).
                - 'stage_name': Human-readable stage string.
                - 'confidence': Float confidence score in [0.0, 1.0].
                - 'probabilities': Dict mapping each stage name to its softmax probability.
                - 'preprocessed_image': Normalized float32 image array (Config.IMG_SIZE, Config.IMG_SIZE, 3).
        """
        # Preprocess if path string is provided
        if isinstance(image, str):
            preproc_img = preprocess_image(image)
        else:
            preproc_img = image.copy()

        # Add batch dimension for inference
        if preproc_img.ndim == 3:
            tensor_in = np.expand_dims(preproc_img, axis=0)
        else:
            tensor_in = preproc_img

        # Execute model forward pass
        raw_probs = self.model(tensor_in, training=False).numpy()[0]

        pred_stage = int(np.argmax(raw_probs))
        confidence = float(raw_probs[pred_stage])

        stage_probs = {
            self.class_names[i]: float(raw_probs[i])
            for i in range(len(self.class_names))
        }

        return {
            "predicted_stage": pred_stage,
            "stage_name": self.class_names[pred_stage],
            "confidence": confidence,
            "probabilities": stage_probs,
            "preprocessed_image": preproc_img,
        }


# Instantiate DiagnosisAgent
diagnosis_agent = DiagnosisAgent(model)
print("[DiagnosisAgent] Initialized and verified.")

In [ ]:
# ============================================================
# SECTION 10.2 — AGENT 2: ExplainabilityAgent
#
# Generates dual-channel explainability:
#   1. Intra-image Grad-CAM heatmap & anatomical quadrant description
#   2. Inter-image Similar-Case Retrieval (Innovation A)
# ============================================================

class ExplainabilityAgent:
    """
    Agent responsible for clinical interpretability and evidentiary backing.

    Combines spatial Grad-CAM saliency with historical case-based retrieval
    to substantiate the automated diagnosis.
    """

    def __init__(
        self,
        cam_model: keras.Model = gradcam_model,
        extractor: keras.Model = embedding_extractor,
        stored_embeddings: np.ndarray = train_embeddings,
        stored_labels: np.ndarray = train_labels_arr,
        stored_filepaths: List[str] = train_fps,
    ):
        """
        Initialize with Grad-CAM model and retrieval components.
        """
        self.cam_model = cam_model
        self.extractor = extractor
        self.stored_embeddings = stored_embeddings
        self.stored_labels = stored_labels
        self.stored_filepaths = stored_filepaths

    def process(
        self,
        image_input: Union[str, np.ndarray],
        diagnosis_result: Dict[str, Any],
        k_similar: int = 3,
    ) -> Dict[str, Any]:
        """
        Generate comprehensive spatial and comparative explanations.

        Args:
            image_input: Image filepath or preprocessed array.
            diagnosis_result: Output dict from DiagnosisAgent.process().
            k_similar: Number of nearest cases to retrieve.

        Returns:
            Dictionary containing:
                - 'gradcam_heatmap': 2D float array normalized to [0, 1].
                - 'overlay_image': RGB uint8 array with superimposed Jet heatmap.
                - 'quadrant_explanation': Natural language description of peak lesions.
                - 'similar_cases': List of k retrieval metadata dicts.
        """
        preproc_img = diagnosis_result["preprocessed_image"]
        stage = diagnosis_result["predicted_stage"]
        conf = diagnosis_result["confidence"]

        # 1. Compute Grad-CAM heatmap
        heatmap, _, _ = compute_gradcam_heatmap(
            preproc_img,
            cam_model=self.cam_model,
            pred_index=stage,
        )

        # 2. Superimpose heatmap onto preprocessed image
        overlay = superimpose_gradcam(preproc_img, heatmap, alpha=0.4)

        # 3. Generate natural-language quadrant description
        quadrant_text = generate_quadrant_explanation(heatmap, stage, conf)

        # 4. Perform case-based retrieval
        similar_cases = find_similar_cases(
            preproc_img,
            k=k_similar,
            extractor=self.extractor,
            stored_embeddings=self.stored_embeddings,
            stored_labels=self.stored_labels,
            stored_filepaths=self.stored_filepaths,
        )

        return {
            "gradcam_heatmap": heatmap,
            "overlay_image": overlay,
            "quadrant_explanation": quadrant_text,
            "similar_cases": similar_cases,
        }


# Instantiate ExplainabilityAgent
explainability_agent = ExplainabilityAgent()
print("[ExplainabilityAgent] Initialized and verified.")

In [ ]:
# ============================================================
# SECTION 10.3 — AGENT 3: AdvisoryAgent
#
# Maps predicted disease stages to standard clinical protocols
# (follow-up intervals, specialist referrals, diagnostic testing)
# based on international ophthalmology guidelines (AAO / NHS).
# Appends a mandatory medical disclaimer.
# ============================================================

class AdvisoryAgent:
    """
    Agent responsible for clinical protocol alignment and next-step recommendations.

    Translates discrete DR stages into evidence-based action plans and attaches
    mandatory regulatory disclaimers.
    """

    # Clinical protocol lookup adhering to American Academy of Ophthalmology (AAO) guidelines
    CLINICAL_PROTOCOLS: Dict[int, Dict[str, str]] = {
        0: {
            "clinical_urgency": "Routine / Annual",
            "action_plan": (
                "No diabetic microvascular lesions observed. Continue routine annual diabetic retinopathy "
                "surveillance. Reinforce systemic glycemic control (HbA1c target < 7.0%), blood pressure "
                "management, and lipid monitoring in primary care."
            ),
            "follow_up": "Annual fundus photography examination (12 months).",
        },
        1: {
            "clinical_urgency": "Non-Urgent / Close Monitoring",
            "action_plan": (
                "Mild non-proliferative changes detected (microaneurysms only). Primary care optimization "
                "of blood glucose, blood pressure, and renal profile. Assess for potential subclinical diabetic "
                "macular edema if visual acuity is reduced."
            ),
            "follow_up": "Repeat dilated fundus examination within 6 to 9 months.",
        },
        2: {
            "clinical_urgency": "Semi-Urgent / Retinal Specialist Referral",
            "action_plan": (
                "Moderate non-proliferative changes present (microaneurysms, dot-and-blot hemorrhages, hard "
                "exudates, or cotton wool spots). High clinical risk of disease progression. Schedule formal "
                "optical coherence tomography (OCT) and comprehensive biomicroscopic dilated evaluation."
            ),
            "follow_up": "Refer to an ophthalmologist for comprehensive evaluation within 3 to 6 months.",
        },
        3: {
            "clinical_urgency": "Urgent Specialist Intervention",
            "action_plan": (
                "Severe non-proliferative retinopathy detected (meeting the '4-2-1 rule': severe intraretinal "
                "hemorrhages in all 4 quadrants, venous beading in 2+ quadrants, or IRMA in 1+ quadrant). "
                "Imminent risk of progression to proliferative DR (50% risk within 1 year). Urgent vitreoretinal "
                "assessment for panretinal photocoagulation (PRP) or anti-VEGF injection readiness."
            ),
            "follow_up": "Urgent ophthalmology referral within 2 to 4 weeks.",
        },
        4: {
            "clinical_urgency": "Emergent / Sight-Threatening",
            "action_plan": (
                "Proliferative Diabetic Retinopathy (PDR) identified. Presence of neovascularization, "
                "vitreous/preretinal hemorrhage, or fibrovascular proliferation posing immediate risk of tractional "
                "retinal detachment and irreversible vision loss. Requires immediate specialist intervention "
                "(panretinal photocoagulation, anti-VEGF therapy, or vitrectomy)."
            ),
            "follow_up": "Emergent referral to a vitreoretinal subspecialist within 24 to 72 hours.",
        },
    }

    DISCLAIMER: str = (
        "IMPORTANT REGULATORY NOTICE: This automated advisory summary is generated by an artificial "
        "intelligence clinical decision-support research system. It does NOT constitute a confirmed medical "
        "diagnosis, prescription, or therapeutic mandate. All findings must be clinically corroborated by a "
        "licensed medical professional (ophthalmologist, optometrist, or retinal specialist)."
    )

    def process(self, stage: int) -> Dict[str, str]:
        """
        Retrieve evidence-based guidance for the diagnosed DR stage.

        Args:
            stage: Integer ICDR DR stage (0 to 4).

        Returns:
            Dictionary containing 'clinical_urgency', 'action_plan', 'follow_up', and 'disclaimer'.
        """
        protocol = self.CLINICAL_PROTOCOLS.get(
            stage,
            {
                "clinical_urgency": "Unknown Stage",
                "action_plan": "Unrecognized disease stage. Manual clinical review required.",
                "follow_up": "Immediate manual examination.",
            },
        )

        return {
            "clinical_urgency": protocol["clinical_urgency"],
            "action_plan": protocol["action_plan"],
            "follow_up": protocol["follow_up"],
            "disclaimer": self.DISCLAIMER,
        }


# Instantiate AdvisoryAgent
advisory_agent = AdvisoryAgent()
print("[AdvisoryAgent] Initialized and verified.")

In [ ]:
# ============================================================
# SECTION 10.4 — AGENT 4: GovernanceAgent & RUN_PIPELINE
#
# Safety Gating Layer:
# If DiagnosisAgent confidence is below CONFIDENCE_THRESHOLD (default 70%):
#   - Actively OVERRIDES the output
#   - Sets 'flagged_for_review': True
#   - Withholds automated action plans
#   - Returns an urgent human review triage notice
#
# run_pipeline(image) orchestrates all 4 agents in sequence.
# ============================================================

class GovernanceAgent:
    """
    Safety and regulatory oversight agent.

    Monitors the outputs of upstream agents and enforces clinical risk thresholds.
    Actively gates and overrides recommendations when model uncertainty is elevated.
    """

    def __init__(self, confidence_threshold: float = Config.CONFIDENCE_THRESHOLD):
        """
        Initialize with configurable safety threshold.

        Args:
            confidence_threshold: Minimum confidence [0.0, 1.0] required to release
                                  automated guidance. Default 0.70 (70%).
        """
        self.confidence_threshold = confidence_threshold

    def evaluate(
        self,
        diagnosis: Dict[str, Any],
        explanation: Dict[str, Any],
        advisory: Dict[str, str],
    ) -> Dict[str, Any]:
        """
        Evaluate upstream agent outputs and enforce safety gating.

        Args:
            diagnosis: Output from DiagnosisAgent.
            explanation: Output from ExplainabilityAgent.
            advisory: Output from AdvisoryAgent.

        Returns:
            Unified structured clinical result dictionary.
        """
        conf = diagnosis["confidence"]
        stage = diagnosis["predicted_stage"]
        stage_name = diagnosis["stage_name"]

        # Check safety threshold
        if conf < self.confidence_threshold:
            flagged = True
            governance_status = "FLAGGED_FOR_HUMAN_REVIEW"
            gating_message = (
                f"SAFETY OVERRIDE ACTIVATED: Model confidence ({conf*100:.1f}%) is below the required "
                f"clinical safety threshold ({self.confidence_threshold*100:.0f}%). Automated recommendations "
                f"have been withheld to protect patient safety. This case has been routed to human ophthalmologist "
                f"triage for manual visual inspection."
            )
            # Override advisory guidance with safe triage protocol
            controlled_advisory = {
                "clinical_urgency": "Triage Required (Low AI Confidence)",
                "action_plan": gating_message,
                "follow_up": "Do NOT initiate treatment based on AI output. Perform manual dilated examination.",
                "disclaimer": advisory["disclaimer"],
            }
        else:
            flagged = False
            governance_status = "AUTOMATED_RECOMMENDATION_APPROVED"
            gating_message = (
                f"Safety check passed: Model confidence ({conf*100:.1f}%) meets or exceeds the required "
                f"safety threshold ({self.confidence_threshold*100:.0f}%)."
            )
            controlled_advisory = advisory

        return {
            "flagged_for_review": flagged,
            "governance_status": governance_status,
            "governance_message": gating_message,
            "diagnosis": {
                "predicted_stage": stage,
                "stage_name": stage_name,
                "confidence": conf,
                "probabilities": diagnosis["probabilities"],
            },
            "explanation": {
                "gradcam_heatmap": explanation["gradcam_heatmap"],
                "overlay_image": explanation["overlay_image"],
                "quadrant_explanation": explanation["quadrant_explanation"],
                "similar_cases": explanation["similar_cases"],
            },
            "advisory": controlled_advisory,
        }


# Instantiate GovernanceAgent
governance_agent = GovernanceAgent(confidence_threshold=Config.CONFIDENCE_THRESHOLD)


def run_pipeline(
    image_input: Union[str, np.ndarray],
    diag_agent: DiagnosisAgent = diagnosis_agent,
    expl_agent: ExplainabilityAgent = explainability_agent,
    adv_agent: AdvisoryAgent = advisory_agent,
    gov_agent: GovernanceAgent = governance_agent,
) -> Dict[str, Any]:
    """
    Execute the multi-agent clinical decision pipeline on a patient fundus image.

    Chains all 4 specialist agents in strict sequence:
        DiagnosisAgent -> ExplainabilityAgent -> AdvisoryAgent -> GovernanceAgent (Gate)

    Args:
        image_input: Absolute file path or preprocessed RGB image array (Config.IMG_SIZE, Config.IMG_SIZE, 3).
        diag_agent: Initialized DiagnosisAgent instance.
        expl_agent: Initialized ExplainabilityAgent instance.
        adv_agent: Initialized AdvisoryAgent instance.
        gov_agent: Initialized GovernanceAgent instance.

    Returns:
        Structured result dictionary with diagnosis, explainability, advisory,
        and safety gating status.
    """
    # 1. Run perception / diagnosis
    diag_result = diag_agent.process(image_input)

    # 2. Run explainability & case retrieval
    expl_result = expl_agent.process(image_input, diag_result, k_similar=3)

    # 3. Formulate standard clinical advisory
    advis_result = adv_agent.process(diag_result["predicted_stage"])

    # 4. Enforce safety gating & governance
    final_result = gov_agent.evaluate(diag_result, expl_result, advis_result)

    return final_result


print("[Multi-Agent Pipeline] run_pipeline(image) compiled and ready.")

In [ ]:
# ============================================================
# SECTION 10.5 — VALIDATION EXAMPLES FOR MULTI-AGENT PIPELINE  # CHANGED: avoid post-evaluation test reuse
#
# Case 1: Standard pipeline flow
# Case 2: Safety-gate threshold demonstration
# ============================================================

def display_pipeline_result(result: Dict[str, Any], query_title: str) -> None:
    """
    Format and print a structured report of the multi-agent pipeline outcome.
    """
    print("\n" + "=" * 80)
    print(f"  MULTI-AGENT DECISION PIPELINE REPORT: {query_title.upper()}")
    print("=" * 80)

    diag = result["diagnosis"]
    gov  = result["governance_status"]
    flag = result["flagged_for_review"]
    adv  = result["advisory"]
    expl = result["explanation"]

    print(f"  Diagnosis Result        : {diag['stage_name']} (Stage {diag['predicted_stage']})")
    print(f"  Model Confidence        : {diag['confidence']*100:.2f}%")
    print(f"  Governance Status       : {gov}")
    print(f"  Flagged for Review?     : {'YES - HUMAN REVIEW MANDATORY' if flag else 'NO - AUTOMATION APPROVED'}")
    print(f"  Safety Gate Details     : {result['governance_message']}")
    print("-" * 80)
    print(f"  Clinical Urgency        : {adv['clinical_urgency']}")
    print(f"  Recommended Action Plan : {adv['action_plan']}")
    print(f"  Follow-up Protocol      : {adv['follow_up']}")
    print("-" * 80)
    print(f"  Spatial Explanation     : {expl['quadrant_explanation']}")
    print("  Case-Based Evidence     : Retrieved top-3 matching historical training cases:")
    for c in expl["similar_cases"]:
        print(f"    - Rank {c['rank']}: Confirmed {c['stage_name']} (Similarity: {c['cosine_similarity']:.3f})")
    print("=" * 80 + "\n")


sample_fp = val_df.iloc[0]["filepath"]  # CHANGED: use validation image for qualitative demo
print("--> Running validation example: standard pipeline")  # CHANGED: identify validation use
res_standard = run_pipeline(sample_fp)
display_pipeline_result(res_standard, "Validation Example (Standard Threshold)")  # CHANGED: label source correctly

print("--> Running validation example: safety-gate demonstration")  # CHANGED: identify validation use
strict_gov = GovernanceAgent(confidence_threshold=0.999)  # intentionally high to trigger gate
res_gated = run_pipeline(sample_fp, gov_agent=strict_gov)
display_pipeline_result(res_gated, "Validation Example (High Safety Threshold)")  # CHANGED: label source correctly

In [ ]:
# ============================================================
# SECTION 10.6 — SECTION SUMMARY
# ============================================================

print("=" * 60)
print("  SECTION 10 COMPLETE — Innovation Feature B")
print("=" * 60)
print("  DiagnosisAgent     : CNN inference & 5-class stage/confidence outputs")
print("  ExplainabilityAgent: Grad-CAM heatmap + quadrant text + case retrieval")
print("  AdvisoryAgent      : AAO clinical protocol mapping + disclaimer")
print("  GovernanceAgent    : active safety gate (<70% confidence overrides output)")
print("  run_pipeline()     : unified multi-agent decision support function")
print("  Next -> Section 11: UI with Gradio & Bonus Features C, D, E")
print("=" * 60)

## Section 11: Optional Gradio Research Interface

The notebook builds an optional Gradio interface for image upload, five-stage probabilities, Grad-CAM visualization, advisory text, and a similar-case gallery. The gallery displays real training reference files with dataset labels when the paths are available. This is a coursework prototype, not a deployed or clinically validated service; launching with a public share URL is optional and is not needed for the assessment.

The interface demonstrates how the analysis components can be presented together. Its confidence threshold is a configurable softmax cutoff, not calibrated clinical uncertainty, and neither the interface nor its outputs should be used for diagnosis or treatment decisions.

In [ ]:
# ============================================================
# SECTION 11.1 — GRADIO INFERENCE WRAPPER FUNCTION
# Formats outputs from the optional research pipeline for Gradio.
# ============================================================


def gradio_predict(
    input_img: np.ndarray,
    confidence_threshold: float = 0.70,
) -> Tuple[str, str, np.ndarray, str, List[Tuple[np.ndarray, str]]]:
    """Format prototype outputs for the Gradio interface."""
    if input_img is None:
        empty_img = np.zeros((Config.IMG_SIZE, Config.IMG_SIZE, 3), dtype=np.uint8)
        return (
            "### Please upload a retinal fundus photograph.",
            "",
            empty_img,
            "",
            [],
        )

    gov_agent = GovernanceAgent(confidence_threshold=confidence_threshold)
    preprocessed_input = preprocess_image(input_img, img_size=Config.IMG_SIZE)
    result = run_pipeline(preprocessed_input, gov_agent=gov_agent)
    diagnosis = result["diagnosis"]
    advisory = result["advisory"]
    explanation = result["explanation"]

    if result["flagged_for_review"]:
        status_md = (
            "### FLAGGED FOR HUMAN REVIEW\n"
            f"Model softmax confidence ({diagnosis['confidence']:.1%}) is below the configured "
            f"display threshold ({confidence_threshold:.0%}). This threshold is not calibrated clinical uncertainty."
        )
    else:
        status_md = (
            "### Research output only\n"
            f"Model softmax confidence: {diagnosis['confidence']:.1%}. A threshold pass does not establish safety."
        )

    probability_lines = [
        f"- **{class_name}:** {probability:.1%}"
        for class_name, probability in diagnosis["probabilities"].items()
    ]
    diagnosis_md = (
        f"## Predicted stage: {diagnosis['stage_name']} (ICDR {diagnosis['predicted_stage']})\n\n"
        "**Five-stage probability distribution:**\n"
        + "\n".join(probability_lines)
    )
    similar_gallery = []
    for case in explanation["similar_cases"]:
        if not os.path.isfile(case["filepath"]):
            print(f"[Gradio] Skipping unavailable reference image: {case['filepath']}")
            continue
        case_img = preprocess_image(case["filepath"], img_size=Config.IMG_SIZE)
        case_img_uint8 = (np.clip(case_img, 0.0, 1.0) * 255).astype(np.uint8)
        caption = (
            f"Rank #{case['rank']} | Dataset label: {case['stage_name']} "
            f"(Stage {case['label']})\nCosine similarity: {case['cosine_similarity']:.3f}"
        )
        similar_gallery.append((case_img_uint8, caption))

    advisory_md = (
        f"**Prototype advisory text:** {advisory['action_plan']}\n\n"
        f"**Follow-up text:** {advisory['follow_up']}\n\n"
        f"*{advisory['disclaimer']}*"
    )
    quadrant_text = (
        "**Grad-CAM saliency (not verified lesion localization):**\n"
        f"{explanation['quadrant_explanation']}"
    )
    return (
        status_md,
        diagnosis_md,
        explanation["overlay_image"],
        f"{quadrant_text}\n\n{advisory_md}",
        similar_gallery,
    )


print("[Gradio] Optional prediction wrapper defined; gallery entries use existing reference image files.")

In [ ]:
# ============================================================
# SECTION 11.2 — BUILD & LAUNCH THE GRADIO APP
#
# Constructs an ergonomic clinical decision support dashboard
# using Gradio Blocks and launches with share=True.
# ============================================================

import gradio as gr


def build_gradio_app() -> gr.Blocks:
    """
    Construct the Gradio clinical decision-support interface.
    """
    custom_theme = gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="blue",
    )

    with gr.Blocks(theme=custom_theme, title="Diabetic Retinopathy Clinical AI") as demo:
        gr.Markdown(
            "# 👁️ Diabetic Retinopathy Multi-Agent Decision Support System\n"
            "### *Deep Learning (EfficientNetB3) + Grad-CAM + Case-Based Reasoning + Safety Governance*\n"
            "Upload a fundus photograph to initiate automated multi-agent staging and explainability analysis."
        )

        with gr.Row():
            # Left Column: Inputs & Controls
            with gr.Column(scale=4):
                input_image = gr.Image(
                    label="Upload Fundus Photograph",
                    type="numpy",
                    sources=["upload", "clipboard"],
                )
                threshold_slider = gr.Slider(
                    minimum=0.50,
                    maximum=0.95,
                    value=0.70,
                    step=0.05,
                    label="Governance Safety Threshold (Default: 70%)",
                    info="Predictions below this confidence are gated and flagged for manual human review.",
                )
                analyze_btn = gr.Button("🔍 Run Multi-Agent Analysis", variant="primary", size="lg")

                # Validation examples keep the held-out test images out of the demo.  # CHANGED: preserve test holdout
                sample_examples = []
                for cls_id in range(Config.NUM_CLASSES):
                    sample_subset = val_df[val_df["label"] == cls_id]  # CHANGED: use validation split
                    if not sample_subset.empty:
                        sample_examples.append(sample_subset.iloc[0]["filepath"])

                if sample_examples:
                    gr.Examples(
                        examples=sample_examples[:5],
                        inputs=input_image,
                        label="Validation Sample Cases (Stages 0 to 4)",  # CHANGED: label source
                    )

            # Right Column: Diagnostic & Governance Outputs
            with gr.Column(scale=6):
                status_box = gr.Markdown("### Upload an image and click 'Run Multi-Agent Analysis'")
                diagnosis_box = gr.Markdown()

        gr.Markdown("---")
        gr.Markdown("## 🔬 Multi-Agent Clinical Explainability & Evidence Dossier")

        with gr.Row():
            with gr.Column(scale=5):
                gr.Markdown("### **1. Spatial Explainability: Grad-CAM Lesion Heatmap**")
                overlay_output = gr.Image(label="Superimposed Retinal Lesion Heatmap", type="numpy")

            with gr.Column(scale=5):
                gr.Markdown("### **2. Comparative Evidence: Similar Verified Historical Cases**")
                gallery_output = gr.Gallery(
                    label="Top-3 Nearest Verified Training Cases (CBR)",
                    columns=3,
                    rows=1,
                    height=280,
                    object_fit="contain",
                )

        gr.Markdown("---")
        with gr.Row():
            advisory_box = gr.Markdown()

        analyze_btn.click(
            fn=gradio_predict,
            inputs=[input_image, threshold_slider],
            outputs=[status_box, diagnosis_box, overlay_output, advisory_box, gallery_output],
        )

    return demo


gradio_app = build_gradio_app()
print("[Gradio] App constructed successfully.")
print("[Launch Note] Call 'gradio_app.launch(share=True)' to launch the interface with a public video-demo link.")

In [ ]:
# ============================================================
# SECTION 11.3 — OPTIONAL GRADIO APP LAUNCH
# This cell does not launch the server during Run All.
# Run one of the launch commands below manually when needed.
# ============================================================

print("[Gradio] App is ready. Launch it manually if you want to use the UI.")

# Local-only access from this notebook environment:
# gradio_app.launch(share=False, debug=False, show_error=True)

# Optional: creates a publicly accessible Gradio URL. Use only when needed.
# gradio_app.launch(share=True, debug=False, show_error=True)

In [ ]:
# ============================================================
# SECTION 11.4 — BONUS FEATURE E: MULTI-STAGE VERIFICATION
#
# Code audit cell confirming that the model operates as a true
# 5-stage classifier (ICDR Stages 0-4) rather than a collapsed
# binary model (DR present vs absent).
# ============================================================

def verify_multistage_classification_compliance(
    m: keras.Model,
    class_names: List[str] = Config.CLASS_NAMES,
) -> bool:
    """
    Formally verify and print multi-stage classification compliance.

    Coursework Criterion E:
    Explicitly confirm the final model outputs all 5 stages (not collapsed to DR/no-DR)
    and note that this satisfies stage-level (not just presence/absence) detection.
    """
    output_shape = m.output_shape
    n_outputs = output_shape[-1]

    print("\n" + "=" * 75)
    print("  BONUS FEATURE E COMPLIANCE: MULTI-STAGE CLASSIFICATION AUDIT")
    print("=" * 75)
    print(f"  Model Name                : {m.name}")
    print(f"  Final Softmax Output Shape: {output_shape} (Dimension: {n_outputs})")
    print(f"  Configured Class Names    : {class_names}")
    print(f"  Number of Discrete Stages : {len(class_names)}")
    print("-" * 75)

    is_compliant = (n_outputs == 5) and (len(class_names) == 5)

    if is_compliant:
        print("  [CONFIRMED] Multi-Stage Classification Requirement Fulfilled:")
        print("  - The model outputs distinct probabilities for all 5 international ICDR stages:")
        print("      Stage 0: No DR")
        print("      Stage 1: Mild NPDR")
        print("      Stage 2: Moderate NPDR")
        print("      Stage 3: Severe NPDR")
        print("      Stage 4: Proliferative DR")
        print("  - The architecture does NOT collapse outputs to binary (DR present / absent).")
        print("  - It satisfies granular, disease-severity clinical grading standards.")
    else:
        print("  [ERROR] Model output dimension does not match 5 classes!")

    print("=" * 75 + "\n")
    return is_compliant


multistage_compliant = verify_multistage_classification_compliance(model)

In [ ]:
# ============================================================
# SECTION 11.5 — END-TO-END NOTEBOOK COMPLETION SUMMARY
# ============================================================

print("=" * 70)
print("  END-TO-END PIPELINE COMPLETE: DIABETIC RETINOPATHY STAGE DETECTION")
print("=" * 70)
print("  1.  Setup & Data Acquisition      : APTOS 2019 (3,662 labelled images)")
print("  2.  Preprocessing                 : Crop + Ben Graham (CLAHE/denoise/edges in gallery + ablation)")
print("  3.  Augmentation & Balancing      : Train-only augmentation; class weights tested in a pilot")
print("  4.  Train/Validation/Test         : Duplicate-aware stratified 70% / 15% / 15% split")
print(f"  5.  Model Architecture            : EfficientNetB3 + Dropout({Config.DROPOUT_RATE})")
print(f"  6.  Two-Phase Training            : top {Config.UNFREEZE_TOP_N} layers, Phase 2 LR={Config.PHASE2_LR}")
print("      Pilot screen                  : hyperparameters, ablations, ResNet50/MobileNetV2 comparison")
print("  7.  Evaluation                    : Accuracy, QWK, P/R/F1, confusion, ROC, DR/referable screening")
print("  8.  Explainability                : Grad-CAM heatmaps + anatomical quadrant text")
print("  9.  Innovation Feature A          : Penultimate embedding similar-case retrieval")
print("  10. Innovation Feature B          : 4-Agent Clinical Decision Pipeline + Safety Gate")
print("  11. Bonus Features C, D, E        : Gradio Web UI + HF Spaces + 5-Stage Multi-class")
print("  12. Exploratory U-Net             : Pseudo-mask lesion localisation demo")
print("  13. Research evidence             : Error analysis + reproducibility manifest")
print("=" * 70)

## Section 12: Exploratory Innovation — U-Net Pseudo-Mask Demonstration

**Purpose:** Explore whether an encoder-decoder U-Net can produce spatially detailed candidate regions alongside five-stage grading and Grad-CAM. The input fundus photographs are real dataset images, but the segmentation targets are synthetic heuristic pseudo-masks, not expert annotations.

### Rationale and Method

Small lesions can be difficult to inspect after image downsampling, while a U-Net's encoder-decoder structure and skip connections preserve spatial detail. This motivates the experiment, but does not establish that its output is a lesion mask. The notebook synthesizes targets from green-channel top-hat/black-hat morphology gated by a Grad-CAM threshold, then trains an auxiliary U-Net with a hybrid binary-cross-entropy and soft-Dice loss.

$$\mathcal{L}_{\text{hybrid}} = 0.5\,\mathcal{L}_{\text{BCE}} + 0.5\left(1 - \frac{2 \sum_i y_i \hat{y}_i + \epsilon}{\sum_i y_i + \sum_i \hat{y}_i + \epsilon}\right)$$

The generated target masks and U-Net overlays are **synthetic pseudo-mask demos**. Any Dice value measures agreement with those heuristic targets only. There are no expert pixel labels or independent segmentation results here, so the overlay is not validated lesion localization and must not be interpreted as clinical evidence. The visualization uses real validation fundus images and should be captioned as a pseudo-mask demonstration.

In [ ]:
# ============================================================
# SECTION 12.1 — SOFT DICE LOSS & METRICS
#
# Implements differentiable Soft Dice Loss and Dice Coefficient
# for extreme foreground-background imbalance in lesion masks.
# ============================================================

def dice_coefficient(y_true: tf.Tensor, y_pred: tf.Tensor, smooth: float = 1e-6) -> tf.Tensor:
    """
    Compute the Sorensen-Dice coefficient for binary segmentation masks.

    Dice = (2 * |X ∩ Y| + smooth) / (|X| + |Y| + smooth)

    Args:
        y_true: Ground truth binary tensor of shape (B, H, W, 1).
        y_pred: Predicted probability tensor of shape (B, H, W, 1) in [0, 1].
        smooth: Smoothing epsilon to prevent division by zero.

    Returns:
        Scalar Dice coefficient tensor in [0, 1].
    """
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    cardinality  = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f)
    return (2.0 * intersection + smooth) / (cardinality + smooth)


def dice_loss(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
    """
    Differentiable Soft Dice Loss: 1.0 - Dice Coefficient.
    """
    return 1.0 - dice_coefficient(y_true, y_pred)


def hybrid_dice_bce_loss(y_true: tf.Tensor, y_pred: tf.Tensor) -> tf.Tensor:
    """
    Hybrid loss combining Binary Cross-Entropy (pixel-level fidelity)
    with Soft Dice Loss (macro spatial overlap).

    Loss = 0.5 * BCE + 0.5 * Dice_Loss
    """
    bce = keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return 0.5 * tf.reduce_mean(bce) + 0.5 * dice


print("[U-Net] Custom Soft Dice Loss and Hybrid Loss compiled.")

In [ ]:
# ============================================================
# SECTION 12.1b — U-NET PRECISION POLICY
# The U-Net demo runs in float32: under the classifier's mixed_float16
# policy, XLA fails while compiling the U-Net gradients. The classifier
# is already built, so this only affects the layers created from here on.
# ============================================================

keras.mixed_precision.set_global_policy("float32")
print("[U-Net] Precision policy:", keras.mixed_precision.global_policy().name)


In [ ]:
# ============================================================
# SECTION 12.2 — AUXILIARY U-NET ARCHITECTURE
# Encoder-decoder network with skip connections, sized from Config.
# ============================================================

def build_unet(
    input_shape: Tuple[int, int, int] = (Config.IMG_SIZE, Config.IMG_SIZE, 3),
) -> keras.Model:
    """Construct an encoder-decoder network for auxiliary lesion masks."""
    inputs = keras.Input(shape=input_shape, name="unet_input")

    def conv_block(x, filters, name_prefix):
        x = layers.Conv2D(filters, (3, 3), padding="same", name=f"{name_prefix}_conv1")(x)
        x = layers.BatchNormalization(name=f"{name_prefix}_bn1")(x)
        x = layers.Activation("relu", name=f"{name_prefix}_relu1")(x)
        x = layers.Conv2D(filters, (3, 3), padding="same", name=f"{name_prefix}_conv2")(x)
        x = layers.BatchNormalization(name=f"{name_prefix}_bn2")(x)
        x = layers.Activation("relu", name=f"{name_prefix}_relu2")(x)
        return x

    c1 = conv_block(inputs, 32, "enc1")
    p1 = layers.MaxPooling2D((2, 2), name="pool1")(c1)
    c2 = conv_block(p1, 64, "enc2")
    p2 = layers.MaxPooling2D((2, 2), name="pool2")(c2)
    c3 = conv_block(p2, 128, "enc3")
    p3 = layers.MaxPooling2D((2, 2), name="pool3")(c3)
    bottleneck = conv_block(p3, 256, "bottleneck")

    up3 = layers.Conv2DTranspose(128, (2, 2), strides=(2, 2), padding="same", name="up3")(bottleneck)
    up3 = layers.Resizing(c3.shape[1], c3.shape[2], name="align_up3")(up3)
    dec3 = conv_block(layers.concatenate([up3, c3], name="concat3"), 128, "dec3")

    up2 = layers.Conv2DTranspose(64, (2, 2), strides=(2, 2), padding="same", name="up2")(dec3)
    up2 = layers.Resizing(c2.shape[1], c2.shape[2], name="align_up2")(up2)
    dec2 = conv_block(layers.concatenate([up2, c2], name="concat2"), 64, "dec2")

    up1 = layers.Conv2DTranspose(32, (2, 2), strides=(2, 2), padding="same", name="up1")(dec2)
    up1 = layers.Resizing(c1.shape[1], c1.shape[2], name="align_up1")(up1)
    dec1 = conv_block(layers.concatenate([up1, c1], name="concat1"), 32, "dec1")

    outputs = layers.Conv2D(1, (1, 1), activation="sigmoid", name="lesion_mask_output")(dec1)
    return keras.Model(inputs=inputs, outputs=outputs, name="Retinal_Lesion_UNet")


unet_model = build_unet()
unet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=hybrid_dice_bce_loss,
    metrics=[dice_coefficient, "binary_accuracy"],
)
print("[U-Net] Retinal Lesion U-Net constructed successfully.")
print(f"  Input resolution: {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"  Total parameters: {unet_model.count_params():,}")

In [ ]:
# ============================================================
# SECTION 12.3 — SEMI-SUPERVISED LESION MASK SYNTHESIZER
#
# Generates heuristic pseudo-masks from Grad-CAM and morphology.
# These are synthesized masks, not manually annotated ground truth.
# ============================================================

def synthesize_retinal_lesion_mask(
    preprocessed_img: np.ndarray,
    stage: int,
    cam_model: keras.Model = gradcam_model,
) -> np.ndarray:
    """Create a heuristic pseudo-mask with shape (IMG_SIZE, IMG_SIZE, 1)."""
    if stage == 0:
        return np.zeros((Config.IMG_SIZE, Config.IMG_SIZE, 1), dtype=np.float32)

    heatmap, _, _ = compute_gradcam_heatmap(
        preprocessed_img, cam_model=cam_model, pred_index=stage
    )
    green = (preprocessed_img[:, :, 1] * 255).astype(np.uint8)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    tophat = cv2.morphologyEx(green, cv2.MORPH_TOPHAT, kernel)
    blackhat = cv2.morphologyEx(green, cv2.MORPH_BLACKHAT, kernel)
    lesion_signals = cv2.add(tophat, blackhat)

    _, binary_lesions = cv2.threshold(lesion_signals, 25, 255, cv2.THRESH_BINARY)
    heatmap_full = cv2.resize(
        heatmap.astype(np.float32),
        (Config.IMG_SIZE, Config.IMG_SIZE),
        interpolation=cv2.INTER_LINEAR,
    )
    saliency_gate = (heatmap_full > 0.35).astype(np.uint8)
    gated_mask = cv2.bitwise_and(binary_lesions, binary_lesions, mask=saliency_gate)

    cleaned_mask = cv2.morphologyEx(gated_mask, cv2.MORPH_OPEN, kernel)
    return (cleaned_mask > 0).astype(np.float32)[..., np.newaxis]


print("[U-Net] Lesion pseudo-mask generator ready; masks are not clinical annotations.")

In [ ]:
# ============================================================
# SECTION 12.4 — TRAIN AUXILIARY U-NET ON SYNTHETIC PSEUDO-MASKS
# Targets are generated from image morphology and Grad-CAM, not annotations.
# ============================================================


def train_auxiliary_unet(
    unet: keras.Model,
    dataframe: pd.DataFrame,
    n_samples: int = 400,
    epochs: int = 5,
    batch_size: int = 16,
) -> keras.callbacks.History:
    """Train a demonstration U-Net against heuristic pseudo-mask targets."""
    sample_df = dataframe.sample(min(n_samples, len(dataframe)), random_state=Config.SEED)
    images = []
    masks = []

    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Creating synthetic pseudo-mask demo targets"):
        image = preprocess_image(row["filepath"])
        mask = synthesize_retinal_lesion_mask(image, int(row["label"]))
        images.append(image)
        masks.append(mask)

    x_train = np.asarray(images, dtype=np.float32)
    y_pseudo = np.asarray(masks, dtype=np.float32)
    print(
        f"[U-Net DEMO] Training on {len(x_train)} real images with synthetic pseudo-masks; "
        f"image shape={x_train.shape}, target shape={y_pseudo.shape}."
    )

    history = unet.fit(
        x_train,
        y_pseudo,
        batch_size=batch_size,
        epochs=epochs,
        validation_split=0.2,
        verbose=1,
    )
    print(
        "[U-Net DEMO] Pseudo-mask validation Dice is target self-consistency only, "
        f"not clinical lesion accuracy. Final Dice: {history.history['dice_coefficient'][-1]:.4f}"
    )
    return history


unet_history = train_auxiliary_unet(unet_model, train_df, n_samples=300, epochs=5)
unet_model.save_weights(os.path.join(Config.CHECKPOINT_DIR, "unet_pseudomask.weights.h5"))
print("[U-Net DEMO] Saved pseudo-mask demonstration checkpoint; no expert lesion ground truth was used.")

In [ ]:
# ============================================================
# SECTION 12.5 — PSEUDO-MASK TARGET VS U-NET OUTPUT DEMO
# Real validation photos; synthetic heuristic targets and unverified predictions.
# ============================================================


def visualize_three_layer_explainability_stack(
    validation_dataframe: pd.DataFrame,
    cls_model: keras.Model = model,
    seg_model: keras.Model = unet_model,
    save_path: Optional[str] = None,
) -> None:
    """Compare real validation images with synthetic targets and U-Net demo outputs."""
    selected_samples = []
    for stage in [1, 2, 3, 4]:
        subset = validation_dataframe[validation_dataframe["label"] == stage]
        if not subset.empty:
            selected_samples.append(subset.iloc[0])

    if not selected_samples:
        raise ValueError("No validation examples are available for the pseudo-mask demonstration.")

    n_rows = len(selected_samples)
    fig, axes = plt.subplots(n_rows, 6, figsize=(25, 4.5 * n_rows), squeeze=False)
    col_headers = [
        "Real validation fundus",
        "Synthetic heuristic target",
        "Five-stage classifier output",
        "Grad-CAM saliency",
        "U-Net probability (unverified)",
        "U-Net overlay demo",
    ]
    for column, title in enumerate(col_headers):
        axes[0, column].set_title(title, fontsize=10, fontweight="bold", pad=12)

    for row_index, row in enumerate(selected_samples):
        image = preprocess_image(row["filepath"])
        true_stage = int(row["label"])
        probabilities = cls_model(image[np.newaxis, ...], training=False).numpy()[0]
        predicted_stage = int(np.argmax(probabilities))
        confidence = float(probabilities[predicted_stage])
        heatmap, _, _ = compute_gradcam_heatmap(
            image, cam_model=gradcam_model, pred_index=predicted_stage
        )
        cam_overlay = superimpose_gradcam(image, heatmap, alpha=0.45)
        pseudo_target = synthesize_retinal_lesion_mask(image, true_stage)[..., 0]
        predicted_mask = seg_model(image[np.newaxis, ...], training=False).numpy()[0, :, :, 0]
        binary_mask = (predicted_mask > 0.4).astype(np.uint8)

        base_image = (image * 255).astype(np.uint8)
        pseudo_overlay = base_image.copy()
        pseudo_overlay[binary_mask == 1] = [0, 255, 64]
        pseudo_overlay = cv2.addWeighted(pseudo_overlay, 0.65, base_image, 0.35, 0)

        axes[row_index, 0].imshow(image)
        axes[row_index, 0].axis("off")
        axes[row_index, 0].set_ylabel(
            f"Dataset label: {Config.CLASS_NAMES[true_stage]}\n(Stage {true_stage})",
            fontsize=9, rotation=0, labelpad=75, va="center",
        )
        axes[row_index, 1].imshow(pseudo_target, cmap="gray", vmin=0, vmax=1)
        axes[row_index, 1].axis("off")
        bar_colors = ["#4A90E2" if i != predicted_stage else "#D0021B" for i in range(5)]
        axes[row_index, 2].barh(
            Config.CLASS_NAMES, probabilities, color=bar_colors, edgecolor="black", linewidth=0.6
        )
        axes[row_index, 2].set_xlim(0, 1.0)
        axes[row_index, 2].set_xlabel("Probability")
        axes[row_index, 2].text(
            0.5, 0.85,
            f"Predicted: {Config.CLASS_NAMES[predicted_stage]}\nConfidence: {confidence:.1%}",
            transform=axes[row_index, 2].transAxes,
            bbox=dict(boxstyle="round,pad=0.3", fc="#FFF9C4", ec="#FBC02D"),
            fontsize=9, fontweight="bold",
        )
        axes[row_index, 3].imshow(cam_overlay)
        axes[row_index, 3].axis("off")
        axes[row_index, 4].imshow(predicted_mask, cmap="gray", vmin=0, vmax=1)
        axes[row_index, 4].axis("off")
        axes[row_index, 5].imshow(pseudo_overlay)
        axes[row_index, 5].axis("off")

    fig.suptitle(
        "Exploratory Demo — Synthetic Pseudo-Targets vs U-Net Output (Not Clinical Segmentation)",
        fontsize=12,
        fontweight="bold",
    )
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"[Pseudo-mask DEMO] Saved target/output comparison -> {save_path}")
    plt.show()


visualize_three_layer_explainability_stack(
    val_df,
    save_path=os.path.join(Config.REPORTS_DIR, "three_layer_explainability_stack.png"),
)

In [ ]:
print("=" * 70)
print("  SECTION 12 COMPLETE — Exploratory U-Net Pseudo-Mask Demonstration")
print("=" * 70)
print("  build_unet()                     : encoder-decoder with skip connections")
print("  hybrid_dice_bce_loss()           : BCE + soft Dice against pseudo-mask targets")
print("  synthesize_retinal_lesion_mask() : heuristic synthetic demo-target generator")
print("  train_auxiliary_unet()           : exploratory self-consistency training only")
print(f"  Artifact generated               : {Config.REPORTS_DIR}/three_layer_explainability_stack.png")
print("  The green overlay is not expert-annotated or clinically validated lesion segmentation.")
print("=" * 70)

## Section 13 — Research Evidence: Error Analysis and Reproducibility

This section is research-only and does not modify the deployed EfficientNetB3 checkpoint or the Gradio application. The model-level ablation and backbone comparison results are produced by the pilot runner in Section 6 (`comparison_trials_results.csv`, `pilot_comparison.png`). This section records the test-set error analysis (adjacent vs distant ordinal errors, high-confidence mistakes, low-confidence Governance cases) and a reproducibility manifest (library versions, hardware, seed, dataset, split files, hyperparameters and checkpoints).

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "core").is_dir():
    PROJECT_ROOT = Path(_PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.research_evidence import (
    ABLATION_VARIANTS,
    build_error_analysis,
    build_reproducibility_manifest,
    save_error_analysis,
    set_reproducible_seed,
)

set_reproducible_seed(getattr(Config, "SEED", 42))
REPORT_DIR = Path(getattr(Config, "REPORTS_DIR", "report_images"))
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, split_frame in (("train", train_df), ("validation", val_df), ("test", test_df)):
    split_frame.to_csv(REPORT_DIR / f"{split_name}_split.csv", index=False)

error_analysis = build_error_analysis(
    y_test_true,
    y_test_pred,
    probabilities=y_test_prob,
)
save_error_analysis(error_analysis, REPORT_DIR / "error_analysis")

reproducibility_manifest = build_reproducibility_manifest(
    config={
        "random_seed": getattr(Config, "SEED", 42),
        "image_size": [Config.IMG_SIZE, Config.IMG_SIZE],
        "batch_size": Config.BATCH_SIZE,
        "phase_1_learning_rate": Config.PHASE1_LR,
        "phase_2_learning_rate": Config.PHASE2_LR,
        "phase_1_max_epochs": Config.PHASE1_EPOCHS,
        "phase_2_max_epochs": Config.PHASE2_EPOCHS,
        "phase_2_unfreeze_top_n": Config.UNFREEZE_TOP_N,
        "dropout_rate": Config.DROPOUT_RATE,
        "class_weights_enabled": Config.USE_CLASS_WEIGHTS,
        "early_stopping_monitor": Config.STOP_MONITOR,
        "early_stopping_mode": Config.STOP_MODE,
        "early_stopping_patience": Config.STOP_PATIENCE,
        "oversampling_enabled": Config.USE_OVERSAMPLING,
        "tta_selected_on_validation": USE_TTA_FOR_EVALUATION,
        "qwk_thresholds_selected_on_validation": np.round(QWK_THRESHOLDS, 6).tolist(),
        "test_predictions": "ordinal_expected_grade_thresholds",
        "confidence_threshold": Config.CONFIDENCE_THRESHOLD,
        "model": "EfficientNetB3",
    },
    output_path=REPORT_DIR / "reproducibility_manifest.json",
    dataset_version="APTOS 2019 Blindness Detection - labelled train_images (Kaggle competition)",
    split_files=[
        str(REPORT_DIR / "train_split.csv"),
        str(REPORT_DIR / "validation_split.csv"),
        str(REPORT_DIR / "test_split.csv"),
    ],
    checkpoint_files=[
        str(Path(Config.CHECKPOINT_DIR) / "best_phase1.weights.h5"),
        str(Path(Config.CHECKPOINT_DIR) / "best_phase2.weights.h5"),
        str(Path(Config.CHECKPOINT_DIR) / "unet_pseudomask.weights.h5"),
    ],
    # Final full-data fit only (Phase 1 + Phase 2); pilot time is in hyperparameter_tuning_results.csv.
    training_duration_seconds=(
        round(sum(FINAL_FIT_SECONDS.values()), 1) if "FINAL_FIT_SECONDS" in globals() else None
    ),
)

print("[Research Evidence] Error analysis and reproducibility artifacts saved.")
print("[Research Evidence] Ablation variants:")
for variant in ABLATION_VARIANTS:
    print(" -", variant.name)
print({key: value for key, value in error_analysis.items() if key != "error_rows"})

In [ ]:
run_archive = shutil.make_archive(str(Path(Config.REPORTS_DIR).parent), "zip", root_dir=Path(Config.REPORTS_DIR).parent)
print("Download the complete run archive from Kaggle Output:", run_archive)
print("Run folder:", Path(Config.REPORTS_DIR).parent)